# Factors / features quantification – supply side  

In this notebook, I quantify the **zonal potential** of leisure destinations through a set of **supply-side factors** (features) defined at the level of NPVM traffic zones.  
These factors will later enter the discrete choice models as components of the zonal attractiveness term \( A_j(A,D) \) for different age–distance segments.

---

## Objectives

The main objectives of this notebook are:

- To **construct and quantify** a set of candidate supply-side factors \( F_h(j) \) describing the leisure potential of each zone \( j \).
- To **standardise and transform** these factors so that they can be meaningfully combined in a discrete choice utility specification.
- To **produce a clean feature table** that can be merged with the demand-side information (observed trips from MTMC) for model estimation.

---

## Conceptual background

On the **supply side**, each zone \( j \) is characterised by a bundle of attributes that make it more or less attractive as a leisure destination.  
Examples of such attributes include, for instance:

- availability and density of **leisure amenities** (e.g. restaurants, cafés, cultural venues, sport facilities),
- presence and intensity of **natural amenities** (e.g. lakes, mountains, forested areas, viewpoints),
- indicators of **urban form** or **built environment** that may proxy opportunities for leisure activities.

In the discrete choice formulation, these attributes are encoded as factors \( F_h(j) \) that enter the systematic utility of choosing zone \( j \) from origin \( i \):

\[
U_{ij}^{(A,D)} = \sum_h \beta_h^{(A,D)} F_h(j) \;+\; \theta^{(A,D)} \, EMU_{ij} \;+\; \varepsilon_{ij}^{(A,D)}
\]

This notebook focuses **only on the supply-side term** \( \sum_h \beta_h^{(A,D)} F_h(j) \).  
The impedance term \( EMU_{ij} \) and the estimation of \(\beta_h^{(A,D)}\) are handled in separate notebooks.

---

## Workflow

The notebook is structured in the following steps:

1. **Load input data**
   - NPVM 2017 traffic zones (geometry and basic attributes).
   - Amenity and service indicators (e.g. OSM-based counts/densities, other external sources).
   - Additional zonal covariates where relevant.

2. **Define candidate factors**
   - Specify a set of candidate features \( F_h(j) \) that conceptually relate to leisure attractiveness.
   - Decide on functional forms (e.g. densities, log-transforms, presence/absence indicators).

3. **Aggregate and join**
   - Aggregate point- or grid-based information to NPVM zones.
   - Join all candidate factors to the zonal layer.

4. **Transform and normalise**
   - Apply transformations (e.g. log, square root) where appropriate to reduce skewness.
   - Normalise or standardise factors (e.g. z-scores, min–max) to make scales comparable.

5. **Quality checks**
   - Inspect distributions (histograms, summary statistics).
   - Check for outliers and missing values.
   - Inspect correlations between factors to avoid extreme multicollinearity.

6. **Export**
   - Save the final zonal feature table for modelling (e.g. as CSV / Parquet / GeoPackage).
   - Ensure that each zone has a unique identifier compatible with the MTMC-based OD data.

---

## Output

The main output of this notebook is a **zonal factors dataset** containing:

- one row per NPVM zone \( j \),
- one column per supply-side factor \( F_h(j) \),
- a stable zone identifier suitable for merging with other data sources.

This dataset will be used in subsequent notebooks for:

- estimating the discrete choice models by age–distance segment, and  
- constructing the final segment-specific attractiveness indices \( A_j(A,D) \) for integration into SIMBA MOBi.


## Preprocessing of Traffic Zones

#### NPVM traffic zones (Verkehrszonen 2017)

For the construction of leisure attractivity indices and trip-based demand models, I use the official NPVM 2017 traffic zones provided in the dataset `NPVM_2017_TrafficZones_CH_full.gpkg`. This layer is based on the original file `Verkehrszonen_Schweiz_NPVM_2017.gpkg` from the VM–UVEK project and contains one polygon per NPVM traffic zone in Switzerland (including lakes), stored in the Swiss reference system LV95 (EPSG:2056).

In addition to the geometry, the dataset includes the following attributes:

- `zone_id`  
  Unique identifier of the NPVM zone (renamed from the original `ID_alt`). This is the primary key used to link zones to all subsequent indicators and model inputs.

- `ID_Gem`, `N_Gem`  
  Code and name of the commune in which the zone is located. Each zone is entirely contained within exactly one commune.

- `ID_KT`, `N_KT`  
  Canton code and canton abbreviation (e.g. ZH, BE, GE). These can be used for aggregation and descriptive analyses at cantonal level.

- `stg_type`, `N_stg_type`  
  Classification of zones according to the presence of major attractors.  
  The vast majority of zones have `stg_type = 1` and `N_stg_type = None` (no special large-scale attractor). A small subset of zones is flagged as:
  - large shopping centres,
  - major leisure and tourism facilities (e.g. Aquaparc, Zoo Zürich, Technorama),
  - airports and other nationally relevant generators (e.g. Zürich airport, large stadiums).  
  These attributes are potentially useful to interpret very high observed leisure demand in a few specific zones and to compare observed trip patterns with amenity-based attractivity measures.

- `ID_SL3`, `N_SL3`  
  Settlement type at the “structural area” level (Städtisch / Intermediär / Ländlich). This distinguishes urban, intermediate and rural environments and can be used, if needed, to stratify analyses or to check how attractivity patterns differ across settlement types.

- `ID_Agglo`, `N_Agglo`  
  Agglomeration code and name, indicating whether and to which metropolitan area the zone belongs (e.g. Zürich, Basel, Genève–Lausanne). This is relevant for interpreting regional patterns of leisure demand and the role of larger functional urban areas.

- `ID_AMR`, `N_AMR`  
  Labour-market region code and name, capturing functional economic regions used in national transport modelling. These attributes can be used to aggregate results to a meaningful regional scale and to compare leisure attractivity with commuting and employment patterns.

- `Area_m2`, `Area_km2`  
  Planar area of each traffic zone in square metres and square kilometres, computed from the polygon geometry in LV95 (EPSG:2056). These variables provide a basic measure of zone size and are used to derive density-based indicators (e.g. amenities per km²).

In the subsequent analysis, `zone_id` serves as the common spatial key to which I attach:

- observed leisure trip destinations from the microcensus (short and long trips by age segment),
- amenity- and supply-based attractivity factors,
- and, later on, the composite leisure attractivity indices \( A_j(A, D) \).

All original classification variables are retained to facilitate interpretation and potential sensitivity analyses (e.g. checking whether results are robust across urban vs. rural zones or in zones with major attractors).


In [ ]:
import geopandas as gpd
import pandas as pd
import os

# ==========================================================
# 1) Paths
# ==========================================================
ORIG_PATH = "TrafficZones_2017/Switzerland/Verkehrszonen_Schweiz_NPVM_2017.gpkg"
OUT_DIR   = "TZ"
OUT_PATH  = os.path.join(OUT_DIR, "TZ.gpkg")

# ==========================================================
# 2) Read original NPVM zones
# ==========================================================
vz = gpd.read_file(ORIG_PATH)

# Ensure CRS is Swiss LV95 / EPSG:2056
if vz.crs is None or vz.crs.to_epsg() != 2056:
    vz = vz.to_crs(2056)
    print("Reprojected to EPSG:2056")

print("\n=== ORIGINAL columns ===")
print(list(vz.columns))

# ==========================================================
# 3) Create IDs
#    - zone_id: internal sequential id (1..N)
#    - npvm_id: reconstructed NPVM-like ID = ID_Gem | stg_type | zone_in_gem
# ==========================================================
# Keep original row order as read (your assumption)
vz = vz.reset_index(drop=True)

# Internal ID for your pipeline
vz["zone_id"] = (vz.index + 1).astype("int32")

# Required fields for NPVM-like ID
if "ID_Gem" not in vz.columns or "stg_type" not in vz.columns:
    raise ValueError("Missing required columns: ID_Gem and/or stg_type")

vz["ID_Gem"] = pd.to_numeric(vz["ID_Gem"], errors="coerce")
vz["stg_type"] = pd.to_numeric(vz["stg_type"], errors="coerce")

if vz["ID_Gem"].isna().any():
    raise ValueError("Some ID_Gem values are missing / non-numeric.")
if vz["stg_type"].isna().any():
    raise ValueError("Some stg_type values are missing / non-numeric.")

vz["ID_Gem"] = vz["ID_Gem"].astype(int)
vz["stg_type"] = vz["stg_type"].astype(int)

# zone number within Gemeinde (1..k), using file order
vz["zone_in_gem"] = vz.groupby("ID_Gem").cumcount() + 1

# Safety: should fit in 3 digits for NPVM format (001..999)
mx = int(vz["zone_in_gem"].max())
if mx > 999:
    print(f"⚠️ WARNING: max zone_in_gem={mx} > 999. NPVM 3-digit suffix would overflow.")

# Reconstruct NPVM-like numeric ID: ID_Gem * 100000 + stg_type * 1000 + zone_in_gem
# (because stg_type is 2 digits, zone_in_gem is 3 digits)
vz["npvm_id"] = (vz["ID_Gem"] * 100000 + vz["stg_type"] * 1000 + vz["zone_in_gem"]).astype("int64")

# ==========================================================
# 4) Compute area
# ==========================================================
vz["Area_m2"] = vz.geometry.area
vz["Area_km2"] = vz["Area_m2"] / 1e6

# Optional: reorder columns (IDs first)
front = ["zone_id", "npvm_id", "ID_Gem", "stg_type", "zone_in_gem", "Area_m2", "Area_km2"]
rest = [c for c in vz.columns if c not in front]
vz = vz[front + rest]

# ==========================================================
# 5) Save
# ==========================================================
os.makedirs(OUT_DIR, exist_ok=True)
vz.to_file(OUT_PATH, driver="GPKG")
print(f"\n✅ Saved cleaned zones to: {OUT_PATH}")

# ==========================================================
# 6) Print examples
# ==========================================================
print("\n=== NEW columns added ===")
print(front)

print("\n=== Head (IDs + a few labels if present) ===")
show_cols = ["zone_id", "npvm_id", "ID_Gem", "stg_type", "zone_in_gem", "Area_km2"]
for c in ["N_Gem", "N_KT", "N_AMR", "stg_type", "N_stg_type"]:
    if c in vz.columns and c not in show_cols:
        show_cols.append(c)
print(vz[show_cols].head(12).to_string(index=False))

# Show a few examples inside selected Gemeinden (first 2 Gemeinden in the file)
gem_sample = vz["ID_Gem"].drop_duplicates().head(2).tolist()
print("\n=== Examples within a Gemeinde (showing formatted NPVM-like ID) ===")
tmp = vz[vz["ID_Gem"].isin(gem_sample)].copy()

# formatted string like '1711|01|033'
tmp["npvm_id_str"] = (
    tmp["ID_Gem"].astype(str)
    + "|"
    + tmp["stg_type"].astype(int).astype(str).str.zfill(2)
    + "|"
    + tmp["zone_in_gem"].astype(int).astype(str).str.zfill(3)
)

print(tmp[["npvm_id_str", "npvm_id", "zone_id", "ID_Gem", "stg_type", "zone_in_gem"]].head(20).to_string(index=False))

print("\nSanity checks:")
print("zone_id unique?", vz["zone_id"].is_unique)
print("npvm_id unique?", vz["npvm_id"].is_unique)


## Liechtenstein and Exclaves

In addition to the Swiss cantons, the NPVM 2017 zoning system includes 13 “special” traffic zones:  
the 11 municipalities of the Principality of Liechtenstein (N_KT = "LIE") and the two foreign exclaves Büsingen (in the canton of Schaffhausen) and Campione d’Italia (in the canton of Ticino, both coded as N_KT = "Enk."). Each of these municipalities forms a single NPVM traffic zone.

These zones are fully represented in the national transport model (SIMBA MOBi) and in the microcensus (MTMC), i.e. they appear as regular origin–destination zones in the OD matrices. However, some of the supply-side predictors used in this thesis are derived from Swiss Federal Statistical Office (BFS) products (e.g. STATPOP, STATENT), which are primarily designed to cover Swiss territory. As a consequence, several zonal indicators (such as population, households, or employment) are missing or only partially captured for Liechtenstein and for the two exclaves.

In subsequent steps, these special zones are treated explicitly: they are retained as valid traffic zones in the OD data, but any missing or unreliable supply-side attributes must either be imputed (e.g. using external statistics or information from neighbouring Swiss regions) or flagged and handled separately in the modelling and interpretation of leisure attractivity.


In [ ]:
import geopandas as gpd

# Load traffic zones
vz = gpd.read_file("TZ/TZ.gpkg")

# Select Liechtenstein + enclaves
special_labels = ["LIE", "Enk."]

vz_special = (
    vz[vz["N_KT"].isin(special_labels)]
    [["zone_id", "N_Gem", "N_KT", "npvm_id", "Area_km2"]]
    .copy()
)

print(vz_special.to_string(index=False))


## F1: Gastronomy

### Data source and spatial coverage

To capture the spatial offer of restaurants and related food facilities, I use an extract of **OpenStreetMap (OSM)** amenities, stored as a point layer in:

- `OSM/Gastronomy/F1_Gastronomy.gpkg`

This layer contains all OSM objects whose `amenity=*` tag belongs to the following set:

- `restaurant`
- `fast_food`
- `cafe`
- `pub`
- `bar`
- `ice_cream`
- `food_court`
- `biergarten`

The extraction boundary is defined such that it fully covers:

- Switzerland,
- the Principality of Liechtenstein,
- the foreign enclaves Büsingen am Hochrhein and Campione d’Italia,

so that gastronomic facilities are available for all NPVM traffic zones that appear in the OD data and in SIMBA MOBi.  
All features are stored as **points** in the Swiss reference system **LV95 / EPSG:2056**.

### Attributes and basic completeness

For each establishment, the dataset typically includes:

- `amenity` – type of establishment (restaurant, fast_food, cafe, bar, pub, …)
- `name` – facility name
- address-related fields:
  - `addr:street`
  - `addr:housenumber`
  - `addr:postcode`
  - `addr:city`
- `cuisine` – one or more cuisine types, where mapped
- further optional tags, e.g. `phone`, `website`, `wheelchair`, `outdoor_seating`, etc.

A completeness check on the raw layer (`Gastronomy.gpkg`, 28 939 rows) shows:

- **100 %** non-missing for `id`, `@id`, `amenity`, `geometry`,
- about **97 %** of records with a valid `name`,
- roughly **40–46 %** with at least some address information (`addr:street`, `addr:housenumber`, `addr:postcode`, `addr:city`),
- `cuisine` is present for a substantial share of `fast_food` and `restaurant` entries, but much less frequently for other amenity types.

### Cleaning and deduplication of gastronomic points

OSM typically contains multiple representations of the *same* facility, for example:

- overlapping nodes and building centroids,
- historical duplicates after geometry edits,
- parallel mapping of the same chain outlet.

Since the attractiveness factors will be aggregated to traffic zones, counting such duplicates as separate restaurants would artificially inflate supply in dense areas. I therefore perform a **spatial deduplication** of facilities with the same name and amenity type.

#### Step 1 – CRS check

The layer is read from disk and, if necessary, reprojected to LV95:

- if the input CRS is not EPSG:2056, it is transformed to **EPSG:2056** to ensure that distances are in metres and spatial thresholds are meaningful.

#### Step 2 – Name and address flags

Two helper flags are computed:

- `valid_name`: `True` if `name` is a non-empty string and not the literal `"None"` (case-insensitive),  
- `has_address`: `True` if at least one of the address fields (`addr:street`, `addr:housenumber`, `addr:postcode`, `addr:city`) is present and not `"None"`.

Only records with `valid_name = True` are considered for deduplication; unnamed amenities are kept as they are.

#### Step 3 – Spatial deduplication within 100 m

For all records with a valid name, I group points by:

- `(amenity, name)`

and, within each group, I identify spatial duplicates based on their mutual distance.

Algorithm (per `(amenity, name)` group):

1. Sort group so that rows with an address (`has_address = True`) come first.  
   This gives priority to points with more complete attribute information.
2. Iterate over the sorted points and, for each point \( i \), compare it with all subsequent points \( j > i \).
3. Compute the Euclidean distance in LV95 coordinates:

4. If \( d < 100 \) metres, then point \( j \) is considered a **duplicate** of point \( i \) and is **removed**.
   - If both points have an address, the first one in the sorted order is kept.
   - If only one has an address, that one is kept.
   - If none has address information, the first one in the group is kept and the others within 100 m are dropped.

Points with different `name` or different `amenity` are never compared against each other and are all retained, even if they are geographically close (e.g. a `cafe` next to a `restaurant` with a different name).

#### Resulting cleaned layer

After applying the 100 m deduplication:

- **Original number of rows**: 28 973 
- **Spatial duplicates removed (≤ 100 m, same amenity + name)**: 93  
- **Rows after cleaning**: 28 882  

The cleaned dataset is saved as:

- `OSM/Gastronomy/F1_Gastronomy_clean.gpkg`

This cleaned layer is used in the subsequent steps, where facilities are spatially joined to NPVM traffic zones and aggregated to build the gastronomic supply factor(s) for the attractiveness term \( A_j(A,D) \).


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os

# ==========================================================
# Paths
# ==========================================================
GASTRO_PATH = "OSM/Gastronomy/F1_Gastronomy.gpkg"
OUT_PATH    = "OSM/Gastronomy/F1_Gastronomy_clean.gpkg"

# ==========================================================
# 1. Load data
# ==========================================================
gastro = gpd.read_file(GASTRO_PATH)
print(f"Original rows: {len(gastro)}")

# Ensure metric CRS (LV95 if possible)
if gastro.crs is None:
    raise ValueError("Gastronomy layer has no CRS defined.")
if gastro.crs.to_epsg() != 2056:
    gastro = gastro.to_crs(2056)
    print("Reprojected Gastronomy layer to EPSG:2056")

# Convenience
NAME_COL   = "name"
ADDR_COLS  = ["addr:street", "addr:housenumber", "addr:postcode", "addr:city"]

def is_valid_str(x):
    """Return True if x is a non-empty string and not the literal 'None' (case-insensitive)."""
    if not isinstance(x, str):
        return False
    return x.strip() != "" and x.strip().lower() != "none"

# ----------------------------------------------------------
# Flag: valid name
# ----------------------------------------------------------
gastro["valid_name"] = gastro[NAME_COL].apply(is_valid_str)

# ----------------------------------------------------------
# Flag: presence of any address information
# ----------------------------------------------------------
def row_has_address(row):
    """Return True if at least one address field is present and not 'None'."""
    for c in ADDR_COLS:
        v = row.get(c)
        if pd.isna(v):
            continue
        if not isinstance(v, str):
            return True
        if v.strip() != "" and v.strip().lower() != "none":
            return True
    return False

gastro["has_address"] = gastro.apply(row_has_address, axis=1)

# ==========================================================
# 2. Single step – spatial deduplication within 100 m
#    for identical (amenity, name)
# ==========================================================
to_drop_spatial = pd.Series(False, index=gastro.index)

# Only consider rows with a valid name
candidates = gastro[gastro["valid_name"]].copy()

# Group by (amenity, name)
grouped = candidates.groupby(["amenity", NAME_COL], sort=False)

EPS = 100.0  # metres

for (amen, nm), grp in grouped:
    if len(grp) < 2:
        continue

    # Sort so that rows with an address come first
    grp = grp.sort_values("has_address", ascending=False)
    idxs = list(grp.index)
    xs = grp.geometry.x.values
    ys = grp.geometry.y.values
    n = len(idxs)

    for i in range(n):
        i_idx = idxs[i]
        if to_drop_spatial.loc[i_idx]:
            continue  # already marked as duplicate

        xi, yi = xs[i], ys[i]
        # compare only with j > i
        for j in range(i + 1, n):
            j_idx = idxs[j]
            if to_drop_spatial.loc[j_idx]:
                continue

            dx = xs[j] - xi
            dy = ys[j] - yi
            dist = (dx * dx + dy * dy) ** 0.5

            if dist <= EPS:
                # same (amenity, name) and within 100 m → drop j
                to_drop_spatial.loc[j_idx] = True

print(f"Spatial duplicates (<=100 m) to drop: {to_drop_spatial.sum()}")

# Apply drops
gastro_clean = gastro.loc[~to_drop_spatial].copy()

# Remove helper columns
gastro_clean = gastro_clean.drop(
    columns=["valid_name", "has_address"],
    errors="ignore"
)

print(f"Rows after cleaning: {len(gastro_clean)}")

# ==========================================================
# 3. Save cleaned layer
# ==========================================================
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
gastro_clean.to_file(OUT_PATH, driver="GPKG")

print(f"Saved cleaned gastronomy layer to: {OUT_PATH}")


### Indicator construction: `F1_gastr_count` and `F1_gastr_div`

Starting from the cleaned OSM layer `OSM/Gastronomy/Gastronomy_clean.gpkg`, I construct two gastronomy-related indicators at NPVM zone level:

- `F1_gastr_count` – **number of gastronomy POIs** per NPVM zone  
- `F1_gastr_div` – **diversity of gastronomy types** per NPVM zone (0–5)

The cleaned layer contains all OSM objects tagged with one of the following `amenity=*` values:

- `restaurant`
- `fast_food`
- `cafe`
- `pub`
- `bar`
- `ice_cream`
- `food_court`
- `biergarten`

#### Spatial assignment of gastronomy POIs to NPVM zones

All OSM gastronomy points are assigned to NPVM 2017 traffic zones (`TZ/TZ.gpkg`) in **LV95 / EPSG:2056** in two steps:

1. **Within-join**

   Each POI is first joined to the NPVM zone whose polygon *contains* the point (`predicate="within"`). This captures all points that clearly fall inside a traffic zone.

2. **Nearest-join for residual points**

   Points that are still not assigned after the first step (e.g. POIs lying exactly on a boundary or slightly outside due to geometry or data issues) are matched to the **nearest** NPVM zone using `sjoin_nearest`.

   After these two steps, every gastronomy POI is associated with exactly one NPVM zone (including Liechtenstein and the two enclaves).

On this basis, the **zonal count** is simply the number of POIs per `zone_id`:

$$
F1^{\text{gastr,count}}(j)
= \bigl|\{\, p : p \text{ is a gastronomy POI and } p \in j \,\}\bigr|
$$

Zones without any gastronomy POIs receive:

$$
F1^{\text{gastr,count}}(j) = 0
$$

and this is stored as `F1_gastr_count` in `TZ.gpkg`.

#### Amenity grouping and diversity measure

OSM distinguishes several detailed `amenity=*` values for gastronomy. For modelling purposes, I aggregate them into **five broader groups**:

- **Drinking places**: `bar`, `pub`, `biergarten` → group `"drinking"`  
- **Restaurants**: `restaurant`, `food_court` → group `"restaurant"`  
- **Cafés**: `cafe` → group `"cafe"`  
- **Fast food**: `fast_food` → group `"fast_food"`  
- **Ice cream / gelaterie**: `ice_cream` → group `"ice_cream"`

Formally, for each POI with amenity value \( a \):

$$
\text{amenity\_group}(a) \in
\{\text{drinking}, \text{restaurant}, \text{cafe}, \text{fast\_food}, \text{ice\_cream}\}
\quad \text{or } \text{None if } a \text{ is outside these categories.}
$$

For each traffic zone \( j \), I compute the **number of distinct gastronomy groups present**:

$$
F1^{\text{gastr,div}}(j)
= \bigl|\{\, \text{amenity\_group}(p) : p \in j,\ \text{amenity\_group}(p) \neq \text{None} \,\}\bigr|
$$

By construction:

-  F1_gastr_div(j) = 0  if the zone has no gastronomy POIs in the selected groups  
-  F1_gastr_div(j) in {1,2,3,4,5} otherwise  

This indicator captures **qualitative diversity** of the gastronomy offer:

- A zone with only restaurants (no cafés, no fast food, etc.) has `F1_gastr_div = 1` regardless of how many restaurants there are.
- A zone with a restaurant, a café and a fast-food outlet has `F1_gastr_div = 3`.

In the discrete choice models, `F1_gastr_count` primarily reflects the **intensity** of the gastronomic supply, while `F1_gastr_div` reflects the **variety of options** available in the zone.

---

### Validation against official STATENT gastronomy workplaces

To assess the reliability of OSM-based indicators, I compare them with an independent official source: the **STATENT 2023** workplace statistics.

From `Jobs_2023/Jobs_2023.gpkg`, I use the attribute:

- `B0856AS` – number of workplaces in **NOGA 56 – Gastronomy**

as an official measure of gastronomy activity.

#### Assignment of STATENT workplaces to NPVM zones

- As target polygons, I use **Swiss NPVM zones only**, excluding Liechtenstein and the two enclaves (`N_KT = "LIE"` and `N_KT = "Enk."`) from the join.
- Each STATENT workplace point is first joined to the zone whose polygon contains it (`predicate="within"`).
- Points without a match are then assigned to the **nearest Swiss NPVM zone** via `sjoin_nearest`.

For each Swiss zone \( j \), I aggregate official gastronomy workplaces:

$$
F1^{\text{gastr,off}}(j)
= \sum_{p \in j} \text{B0856AS}_p
$$

This value is stored as `F1_gastr_off_counts` (official gastronomy workplaces per NPVM zone).

#### Correlation between OSM counts and official gastronomy workplaces

For all **Swiss zones** (excluding `LIE` and `Enk.`), I compare:

- `F1_gastr_count` – OSM-based gastronomy POI count  
- `F1_gastr_off_counts` – official gastronomy workplaces (STATENT, NOGA 56)

using both **Pearson** and **Spearman** correlation.

- On **all Swiss zones (including zeros)**, the correlations are approximately:

  - Pearson:  
    $$
    \rho_{\text{Pearson}} \bigl(F1^{\text{gastr,count}}, F1^{\text{gastr,off}}\bigr)
    \approx 0.69
    $$
  - Spearman:  
    $$
    \rho_{\text{Spearman}} \bigl(F1^{\text{gastr,count}}, F1^{\text{gastr,off}}\bigr)
    \approx 0.65
    $$

- Restricting to zones where **both indicators are strictly positive**:

  $$
  F1^{\text{gastr,count}}(j) > 0,\quad
  F1^{\text{gastr,off}}(j) > 0
  $$

  the correlations remain similar:

  - Pearson:
    $$
    \rho_{\text{Pearson}} \approx 0.67
    $$
  - Spearman:
    $$
    \rho_{\text{Spearman}} \approx 0.60
    $$

#### Interpretation

These values indicate a **moderately strong association** between the OSM-based POI counts and the official workplace-based measure of gastronomy:

- Zones with many OSM gastronomy POIs tend to host many official gastronomy workplaces, and vice versa.
- Deviations between the two may reflect differences in mapping completeness, classification, or temporal mismatches, but the overall spatial pattern is consistent.

For the purposes of this thesis, I therefore consider:

- `F1_gastr_count` a **plausible proxy** for the intensity of gastronomy supply at zonal level.
- `F1_gastr_div` a complementary measure of **variety**, capturing the richness of the local gastronomy mix.

Together, these indicators will enter the zonal attractiveness term for gastronomy-related leisure trips in the discrete choice models.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==========================================================
# Paths
# ==========================================================
TZ_PATH          = "TZ/TZ.gpkg"
GASTRO_CLEAN     = "OSM/Gastronomy/F1_Gastronomy_clean.gpkg"
JOBS_PATH        = "Jobs_2023/Jobs_2023.gpkg"

SPECIAL_LABELS   = ["LIE", "Enk."]   # Liechtenstein + enclaves


# ==========================================================
# 1. Load traffic zones and gastronomy points
# ==========================================================
tz = gpd.read_file(TZ_PATH)
print(f"Traffic zones: {len(tz)}")

if tz.crs is None or tz.crs.to_epsg() != 2056:
    tz = tz.to_crs(2056)
    print("Reprojected TZ to EPSG:2056")

gastro = gpd.read_file(GASTRO_CLEAN)
print(f"Gastronomy points (clean): {len(gastro)}")

if gastro.crs is None:
    raise ValueError("Gastronomy_clean layer has no CRS defined.")
if gastro.crs.to_epsg() != 2056:
    gastro = gastro.to_crs(2056)
    print("Reprojected Gastronomy layer to EPSG:2056")

if "amenity" not in gastro.columns:
    raise ValueError("'amenity' column not found in gastronomy data.")


# ==========================================================
# 1b. Summary tables: amenity types
# ==========================================================
amenity_counts = (
    gastro["amenity"]
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts(dropna=False)
    .rename_axis("amenity")
    .reset_index(name="n")
)
amenity_counts["share"] = amenity_counts["n"] / len(gastro)
print("\n=== Amenity value counts (raw) ===")
print(amenity_counts.to_string(index=False))


# ==========================================================
# 2. Spatial join: assign every gastronomy POI to an NPVM zone
#    - first 'within'
#    - then missing ones via nearest zone (including LIE / Enk.)
# ==========================================================
join_g = gpd.sjoin(
    gastro,
    tz[["zone_id", "N_KT", "geometry"]],
    how="left",
    predicate="within",
    lsuffix="poi",
    rsuffix="tz"
)

missing = join_g[join_g["zone_id"].isna()].copy()
print(f"\nGastronomy points without zone after 'within': {len(missing)}")

if not missing.empty:
    # Keep only the POI geometry for the nearest-join (avoid duplicated columns)
    missing_clean = missing.drop(
        columns=[c for c in missing.columns if c.startswith("index_") or c in ["zone_id", "N_KT"]],
        errors="ignore"
    )

    nearest = gpd.sjoin_nearest(
        missing_clean,
        tz[["zone_id", "N_KT", "geometry"]],
        how="left",
        distance_col="dist_to_zone",
        lsuffix="poi",
        rsuffix="tz"
    )

    join_g.loc[missing.index, "zone_id"] = nearest["zone_id"].values
    join_g.loc[missing.index, "N_KT"]    = nearest["N_KT"].values

join_g = join_g[join_g["zone_id"].notna()].copy()
join_g["zone_id"] = join_g["zone_id"].astype("int32")

print(f"Gastronomy points with assigned zone: {len(join_g)}")


# ==========================================================
# 3. F1_gastr_count: count of gastronomy POIs per NPVM zone
# ==========================================================
count_per_zone = (
    join_g
    .groupby("zone_id", dropna=True)
    .size()
    .reset_index(name="F1_gastr_count")
)

tz = tz.merge(count_per_zone, on="zone_id", how="left")
tz["F1_gastr_count"] = tz["F1_gastr_count"].fillna(0).astype("int32")

print("\nExample of F1_gastr_count:")
print(tz[["zone_id", "F1_gastr_count"]].head())


# ==========================================================
# 4. F1_gastr_div: diversity of amenity groups (0–5)
#    Groups:
#      - bar, pub, biergarten           → "drinking"
#      - restaurant, food_court        → "restaurant"
#      - cafe                          → "cafe"
#      - fast_food                     → "fast_food"
#      - ice_cream                     → "ice_cream"
# ==========================================================
GROUP_MAP = {
    "bar": "drinking",
    "pub": "drinking",
    "biergarten": "drinking",
    "restaurant": "restaurant",
    "food_court": "restaurant",
    "cafe": "cafe",
    "fast_food": "fast_food",
    "ice_cream": "ice_cream",
}

def amenity_group(a):
    if not isinstance(a, str):
        return None
    return GROUP_MAP.get(a.strip().lower(), None)

join_g["amenity_group"] = join_g["amenity"].apply(amenity_group)

# table: group counts (useful in report)
group_counts = (
    join_g["amenity_group"]
    .value_counts(dropna=False)
    .rename_axis("amenity_group")
    .reset_index(name="n")
)
group_counts["share"] = group_counts["n"] / len(join_g)
print("\n=== Amenity-group counts (after mapping) ===")
print(group_counts.to_string(index=False))

# table: mapping coverage (amenity -> group)
amenity_to_group = (
    join_g[["amenity", "amenity_group"]]
    .drop_duplicates()
    .sort_values(["amenity_group", "amenity"], na_position="last")
)
print("\n=== Amenity -> group mapping (distinct pairs) ===")
print(amenity_to_group.to_string(index=False))

div_per_zone = (
    join_g
    .dropna(subset=["amenity_group"])
    .groupby("zone_id")["amenity_group"]
    .nunique()
    .reset_index(name="F1_gastr_div")
)

tz = tz.merge(div_per_zone, on="zone_id", how="left")
tz["F1_gastr_div"] = tz["F1_gastr_div"].fillna(0).astype("int32")

print("\nExample of F1_gastr_div:")
print(tz[["zone_id", "F1_gastr_count", "F1_gastr_div"]].head())


# ==========================================================
# 5. VALIDATION CHECK WITH STATENT JOBS (B0856AS)
#    - Assign STATENT workplace points to Swiss zones only
#      (exclude LIE / Enk. as targets)
#    - Sum B0856AS per zone
#    - Save result as F1_gastr_off_counts in TZ
#    - Correlate OSM count vs official gastronomy workplaces
# ==========================================================
jobs = gpd.read_file(JOBS_PATH)
print(f"\nJobs points: {len(jobs)}")

if jobs.crs is None:
    raise ValueError("Jobs_2023 layer has no CRS defined.")
if jobs.crs.to_epsg() != 2056:
    jobs = jobs.to_crs(2056)
    print("Reprojected Jobs layer to EPSG:2056")

if "B0856AS" not in jobs.columns:
    raise ValueError("Column 'B0856AS' not found in Jobs_2023.gpkg")

# Ensure numeric; clamp negatives to 0 (safety)
jobs["B0856AS"] = pd.to_numeric(jobs["B0856AS"], errors="coerce")
jobs["B0856AS"] = jobs["B0856AS"].fillna(0)
jobs.loc[jobs["B0856AS"] < 0, "B0856AS"] = 0

# Use only Swiss zones as targets (exclude LIE / Enk.)
tz_ch = tz[~tz["N_KT"].isin(SPECIAL_LABELS)].copy()

join_jobs = gpd.sjoin(
    jobs,
    tz_ch[["zone_id", "geometry"]],
    how="left",
    predicate="within",
    lsuffix="job",
    rsuffix="tz"
)

missing_j = join_jobs[join_jobs["zone_id"].isna()].copy()
print(f"Jobs points without zone after 'within': {len(missing_j)}")

if not missing_j.empty:
    missing_j_clean = missing_j.drop(
        columns=[c for c in missing_j.columns if c.startswith("index_") or c == "zone_id"],
        errors="ignore"
    )

    nearest_j = gpd.sjoin_nearest(
        missing_j_clean,
        tz_ch[["zone_id", "geometry"]],
        how="left",
        distance_col="dist_to_zone",
        lsuffix="job",
        rsuffix="tz"
    )

    join_jobs.loc[missing_j.index, "zone_id"] = nearest_j["zone_id"].values

join_jobs = join_jobs[join_jobs["zone_id"].notna()].copy()
join_jobs["zone_id"] = join_jobs["zone_id"].astype("int32")

jobs_zone = (
    join_jobs
    .groupby("zone_id", dropna=True)["B0856AS"]
    .sum()
    .reset_index(name="F1_gastr_off_counts")
)

tz = tz.merge(jobs_zone, on="zone_id", how="left")
tz["F1_gastr_off_counts"] = tz["F1_gastr_off_counts"].fillna(0).astype("int32")


# ==========================================================
# 6. Correlation between OSM counts and STATENT gastronomy
# ==========================================================
df_ch = tz.loc[~tz["N_KT"].isin(SPECIAL_LABELS)].copy()
print(f"\nSwiss zones used for correlation (excluding LIE / Enk.): {len(df_ch)}")

pearson_all  = df_ch["F1_gastr_count"].corr(df_ch["F1_gastr_off_counts"], method="pearson")
spearman_all = df_ch["F1_gastr_count"].corr(df_ch["F1_gastr_off_counts"], method="spearman")

print("\nCorrelation on ALL Swiss zones (including zeros):")
print(f"  Pearson  (OSM count vs STATENT B0856AS): {pearson_all:.3f}")
print(f"  Spearman (OSM count vs STATENT B0856AS): {spearman_all:.3f}")

mask_nz = (df_ch["F1_gastr_count"] > 0) & (df_ch["F1_gastr_off_counts"] > 0)
df_nz = df_ch.loc[mask_nz].copy()

pearson_nz  = df_nz["F1_gastr_count"].corr(df_nz["F1_gastr_off_counts"], method="pearson")
spearman_nz = df_nz["F1_gastr_count"].corr(df_nz["F1_gastr_off_counts"], method="spearman")

print(f"\nZones with both OSM count > 0 and STATENT B0856AS > 0: {len(df_nz)}")
print("Correlation on zones with non-zero values:")
print(f"  Pearson  (OSM count vs STATENT B0856AS): {pearson_nz:.3f}")
print(f"  Spearman (OSM count vs STATENT B0856AS): {spearman_nz:.3f}")


# ==========================================================
# 7. Save updated TZ with F1_gastr_* (including LIE + Enk.)
# ==========================================================
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(
    f"\nUpdated TZ saved to: {TZ_PATH}\n"
    "Added variables: F1_gastr_count, F1_gastr_div, F1_gastr_off_counts"
)


# ==========================================================
# 8. Quick maps (ordered)
# ==========================================================
MAP_SPECS = [
    ("F1_gastr_count",       "F1 (Gastronomy) – POI count per NPVM zone"),
    ("F1_gastr_div",         "F1 (Gastronomy) – Diversity of gastronomy groups (0–5)"),
    ("F1_gastr_off_counts",  "Validation – STATENT gastronomy workplaces (B0856AS) per NPVM zone"),
]

for col, title in MAP_SPECS:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    tz.plot(
        column=col,
        legend=True,
        ax=ax,
        edgecolor="black",
        linewidth=0.1
    )
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


## Gastronomy floor area

### Idea and data

Goal: estimate **how much building floor area** is associated with gastronomy uses in each NPVM traffic zone.

Data:

- **Traffic zones**: `TZ/TZ.gpkg`  
  – NPVM 2017 polygons, key attributes `zone_id`, `N_KT`, CRS = EPSG:2056.

- **Gastronomy POIs (OSM)**: `OSM/Gastronomy/Gastronomy_clean.gpkg`  
  – OSM points with `amenity` in:
  - `restaurant`
  - `fast_food`
  - `cafe`
  - `pub`
  - `bar`
  - `ice_cream`
  - `food_court`
  - `biergarten`

- **Buildings (swissTLM3D)**: `swissTLM3D_2025/SWISSTLM3D_2025.gpkg`, layer  
  `tlm_bauten_gebaeude_footprint` – building footprint polygons, CRS = EPSG:2056.

All layers are used in **LV95 / EPSG:2056**, so areas and distances are in metres.

---

### 1. Building area from swissTLM3D

From the building footprint layer I keep:

- `uuid` – building ID  
- `objektart`, `nutzung` – type / use information (only used for optional diagnostics)  
- `geometry` – footprint polygon

and compute the building area in square metres as:

- `bldg_area_m2 = area(geometry)` (in m² because EPSG:2056 is metric)

This provides the geometric base to link OSM POIs to real buildings.

---

### 2. Matching OSM gastronomy ↔ buildings

For each gastronomy POI:

1. **“Within” join**  
   I perform an `sjoin` with `predicate="within"` between OSM points and building polygons.  
   - If the point falls inside one or more buildings, I keep the first match and copy:
     - `bldg_uuid`
     - `bldg_area_m2`
     - (optionally `bldg_objektart`, `bldg_nutzung`)

2. **“Nearest” join for remaining points**  
   For POIs that are not contained in any building:
   - I use `sjoin_nearest` with the building layer,
   - obtain the nearest building and its distance `dist_to_bldg_m`,
   - and assign that building’s `bldg_area_m2` to the POI.

After this step, almost all OSM gastronomy points have an associated building area (either “within” or via nearest building).

---

### 3. Treatment of Liechtenstein and enclaves

NPVM zones with:

- `N_KT = "LIE"` (Liechtenstein)
- `N_KT = "Enk."` (Büsingen am Hochrhein, Campione d’Italia)

are not consistently covered by swissTLM3D.

For POIs in these zones:

- I do **not** rely on the swissTLM3D building match for area estimation,  
- their area is treated as “missing” and will be imputed from the Swiss distributions (see below).

---

### 4. Plausible range of building area

To filter out clearly unrealistic buildings for a **single** gastronomy POI, I define a plausible range:

$$
15 \,\text{m}^2 \;\leq\; bldg\_area\_m2 \;\leq\; 1000 \,\text{m}^2
$$

Handling:

- if `bldg_area_m2` is missing or outside this interval, it is set to `NaN`,
- these POIs will later receive an area via sampling (step 6).

The POIs with valid area (in [15, 1000] m², Swiss zones only) form the **clean sample** used to estimate the area distribution by amenity type.

---

### 5. Lognormal distributions for building area

On the valid Swiss POIs I define:

- `log_area = log(bldg_area_m2)`

and compute:

- **Global distribution** (all valid POIs):
  - `μ_all` = mean of `log_area`  
  - `σ_all` = standard deviation of `log_area`

- **Amenity-specific distributions** (only if sample size is sufficient):
  - for each `amenity = a` with at least 30 valid POIs:
    - `μ_a` = mean of `log_area` among POIs of type `a`  
    - `σ_a` = standard deviation of `log_area` for type `a`

Interpretation:

- for amenity `a` with enough observations:

  $$
  \log(\text{area}) \sim \mathcal{N}(\mu_a, \sigma_a^2)
  $$

- for amenity types with few observations:

  $$
  \log(\text{area}) \sim \mathcal{N}(\mu_{\text{all}}, \sigma_{\text{all}}^2)
  $$

Sampling is performed from the corresponding lognormal distribution and then **truncated** to [15, 1000] m² via simple rejection sampling (if a draw is outside the range, draw again).

---

### 6. POI-level area: `F1_gastr_area_m2`

For each gastronomy POI \( p \) (including those in LIE / Enk.) I define:

- if there is a valid building area:

  $$
  F1_{\text{gastr\_area\_m2}}(p) = bldg\_area\_m2(p)
  $$

- if the area is missing or out of range:

  $$
  F1_{\text{gastr\_area\_m2}}(p) = \tilde{A}(p)
  $$

where:

- \( \tilde{A}(p) \) is an area sampled from:
  - the amenity-specific lognormal distribution (if available for that amenity), or
  - the global lognormal distribution otherwise,
- and is always forced into the interval [15, 1000] m².

This ensures that every OSM gastronomy point receives a coherent and plausible building area, even when the exact building match is unreliable or missing.

---

### 7. Aggregation to NPVM zones

Each POI already has a `zone_id`, obtained via spatial join between `Gastronomy_clean` and `TZ.gpkg`:

- first with `predicate="within"`  
- then, for any remaining points, with `sjoin_nearest` to attach the closest NPVM zone

(Liechtenstein and the enclaves are included here as standard zones).

The total gastronomy floor area for zone \( j \) is:

$$
F1^{\text{area,tot}}(j)
=
\sum_{p \in j} F1_{\text{gastr\_area\_m2}}(p)
$$

This is stored in the NPVM layer as:

- `F1_gastr_area_total` – total floor area (m²) associated with gastronomy POIs in each zone.

Together with:

- `F1_gastr_count` – number of gastronomy POIs per zone,  
- `F1_gastr_div` – diversity of amenity groups (drinking / restaurant / cafe / fast_food / ice_cream),

`F1_gastr_area_total` completes the description of the gastronomy supply per zone, combining **how many** POIs there are, **how diverse** they are, and **how much floor space** they occupy.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -------------------------------------------------------------
# Paths
# -------------------------------------------------------------
TZ_PATH       = "TZ/TZ.gpkg"
GASTRO_PATH   = "OSM/Gastronomy/F1_Gastronomy_clean.gpkg"
TLM_PATH      = "swissTLM3D_2025/SWISSTLM3D_2025.gpkg"
TLM_BLD_LAYER = "tlm_bauten_gebaeude_footprint"

SPECIAL_LABELS = ["LIE", "Enk."]   # Liechtenstein + enclaves
np.random.seed(123)

# -------------------------------------------------------------
# Parameters
# -------------------------------------------------------------
NEAREST_TZ_MAX_M   = 50.0    # threshold (meters) for nearest zone assignment
NEAREST_BLDG_MAX_M = 50.0    # threshold (meters) for nearest building assignment
AREA_MIN_M2 = 15.0
AREA_MAX_M2 = 1000.0
MIN_AMENITY_N = 30           # amenity-specific distribution threshold

# -------------------------------------------------------------
# 1. Load data: TZ, OSM gastronomy, swissTLM3D buildings
# -------------------------------------------------------------
tz = gpd.read_file(TZ_PATH)
print(f"Traffic zones: {len(tz)}")

if tz.crs is None or tz.crs.to_epsg() != 2056:
    tz = tz.to_crs(2056)
    print("Reprojected TZ to EPSG:2056")

gdf_gastro = gpd.read_file(GASTRO_PATH)
print(f"Gastronomy points (clean): {len(gdf_gastro)}")

if gdf_gastro.crs is None or gdf_gastro.crs.to_epsg() != 2056:
    gdf_gastro = gdf_gastro.to_crs(2056)
    print("Reprojected Gastronomy to EPSG:2056")

# Create a stable POI id
gdf_gastro = gdf_gastro.reset_index(drop=True)
gdf_gastro["poi_id"] = gdf_gastro.index.astype("int64")

# -------------------------------------------------------------
# 2. Load building footprints
# -------------------------------------------------------------
gdf_geb = gpd.read_file(TLM_PATH, layer=TLM_BLD_LAYER)
print(f"Building footprints (raw): {len(gdf_geb)}")

if gdf_geb.crs is None or gdf_geb.crs.to_epsg() != 2056:
    gdf_geb = gdf_geb.to_crs(2056)
    print("Reprojected buildings to EPSG:2056")

gdf_geb = gdf_geb[gdf_geb.geometry.notna() & (~gdf_geb.geometry.is_empty)].copy()

# -------------------------------------------------------------
# 3. Compute building area and keep relevant columns
# -------------------------------------------------------------
gdf_geb["bldg_area_m2"] = gdf_geb.geometry.area

print("\nDescriptive stats of building areas (m²):")
print(gdf_geb["bldg_area_m2"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

bld_cols = ["uuid", "objektart", "nutzung", "bldg_area_m2", "geometry"]
bld_cols = [c for c in bld_cols if c in gdf_geb.columns]
geb_sel = gdf_geb[bld_cols].copy()

if "objektart" in geb_sel.columns:
    print("\nSample of objektart value counts:")
    print(geb_sel["objektart"].value_counts().head(10))

# -------------------------------------------------------------
# 4. Join POIs to traffic zones (zone_id, N_KT) with threshold
#    - within first
#    - nearest for remaining (<= 50 m), else keep NaN
# -------------------------------------------------------------
gastro_bld = gdf_gastro.copy()
gastro_bld["zone_id"] = np.nan
gastro_bld["N_KT"] = None
gastro_bld["dist_to_zone_m"] = np.nan

left_tz = gpd.GeoDataFrame(
    gastro_bld[["poi_id", "geometry"]].copy(),
    geometry="geometry",
    crs=gastro_bld.crs,
)
right_tz = tz[["zone_id", "N_KT", "geometry"]].copy()

join_tz = gpd.sjoin(left_tz, right_tz, how="left", predicate="within")

# assign within
within_info = (
    join_tz.dropna(subset=["zone_id"])
    .groupby("poi_id")[["zone_id", "N_KT"]]
    .first()
)

mask_within = gastro_bld["poi_id"].isin(within_info.index)
gastro_bld.loc[mask_within, "zone_id"] = gastro_bld.loc[mask_within, "poi_id"].map(within_info["zone_id"]).astype("float64")
gastro_bld.loc[mask_within, "N_KT"]    = gastro_bld.loc[mask_within, "poi_id"].map(within_info["N_KT"])

n_zone_within = int(mask_within.sum())
print(f"\nZone assignment by WITHIN: {n_zone_within}")

# nearest for remaining
missing_zone_mask = gastro_bld["zone_id"].isna()
n_missing_after_within = int(missing_zone_mask.sum())
print(f"POIs without zone after WITHIN: {n_missing_after_within}")

n_zone_nearest = 0
n_zone_unassigned = 0

if n_missing_after_within > 0:
    left_tz_near = gpd.GeoDataFrame(
        gastro_bld.loc[missing_zone_mask, ["poi_id", "geometry"]].copy(),
        geometry="geometry",
        crs=gastro_bld.crs,
    )

    nearest_tz = gpd.sjoin_nearest(
        left_tz_near,
        right_tz,
        how="left",
        distance_col="dist_to_zone_m",
    )

    # apply threshold
    nearest_tz.loc[nearest_tz["dist_to_zone_m"] > NEAREST_TZ_MAX_M, ["zone_id", "N_KT"]] = np.nan

    tz_near_info = (
        nearest_tz.dropna(subset=["zone_id"])
        .groupby("poi_id")[["zone_id", "N_KT", "dist_to_zone_m"]]
        .first()
    )

    mask_near = gastro_bld["poi_id"].isin(tz_near_info.index)
    gastro_bld.loc[mask_near, "zone_id"] = gastro_bld.loc[mask_near, "poi_id"].map(tz_near_info["zone_id"]).astype("float64")
    gastro_bld.loc[mask_near, "N_KT"]    = gastro_bld.loc[mask_near, "poi_id"].map(tz_near_info["N_KT"])
    gastro_bld.loc[mask_near, "dist_to_zone_m"] = gastro_bld.loc[mask_near, "poi_id"].map(tz_near_info["dist_to_zone_m"]).astype("float64")

    n_zone_nearest = int(mask_near.sum())
    n_zone_unassigned = int(gastro_bld["zone_id"].isna().sum())

print(f"Zone assignment by NEAREST (<= {NEAREST_TZ_MAX_M:.0f} m): {n_zone_nearest}")
print(f"POIs still without zone (kept NaN): {n_zone_unassigned}")

# keep only those with a zone (recommended for aggregation)
gastro_bld = gastro_bld[gastro_bld["zone_id"].notna()].copy()
gastro_bld["zone_id"] = gastro_bld["zone_id"].astype("int32")

print(f"POIs kept for zone-level aggregation: {len(gastro_bld)}")

# -------------------------------------------------------------
# 5. Match each gastronomy POI to a building (within + nearest) with threshold
# -------------------------------------------------------------
gastro_bld["bldg_area_m2"] = np.nan
gastro_bld["dist_to_bldg_m"] = np.nan
gastro_bld["bldg_match"] = "none"

left_within = gpd.GeoDataFrame(
    gastro_bld[["poi_id", "geometry"]].copy(),
    geometry="geometry",
    crs=gastro_bld.crs,
)
right_bld = gpd.GeoDataFrame(
    geb_sel.copy(),
    geometry="geometry",
    crs=geb_sel.crs,
)

# 5.1 WITHIN
join_within = gpd.sjoin(left_within, right_bld, how="left", predicate="within")

within_info = (
    join_within.dropna(subset=["bldg_area_m2"])
    .groupby("poi_id")[["bldg_area_m2"]]
    .first()
)

mask_bld_within = gastro_bld["poi_id"].isin(within_info.index)
gastro_bld.loc[mask_bld_within, "bldg_area_m2"] = gastro_bld.loc[mask_bld_within, "poi_id"].map(within_info["bldg_area_m2"])
gastro_bld.loc[mask_bld_within, "bldg_match"] = "within"

n_bld_within = int(mask_bld_within.sum())
print(f"\nBuilding match by WITHIN: {n_bld_within}")

# 5.2 NEAREST (<= 50 m)
missing_bld_mask = gastro_bld["bldg_area_m2"].isna()
n_missing_bld = int(missing_bld_mask.sum())
print(f"POIs without building after WITHIN: {n_missing_bld}")

n_bld_nearest = 0

if n_missing_bld > 0:
    left_nearest = gpd.GeoDataFrame(
        gastro_bld.loc[missing_bld_mask, ["poi_id", "geometry"]].copy(),
        geometry="geometry",
        crs=gastro_bld.crs,
    )

    nearest = gpd.sjoin_nearest(
        left_nearest,
        right_bld,
        how="left",
        distance_col="dist_to_bldg_m",
    )

    # threshold
    nearest.loc[nearest["dist_to_bldg_m"] > NEAREST_BLDG_MAX_M, "bldg_area_m2"] = np.nan

    nearest_info = (
        nearest.dropna(subset=["bldg_area_m2"])
        .groupby("poi_id")[["bldg_area_m2", "dist_to_bldg_m"]]
        .first()
    )

    mask_bld_near = gastro_bld["poi_id"].isin(nearest_info.index)
    gastro_bld.loc[mask_bld_near, "bldg_area_m2"] = gastro_bld.loc[mask_bld_near, "poi_id"].map(nearest_info["bldg_area_m2"])
    gastro_bld.loc[mask_bld_near, "dist_to_bldg_m"] = gastro_bld.loc[mask_bld_near, "poi_id"].map(nearest_info["dist_to_bldg_m"]).astype("float64")
    gastro_bld.loc[mask_bld_near, "bldg_match"] = "nearest"

    n_bld_nearest = int(mask_bld_near.sum())

print(f"Building match by NEAREST (<= {NEAREST_BLDG_MAX_M:.0f} m): {n_bld_nearest}")
print(f"POIs with any building match (within+nearest): {int(gastro_bld['bldg_area_m2'].notna().sum())} / {len(gastro_bld)}")

# -------------------------------------------------------------
# 6. Filter building areas and prepare for imputation
# -------------------------------------------------------------
# We only trust swissTLM3D for CH; for LIE/Enk. we force NaN
mask_special = gastro_bld["N_KT"].isin(SPECIAL_LABELS)
gastro_bld.loc[mask_special, "bldg_area_m2"] = np.nan
gastro_bld.loc[mask_special, "bldg_match"] = "none"
gastro_bld.loc[mask_special, "dist_to_bldg_m"] = np.nan

print(
    "\nCH POIs with raw building area before range filter: ",
    int(gastro_bld.loc[~mask_special, "bldg_area_m2"].notna().sum()),
)

print("\nDescriptive stats of matched building areas (all, pre-filter):")
print(gastro_bld["bldg_area_m2"].dropna().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

valid_range = (gastro_bld["bldg_area_m2"] >= AREA_MIN_M2) & (gastro_bld["bldg_area_m2"] <= AREA_MAX_M2)
gastro_bld.loc[~valid_range, "bldg_area_m2"] = np.nan
gastro_bld.loc[gastro_bld["bldg_area_m2"].isna(), "bldg_match"] = "none"

print(f"\nPOIs with building area in [{AREA_MIN_M2:.0f},{AREA_MAX_M2:.0f}] m²: {int(gastro_bld['bldg_area_m2'].notna().sum())}")

# -------------------------------------------------------------
# 7. Lognormal imputation for missing areas (by AMENITY only; fallback global)
#    Output variable name: F1_gastr_area
# -------------------------------------------------------------
valid_ch = gastro_bld[(~mask_special) & gastro_bld["bldg_area_m2"].notna()].copy()
print(f"\nValid CH POIs with matched building area (post-filter): {len(valid_ch)}")

if len(valid_ch) > 0:
    valid_ch["log_area"] = np.log(valid_ch["bldg_area_m2"])

    mu_all = valid_ch["log_area"].mean()
    sigma_all = valid_ch["log_area"].std(ddof=1)

    amen_stats = (
        valid_ch.groupby("amenity")["log_area"]
        .agg(["mean", "std", "count"])
        .rename(columns={"mean": "mu", "std": "sigma"})
    )

    mu_dict = amen_stats["mu"].to_dict()
    sigma_dict = amen_stats["sigma"].to_dict()
    count_dict = amen_stats["count"].to_dict()

    print("\nPer-amenity log-area stats (top by count):")
    print(amen_stats.sort_values("count", ascending=False).head(15))
else:
    mu_all = np.log(100.0)
    sigma_all = 0.5
    mu_dict, sigma_dict, count_dict = {}, {}, {}

def sample_area_for_row(row, max_tries=30):
    """
    Sample a plausible building area in [AREA_MIN_M2, AREA_MAX_M2] for a given amenity.
    Uses per-amenity lognormal parameters if there are enough observations,
    otherwise falls back to the global distribution.
    """
    a = row.get("amenity", None)
    mu = mu_all
    sigma = sigma_all

    if a in count_dict and count_dict[a] >= MIN_AMENITY_N:
        if not np.isnan(mu_dict[a]) and not np.isnan(sigma_dict[a]) and sigma_dict[a] > 0:
            mu = mu_dict[a]
            sigma = sigma_dict[a]

    for _ in range(max_tries):
        val = np.random.lognormal(mean=mu, sigma=sigma)
        if AREA_MIN_M2 <= val <= AREA_MAX_M2:
            return float(val)

    return float(np.clip(np.exp(mu), AREA_MIN_M2, AREA_MAX_M2))

# Final per-POI area proxy
gastro_bld["F1_gastr_area"] = gastro_bld["bldg_area_m2"]

to_impute = gastro_bld["F1_gastr_area"].isna()
n_to_impute = int(to_impute.sum())
print(f"\nPOIs needing area imputation (including LIE/Enk.): {n_to_impute}")

gastro_bld.loc[to_impute, "F1_gastr_area"] = gastro_bld.loc[to_impute].apply(
    sample_area_for_row,
    axis=1
)

print("Remaining NaN in F1_gastr_area:", int(gastro_bld["F1_gastr_area"].isna().sum()))

# Summary counters you asked for
n_zone_total = len(gdf_gastro)
n_zone_kept = len(gastro_bld)

n_bld_within_final  = int((gastro_bld["bldg_match"] == "within").sum())
n_bld_nearest_final = int((gastro_bld["bldg_match"] == "nearest").sum())
n_bld_none_final    = int((gastro_bld["bldg_match"] == "none").sum())
n_imputed_final     = int((gastro_bld["bldg_area_m2"].isna()).sum())
n_observed_final    = int((gastro_bld["bldg_area_m2"].notna()).sum())

print("\n--- SUMMARY ---")
print(f"Zone assignment: within={n_zone_within}, nearest(<= {NEAREST_TZ_MAX_M:.0f}m)={n_zone_nearest}, dropped(no zone)={n_zone_total - n_zone_kept}")
print(f"Building match: within={n_bld_within_final}, nearest(<= {NEAREST_BLDG_MAX_M:.0f}m)={n_bld_nearest_final}, none={n_bld_none_final}")
print(f"Area proxy: observed={n_observed_final}, imputed={n_imputed_final}, total={len(gastro_bld)}")

# -------------------------------------------------------------
# 7b. Plot distributions (observed vs imputed vs all)
# -------------------------------------------------------------
obs_vals = gastro_bld.loc[gastro_bld["bldg_area_m2"].notna(), "F1_gastr_area"].astype(float)
imp_vals = gastro_bld.loc[gastro_bld["bldg_area_m2"].isna(), "F1_gastr_area"].astype(float)
all_vals = gastro_bld["F1_gastr_area"].astype(float)

print("\nDistribution checks (m²):")
print(f"  Observed: n={len(obs_vals)} | mean={obs_vals.mean():.1f} | median={obs_vals.median():.1f}")
print(f"  Imputed : n={len(imp_vals)} | mean={imp_vals.mean():.1f} | median={imp_vals.median():.1f}")
print(f"  All     : n={len(all_vals)} | mean={all_vals.mean():.1f} | median={all_vals.median():.1f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(all_vals, bins=60)
ax.set_xlabel("F1_gastr_area (m²)")
ax.set_ylabel("Count")
ax.set_title("Distribution of gastronomy area proxy (all POIs)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(obs_vals, bins=60)
ax.set_xlabel("F1_gastr_area (m²)")
ax.set_ylabel("Count")
ax.set_title("Distribution of gastronomy area proxy (observed from swissTLM3D)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(imp_vals, bins=60)
ax.set_xlabel("F1_gastr_area (m²)")
ax.set_ylabel("Count")
ax.set_title("Distribution of gastronomy area proxy (imputed)")
plt.tight_layout()
plt.show()

# Optional: log-scale plot (often more readable)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log(all_vals), bins=60)
ax.set_xlabel("log(F1_gastr_area)")
ax.set_ylabel("Count")
ax.set_title("Distribution of gastronomy area proxy (log scale, all POIs)")
plt.tight_layout()
plt.show()

# -------------------------------------------------------------
# 8. Aggregate to zones: F1_gastr_area_total
# -------------------------------------------------------------
area_zone = (
    gastro_bld
    .dropna(subset=["zone_id"])
    .groupby("zone_id", dropna=True)["F1_gastr_area"]
    .sum()
    .reset_index(name="F1_gastr_area_total")
)

tz = tz.merge(area_zone, on="zone_id", how="left")
tz["F1_gastr_area_total"] = tz["F1_gastr_area_total"].fillna(0).astype("float64")

print("\nExample of F1_gastr_area_total:")
print(tz[["zone_id", "F1_gastr_area_total"]].head())

# -------------------------------------------------------------
# 9. Save updated TZ and plot map
# -------------------------------------------------------------
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\nUpdated TZ.gpkg with F1_gastr_area_total saved to: {TZ_PATH}")

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
tz.plot(
    column="F1_gastr_area_total",
    legend=True,
    ax=ax,
    edgecolor="black",
    linewidth=0.1
)
ax.set_title("F1_gastr_area_total – Total gastronomy footprint proxy per NPVM zone", fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()


### Validation against official gastronomy FTE (STATENT)

As an additional consistency check for the gastronomy floor-area indicator, I use **STATENT 2023** workplace data:

- source: `Jobs_2023/Jobs_2023.gpkg`
- variable: `B0856VZA` — full-time equivalents (FTE) in **gastronomy** (NOGA 56).

The STATENT workplaces are stored as points and assigned to NPVM traffic zones as follows:

1. **Swiss zones only as targets**  
   - I use `TZ/TZ.gpkg` and restrict the join targets to zones with  
     \(`N_KT` ∉ {`"LIE"`, `"Enk."`}\), i.e. I exclude Liechtenstein and the two foreign enclaves.

2. **Two-step spatial assignment**  
   - first, a `within` join between STATENT points and NPVM polygons,  
   - then, for remaining points, a `sjoin_nearest` to the closest NPVM zone (still excluding LIE / Enk.).

3. **Aggregation to zones**  
   - for each zone \( j \), I sum `B0856VZA` over all assigned workplaces:  

     $$
     F1^{\text{gastr,FTE}}(j)
     =
     \sum_{w \in j} \text{B0856VZA}_w
     $$

   - the resulting zonal totals are stored in the NPVM layer as  
     `F1_gastr_FTE` (full-time equivalents in gastronomy).

I then compare `F1_gastr_FTE` with the previously defined floor-area indicator `F1_gastr_area_total` (gastronomy-related building area in m²) on **Swiss zones only** (excluding `N_KT = "LIE"` and `"Enk."`).

I report both Pearson and Spearman correlations:

- **All Swiss zones (including zeros)**  

  - Pearson (area vs FTE): **[value ≈ 0.605]**  
  - Spearman (area vs FTE): **[value ≈ 0.620]**

- **Only zones with both indicators strictly positive**  

  - Pearson (area vs FTE): **[value ≈ 0.603]**  
  - Spearman (area vs FTE): **[value ≈ 0.559]**

The positive and relatively high correlations indicate that the OSM-based gastronomy floor area is broadly consistent with the official employment intensity in the gastronomy sector, and can therefore be considered a reasonable proxy for the **supply** of gastronomic opportunities at the zonal level.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

# ==========================================================
# Paths
# ==========================================================
TZ_PATH    = "TZ/TZ.gpkg"
JOBS_PATH  = "Jobs_2023/Jobs_2023.gpkg"
SPECIAL    = ["LIE", "Enk."]   # Liechtenstein + enclaves

# ==========================================================
# 1. Load zones and check F1_gastr_area_total
# ==========================================================
tz = gpd.read_file(TZ_PATH)
print(f"Traffic zones: {len(tz)}")

if tz.crs is None or tz.crs.to_epsg() != 2056:
    tz = tz.to_crs(2056)
    print("Reprojected TZ to EPSG:2056")

if "F1_gastr_area_total" not in tz.columns:
    raise ValueError("Column 'F1_gastr_area_total' not found in TZ.gpkg. "
                     "Run the gastronomy-area script first.")

# ==========================================================
# 2. Load STATENT jobs and prepare FTE in gastronomy
# ==========================================================
jobs = gpd.read_file(JOBS_PATH)
print(f"Jobs points: {len(jobs)}")

if jobs.crs is None or jobs.crs.to_epsg() != 2056:
    jobs = jobs.to_crs(2056)
    print("Reprojected Jobs_2023 to EPSG:2056")

if "B0856VZA" not in jobs.columns:
    raise ValueError("Column 'B0856VZA' (gastronomy FTE) not found in Jobs_2023.gpkg")

# Ensure numeric and clean codes (negative or NaN → 0)
jobs["B0856VZA"] = pd.to_numeric(jobs["B0856VZA"], errors="coerce")
jobs["B0856VZA"] = jobs["B0856VZA"].where(jobs["B0856VZA"] >= 0, 0)

# ==========================================================
# 3. Assign jobs to Swiss zones (exclude LIE / Enk. as targets)
# ==========================================================
tz_ch = tz[~tz["N_KT"].isin(SPECIAL)].copy()

# First: within
join_jobs = gpd.sjoin(
    jobs,
    tz_ch[["zone_id", "geometry"]],
    how="left",
    predicate="within"
)

missing = join_jobs[join_jobs["zone_id"].isna()].copy()
print(f"Jobs points without zone after 'within': {len(missing)}")

# Then: nearest for remaining points
if not missing.empty:
    cols_to_drop = [c for c in ["zone_id", "index_right", "index_left"] if c in missing.columns]
    missing_clean = missing.drop(columns=cols_to_drop, errors="ignore")

    nearest = gpd.sjoin_nearest(
        missing_clean,
        tz_ch[["zone_id", "geometry"]],
        how="left",
        distance_col="dist_to_zone"
    )

    # Align back using missing index
    join_jobs.loc[missing.index, "zone_id"] = nearest["zone_id"].values

# Drop any residual jobs without zone
join_jobs = join_jobs[join_jobs["zone_id"].notna()].copy()
join_jobs["zone_id"] = join_jobs["zone_id"].astype("int32")

# ==========================================================
# 4. Aggregate gastronomy FTE (B0856VZA) per zone
# ==========================================================
fte_zone = (
    join_jobs
    .groupby("zone_id", dropna=True)["B0856VZA"]
    .sum(min_count=1)
    .reset_index(name="F1_gastr_FTE")
)

fte_zone["F1_gastr_FTE"] = fte_zone["F1_gastr_FTE"].fillna(0).astype("float64")

print("\nExample F1_gastr_FTE per zone:")
print(fte_zone.head())

# Merge into TZ (note: LIE / Enk. will get NaN → 0)
tz = tz.merge(fte_zone, on="zone_id", how="left")
tz["F1_gastr_FTE"] = tz["F1_gastr_FTE"].fillna(0).astype("float64")

# Save updated zones
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\nUpdated TZ.gpkg with F1_gastr_FTE saved to: {TZ_PATH}")

# ==========================================================
# 5. Correlation: F1_gastr_area_total vs F1_gastr_FTE
#    (Swiss zones only, exclude LIE / Enk.)
# ==========================================================
df_ch = tz[~tz["N_KT"].isin(SPECIAL)].copy()

# Make sure the two variables exist and are numeric
for col in ["F1_gastr_area_total", "F1_gastr_FTE"]:
    df_ch[col] = pd.to_numeric(df_ch[col], errors="coerce").fillna(0)

# (a) Correlation on all Swiss zones (including zeros)
pearson_all = df_ch["F1_gastr_area_total"].corr(df_ch["F1_gastr_FTE"], method="pearson")
spearman_all = df_ch["F1_gastr_area_total"].corr(df_ch["F1_gastr_FTE"], method="spearman")

print("\nCorrelation on ALL Swiss zones (including zeros):")
print(f"  Pearson  (area vs FTE): {pearson_all:.3f}")
print(f"  Spearman (area vs FTE): {spearman_all:.3f}")

# (b) Correlation on zones with both indicators > 0
mask_nz = (df_ch["F1_gastr_area_total"] > 0) & (df_ch["F1_gastr_FTE"] > 0)
df_nz = df_ch.loc[mask_nz].copy()
print(f"\nSwiss zones with F1_gastr_area_total > 0 and F1_gastr_FTE > 0: {len(df_nz)}")

pearson_nz = df_nz["F1_gastr_area_total"].corr(df_nz["F1_gastr_FTE"], method="pearson")
spearman_nz = df_nz["F1_gastr_area_total"].corr(df_nz["F1_gastr_FTE"], method="spearman")

print("Correlation on non-zero Swiss zones:")
print(f"  Pearson  (area vs FTE): {pearson_nz:.3f}")
print(f"  Spearman (area vs FTE): {spearman_nz:.3f}")


## F2: Visit to relatives

### F2 – Residential population and households

#### Data source

As a proxy for the “visit to relatives / friends” potential in each NPVM traffic zone, I use the **STATPOP 2024** hectare grid from the Federal Statistical Office (FSO). STATPOP is an annual register-based statistic of the resident population and private households in Switzerland, based on administrative registers.

The dataset provides **100×100 m grid cells** (LV95, EPSG:2056) with:

- total resident population,  
- number of private households,  
- additional identification and quality variables (e.g. commune, year, HPI).

Source (STATPOP geodata, FSO):  
https://www.bfs.admin.ch/bfs/de/home/dienstleistungen/geostat/geodaten-bundesstatistik/gebaeude-wohnungen-haushalte-personen/bevoelkerung-haushalte-ab-2010.html  

From the original delivery I retain in particular:

- `BBTOT` (or `BTOT`): total resident population per grid cell,  
- `HPTOT`: total number of private households per grid cell,  
- plus basic meta-information (reference years, commune IDs, quality indicator `HPI`).

The STATPOP grid is stored as a point layer in `Population_2024/Population_2024.gpkg`, where each point corresponds to the **south-west corner** of a 100×100 m cell in **EPSG:2056**.

---

#### Target zones

Population and household counts are aggregated to the official **NPVM 2017 traffic zones**, stored in `TZ/TZ.gpkg`. This layer contains one polygon per traffic zone in Switzerland (including lakes), with:

- a unique zone identifier `zone_id`,  
- attributes for commune (`ID_Gem`, `N_Gem`),  
- canton (`ID_KT`, `N_KT`),  
- settlement type, agglomeration and labour-market region,  
- geometry in **LV95 / EPSG:2056**.

`zone_id` is the primary spatial key used throughout the modelling framework.

---

#### Spatial assignment of STATPOP cells to traffic zones

1. **CRS harmonisation**

   Both layers (`TZ.gpkg` and `Population_2024.gpkg`) are checked and, if necessary, reprojected to **EPSG:2056** to ensure that:

   - spatial predicates (`within`, `nearest`) are geometrically correct,  
   - areas and distances are interpreted in metric units.

2. **Special cantonal labels (LIE / Enk.)**

   The NPVM layer includes **13 traffic zones** with special cantonal labels:

   - `N_KT = "LIE"`: 11 municipalities of **Liechtenstein** (Vaduz, Balzers, Schaan, etc.),  
   - `N_KT = "Enk."`: two foreign enclaves/exclaves fully surrounded by Swiss territory:  
     - **Büsingen am Hochrhein** (in canton Schaffhausen),  
     - **Campione d’Italia** (in canton Ticino).

   Since STATPOP is a **Swiss** official statistic, these areas are **not** consistently covered by the STATPOP grid. For this reason:

   - the 13 corresponding NPVM zones are **excluded as targets** in the spatial join,  
   - they are treated separately with manual demographic information (see below).

3. **Two-step spatial join (Swiss zones only)**

   Let `tz_ch` be the subset of NPVM zones with `N_KT` not in `{ "LIE", "Enk." }`.  
   The assignment of STATPOP cells to zones proceeds in two steps:

   - **Step 1 – “within” join**  
     Each STATPOP point is first joined to the NPVM zone whose polygon **contains** the point  
     (`predicate="within"`, target = `tz_ch`).  
     This captures all grid cells whose south-west corner lies strictly inside a zone polygon.

   - **Step 2 – nearest-neighbour join for remaining points**  
     Some points may lie exactly on boundaries or just outside due to the way cell vertices are defined.  
     For all cells still without an assigned `zone_id` after Step 1, a second join is performed using  
     `sjoin_nearest` on `tz_ch`, which:

     - attaches each remaining grid cell to the **geographically nearest** NPVM zone,  
     - stores the distance to the matched zone as an auxiliary field (`dist_to_zone`).

   After these two steps, all STATPOP cells belonging to Swiss territory are assigned to a single Swiss NPVM zone.

4. **Dropping residual unmatched cells**

   Any residual cells without `zone_id` (if present) are dropped from the aggregation. In practice this number should be zero or very small and has negligible impact on zonal totals.

---

#### Aggregation to zone level

For every STATPOP grid cell with a valid `zone_id`, I compute zonal totals:

- **Total resident population**

  $$
  F2^{\text{pop,tot}}(j)
  = \sum_{c \in j} \text{BBTOT}_c
  $$

  where \(c\) indexes STATPOP cells assigned to zone \(j\).

- **Total number of households**

  $$
  F2^{\text{hh,tot}}(j)
  = \sum_{c \in j} \text{HPTOT}_c
  $$

The result is a table with one row per Swiss NPVM zone and two new columns:

- `F2_pop_total`: total resident population in zone \( j \),  
- `F2_hh_total`: total number of private households in zone \( j \).

Zones without any STATPOP cells receive a value of **0** for both variables.  
These zonal totals are merged back into the NPVM layer and saved in `TZ/TZ.gpkg`, so that population and households are available as attributes for all subsequent analyses.

From a conceptual point of view, `F2_pop_total` and `F2_hh_total` measure the **resident base** in each zone, which serves as a natural proxy for the **potential presence of relatives and acquaintances**. Zones with more residents (and more households) are more likely to host relatives and friends that can be visited, and therefore tend to be more attractive for “visit to relatives” trips.

---

#### Treatment of Liechtenstein and foreign enclaves

The 13 NPVM zones corresponding to **Liechtenstein** and the enclaves/exclaves (Büsingen am Hochrhein, Campione d’Italia) raise a specific modelling issue:

- They are **explicitly represented** in the NPVM 2017 zoning and in the SIMBA MOBi model,  
- they appear in the **MTMC** travel survey as potential origins and destinations,  
- but they are **not** fully covered by Swiss federal statistics such as STATPOP.

If I relied only on STATPOP, these zones would end up with **zero or missing** population/household counts, which is clearly inconsistent with their actual demographic situation and would bias the F2 factor.

To resolve this, I proceed as follows.

##### 1. Manual population totals

For each of the 13 municipalities, I take the official number of residents from external sources:

- **Liechtenstein municipalities**:  
  population figures from *Liechtenstein in Zahlen 2025*, Amt für Statistik:  
  https://www.statistikportal.li/statistikportal/publications/102-liechtenstein-in-zahlen/2025/01/1/102.2025.01.1_02_liechtenstein-in-zahlen-2025.pdf  

- **Büsingen am Hochrhein**:  
  demographic data from:  
  https://ugeo.urbistat.com/AdminStat/de/de/demografia/dati-sintesi/busingen-am-hochrhein/20175339/4  

- **Campione d’Italia**:  
  demographic time series from:  
  https://www.tuttitalia.it/lombardia/21-campione-d-italia/statistiche/popolazione-andamento-demografico/  

These values are written directly into `F2_pop_total` for the corresponding NPVM zones.

##### 2. Approximating household totals via the Swiss persons-per-household ratio

Because there is no harmonised household statistic at the same spatial resolution for Liechtenstein, Büsingen and Campione, I approximate the number of households using the **Swiss average persons-per-household ratio** derived from STATPOP:

$$
r_{\text{CH}} =
\frac{\sum_{j \in \text{CH}} F2^{\text{pop,tot}}(j)}
     {\sum_{j \in \text{CH}} F2^{\text{hh,tot}}(j)}
$$

where the sums run over all **Swiss** NPVM zones (excluding `N_KT = "LIE"` and `"Enk."`).

For each special zone \( j \), the inferred household total is:


$$
F2^{\text{hh,tot}}(j) \approx
\text{round}\!\left(
\frac{F2^{\text{pop,tot}}(j)}{r_{\text{CH}}}
\right)
$$

i.e. the population of the municipality divided by the Swiss average persons-per-household, rounded to the nearest integer.

This implies the working assumption that **average household size** in Liechtenstein, Büsingen and Campione is similar to the Swiss average. While this is a simplification, it provides a **transparent and reproducible** way to obtain plausible `F2_hh_total` values in the absence of directly compatible data.

##### 3. Conceptual note

This manual correction illustrates a more general issue:

> NPVM zones and the MTMC travel survey cover certain foreign territories (Liechtenstein and enclaves),  
> but not all predictor variables can be sourced from Swiss federal statistics for these areas.

For the F2 factor (residential population and households), I resolve this by combining:

- external official demographic statistics, and  
- a simple, documented assumption on household size.

For other factors (jobs, land use, amenities, etc.), similar decisions may be necessary, or these zones might need to be examined in sensitivity analyses (e.g. checking robustness when they are excluded or treated separately).

---

#### Current status of F2

- `F2_pop_total` and `F2_hh_total` are now available for **all NPVM 2017 traffic zones**, including Liechtenstein and the two enclaves/exclaves.
- At this stage, these variables are kept as **absolute totals**.  
  In later steps, I will derive:
  - relative measures (e.g. per km², per building, per leisure facility),
  - transformations (e.g. log, z-scores),
  - and possibly age-specific indicators (e.g. elderly population)  
  to refine the “visit to relatives” factor before integrating it into the discrete choice models.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os

# ==========================================================
# 1. Paths
# ==========================================================
TZ_PATH  = "TZ/TZ.gpkg"
POP_PATH = "Population_2024/Population_2024.gpkg"

# ==========================================================
# 2. Read data
# ==========================================================
tz = gpd.read_file(TZ_PATH)           # NPVM zones (polygons, with zone_id)
pop = gpd.read_file(POP_PATH)        # STATPOP 2024 points (100x100 m grid cells)

# Ensure both layers are in LV95 / EPSG:2056
if tz.crs is None or tz.crs.to_epsg() != 2056:
    tz = tz.to_crs(2056)
    print("Reprojected TZ to EPSG:2056")

if pop.crs is None or pop.crs.to_epsg() != 2056:
    pop = pop.to_crs(2056)
    print("Reprojected POP to EPSG:2056")

# Cantonal label for special zones (Liechtenstein + exclaves)
SPECIAL_LABELS = ["LIE", "Enk."]

# ==========================================================
# 3. Choose population and household variables
# ==========================================================
# Some STATPOP versions use BTOT instead of BBTOT → handle both
if "BBTOT" in pop.columns:
    POP_COL = "BBTOT"
elif "BTOT" in pop.columns:
    POP_COL = "BTOT"
else:
    raise ValueError("Population column (BBTOT/BTOT) not found in Population_2024.gpkg")

HH_COL = "HPTOT"
if HH_COL not in pop.columns:
    raise ValueError("Household column HPTOT not found in Population_2024.gpkg")

# Cast to numeric for safety
for c in [POP_COL, HH_COL]:
    pop[c] = pd.to_numeric(pop[c], errors="coerce")

# ==========================================================
# 4. Spatial join: assign each STATPOP point to a *Swiss* NPVM zone
#    - we EXCLUDE Liechtenstein and exclaves as join targets
#    - first using 'within'
#    - then remaining points using 'nearest'
# ==========================================================

# Use only Swiss zones as targets for the join
tz_ch = tz[~tz["N_KT"].isin(SPECIAL_LABELS)].copy()

# First join: within
join_within = gpd.sjoin(
    pop,
    tz_ch[["zone_id", "geometry"]],
    how="left",
    predicate="within"
)

missing = join_within[join_within["zone_id"].isna()].copy()

if not missing.empty:
    # Drop any sjoin-related columns that could clash in the next join
    cols_to_drop = [c for c in ["zone_id", "index_right", "index_left"] if c in missing.columns]
    missing_clean = missing.drop(columns=cols_to_drop, errors="ignore")

    # Nearest-zone assignment (again: only Swiss zones as targets)
    nearest = gpd.sjoin_nearest(
        missing_clean,
        tz_ch[["zone_id", "geometry"]],
        how="left",
        distance_col="dist_to_zone"
    )

    # Update zone_id in the main joined dataframe
    join_within.loc[missing.index, "zone_id"] = nearest["zone_id"].values

# Final check
n_missing_final = join_within["zone_id"].isna().sum()

# Optionally drop them if any remain (should be zero or very few)
join_within = join_within[join_within["zone_id"].notna()].copy()

# ==========================================================
# 5. Aggregate to zone level: F2_pop_total and F2_hh_total (Swiss zones)
# ==========================================================
agg = (
    join_within
    .groupby("zone_id", dropna=True)
    .agg(
        F2_pop_total=(POP_COL, "sum"),
        F2_hh_total=(HH_COL,  "sum")
    )
    .reset_index()
)

# If some zones have no STATPOP points, they will get NaN → replace with 0
agg["F2_pop_total"] = agg["F2_pop_total"].fillna(0)
agg["F2_hh_total"]  = agg["F2_hh_total"].fillna(0)

# ==========================================================
# 6. Merge with TZ
# ==========================================================
tz = tz.merge(agg, on="zone_id", how="left")

tz["F2_pop_total"] = tz["F2_pop_total"].fillna(0)
tz["F2_hh_total"]  = tz["F2_hh_total"].fillna(0)

# ==========================================================
# 7. Compute average persons-per-household ratio for Switzerland
#    (excluding Liechtenstein + exclaves)
# ==========================================================
mask_ch = ~tz["N_KT"].isin(SPECIAL_LABELS)

total_pop_ch = tz.loc[mask_ch, "F2_pop_total"].sum()
total_hh_ch  = tz.loc[mask_ch, "F2_hh_total"].sum()

ratio_ch = total_pop_ch / total_hh_ch

# ==========================================================
# 8. Manually set population and households for Liechtenstein + exclaves
# ==========================================================
manual_pop = {
    # Liechtenstein (11 municipalities)
    "Vaduz":              5826,
    "Eschen":             4607,
    "Triesen":            5532,
    "Mauren":             4589,
    "Balzers":            4747,
    "Gamprin":            1768,
    "Triesenberg":        2671,
    "Ruggell":            2523,
    "Schaan":             6109,
    "Schellenberg":       1155,
    "Planken":             488,
    # Exclaves
    "Büsingen":           1444,
    "Campione d'Italia":  1757,
}

for gem_name, pop_val in manual_pop.items():
    sel = tz["N_Gem"] == gem_name
    if sel.any():
        tz.loc[sel, "F2_pop_total"] = pop_val
        # households = nearest integer: pop / ratio_ch
        tz.loc[sel, "F2_hh_total"] = np.round(pop_val / ratio_ch).astype(int)
    else:
        print(f"Warning: municipality name '{gem_name}' not found in TZ layer.")

# ==========================================================
# 9. Save back to TZ.gpkg (overwrite)
# ==========================================================
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

tz.plot(
    column="F2_pop_total",
    legend=True,
    ax=ax,
    edgecolor="black",
    linewidth=0.1
)

ax.set_title("F2_pop_total – Resident population per NPVM zone", fontsize=12)
ax.set_axis_off()

plt.tight_layout()
plt.show()



fig, ax = plt.subplots(1, 1, figsize=(8, 8))

tz.plot(
    column="F2_hh_total",
    legend=True,
    ax=ax,
    edgecolor="black",
    linewidth=0.1
)

ax.set_title("F2_hh_total – Households per NPVM zone", fontsize=12)
ax.set_axis_off()

plt.tight_layout()
plt.show()



## F5 – Sport opportunities

For the **sport** dimension of the attractivity index \(F5\), I derive facility-based sport opportunities from **OpenStreetMap (OSM)** and use **authoritative datasets** (swissTLM3D and STATENT 2023) for **validation and auxiliary proxies** (official facility counts, employment, and surface estimates). The indicators capture:

- how many sport opportunities exist in each zone,  
- how diverse they are,  
- how many establishments and employees they represent,  
- and (as a proxy) how much surface is dedicated to sport.

---

### Source datasets

- **Traffic zones**
  - `TZ/TZ.gpkg`
  - NPVM Traffic Zones 2017, CRS: EPSG:2056 (CH1903+ / LV95)  
  - Key field: `zone_id` (aggregation unit for all F5 indicators).

- **OSM sport POIs**
  - Source: OSM exports via Overpass using sport-related tags.
  - All geometries converted to **points** (centroids for lines/polygons).
  - Reprojected to **EPSG:2056** and stored as `OSM/Sport/Sport.gpkg`.

- **Official sport facilities – swissTLM3D (for comparison and surface proxy)**
  - `swissTLM3D_2025/Bauten/Sport.gpkg` – sport facility polygons.  
  - `swissTLM3D_2025/Bauten/SportLine.gpkg` – linear sport installations (e.g. ski jumps, bob tracks).  
  - `swissTLM3D_2025/Areal/LeisureUse.gpkg` – functional leisure areas, filtered for:
    - `Sportplatzareal`
    - `Schwimmbadareal`
    - `Golfplatzareal`
    - `Pferderennbahnareal`

- **Jobs 2023 (STATENT)**  
  - `Jobs_2023/Jobs_2023.gpkg`  
  - Workplaces and employment points (100 × 100 m), CRS: EPSG:2056.  
  - Used columns for **NOGA 93** (Sports, amusement and recreation):
    - `B0893VZA` – full-time equivalents (FTE)  
    - `B0893AS` – number of establishments (Arbeitsstätten)

---

### Pre-processing and consistency check (OSM vs official)

1. Merge official swissTLM3D sources (`Sport`, `SportLine`, `LeisureUse`) and compute centroids.  
2. Perform a **nearest-neighbour join** to OSM sport POIs (max 100 m) to assess coverage.
3. Results:
   - Total official objects: 21,972  
   - Matched with OSM within 100 m: 18,301 (≈ 83%)  
   - Unmatched: 3,671 (≈ 17%)  
   - High coverage for `Sportplatz`, lower for `Golfplatzareal` and `Scheibenstand`.

> Note: Official data are used **only for comparison/validation** and for deriving a **surface proxy** (see below).  
> No official-based corrections are applied to OSM sport POI counts (e.g. no substitution of swimming facilities).

---

### Indicators

#### 1) OSM-based sport availability

Each OSM sport POI is assigned to a traffic zone \(j\).

- **Count of sport POIs per zone**
  \[
  F5_{\text{sport\_count}}(j)
  = \sum_{i \in \text{sport POIs}} \mathbf{1}\{\text{POI } i \in \text{zone } j\}
  \]

- **Diversity of sport categories**
  - Each feature is assigned **one category** (precedence `leisure` > `building` to avoid double counting).
  - Recoding for near-duplicates:
    - `fitness_centre` and `fitness_station` → **fitness**
    - `swimming_area` and `paddling_pool` → **swimming**
  - Resulting variable: `F5_sport_div` (number of distinct categories present in zone \(j\)).

#### 2) Official swissTLM3D facilities (comparison layer)

All official polygons/lines/areals are converted to centroids and assigned to traffic zones.

- **Count of official facilities per zone**
  \[
  F5_{\text{sport\_offTLM\_count}}(j)
  = \sum_{k \in \text{official facilities}} \mathbf{1}\{\text{facility } k \in \text{zone } j\}
  \]

#### 3) STATENT establishments and employment

STATENT 2023 points for **NOGA 93** are aggregated by traffic zone:

- `F5_sport_off_count` = sum of `B0893AS` (establishments)  
- `F5_sport_FTE` = sum of `B0893VZA` (full-time equivalents)

#### 4) Sport area (surface proxy)

To approximate the physical **extent** of sport infrastructure:

- Each OSM sport POI is matched to the nearest swissTLM3D sport polygon (within 100 m) to transfer an area proxy.  
- Missing areas are **imputed** using log-normal distributions fitted on valid matches by OSM category.  
- Aggregated by zone as:
  \[
  F5_{\text{sport\_area\_total}}(j)
  = \sum_{i \in \text{sport POIs in zone } j} \text{area}_i
  \]

---

### Summary of F5 variables stored in `TZ.gpkg`

| Variable | Description |
|---|---|
| `F5_sport_count` | Count of OSM sport POIs per zone |
| `F5_sport_div` | Diversity of sport categories per zone (OSM; precedence + recoding) |
| `F5_sport_offTLM_count` | Count of official sport facilities (swissTLM3D; comparison) |
| `F5_sport_off_count` | Number of sport establishments (STATENT 2023; NOGA 93) |
| `F5_sport_FTE` | Full-time equivalent jobs in sport sector (STATENT 2023; NOGA 93) |
| `F5_sport_area_total` | Total estimated sport area per zone (m²; OSM + official area proxy) |

All indicators are computed at the NPVM traffic-zone level and stored in `TZ/TZ.gpkg` as part of the attractivity factor construction.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

try:
    import fiona  # to read all layers in gpkg
except Exception:
    fiona = None

# ==========================================================
# PATHS
# ==========================================================
TZ_PATH            = Path("TZ/TZ.gpkg")
OSM_SPORT_GPKG     = Path("OSM/Sport/Sport.gpkg")   # Overpass export (may have multiple layers / mixed geoms)

SPORT_BAUTEN_PATH  = Path("swissTLM3D_2025/Bauten/Sport.gpkg")
SPORTLINE_PATH     = Path("swissTLM3D_2025/Bauten/SportLine.gpkg")
LEISURE_AREAL_PATH = Path("swissTLM3D_2025/Areal/LeisureUse.gpkg")

SPORT_AREAL_TYPES = [
    "Sportplatzareal",
    "Schwimmbadareal",
    "Golfplatzareal",
    "Pferderennbahnareal",
]

SPECIAL_LABELS = ["LIE", "Enk."]   # Liechtenstein + enclaves label in N_KT
ZONE_ID_FIELD  = "zone_id"
TARGET_EPSG    = 2056

# ==========================================================
# Helpers
# ==========================================================
def ensure_epsg(gdf: gpd.GeoDataFrame, epsg: int, label: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError(f"❌ {label}: CRS is missing.")
    if gdf.crs.to_epsg() != epsg:
        gdf = gdf.to_crs(epsg)
    return gdf

def clean_geom(gdf: gpd.GeoDataFrame, label: str) -> gpd.GeoDataFrame:
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def read_gpkg_all_layers(path: Path) -> gpd.GeoDataFrame:
    if fiona is None:
        return gpd.read_file(path)

    layers = fiona.listlayers(path)
    if len(layers) == 1:
        return gpd.read_file(path, layer=layers[0])

    gdfs = []
    print(f"📚 Layers in {path.name}: {layers}")
    for lyr in layers:
        g = gpd.read_file(path, layer=lyr)
        g["__src_layer"] = lyr
        gdfs.append(g)

    out = pd.concat(gdfs, ignore_index=True)
    return gpd.GeoDataFrame(out, geometry="geometry", crs=gdfs[0].crs)

def pointify(gdf: gpd.GeoDataFrame, label: str) -> gpd.GeoDataFrame:
    gdf = clean_geom(gdf, label)
    gtypes = gdf.geometry.geom_type.value_counts(dropna=False).to_dict()
    print(f"📊 {label} geom types (before): {gtypes}")

    is_point = gdf.geometry.geom_type == "Point"
    gdf.loc[~is_point, "geometry"] = gdf.loc[~is_point, "geometry"].centroid

    bad = gdf.geometry.geom_type != "Point"
    if bad.any():
        gdf.loc[bad, "geometry"] = gdf.loc[bad, "geometry"].representative_point()

    gtypes2 = gdf.geometry.geom_type.value_counts(dropna=False).to_dict()
    print(f"📊 {label} geom types (after): {gtypes2}")
    return gdf

def norm_str(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip().str.lower()
    x = x.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
    return x

def print_vc(title: str, s: pd.Series, topn: int = 999) -> None:
    vc = s.value_counts(dropna=True).head(topn)
    print(f"\n{title}")
    print(vc.to_string())

# ==========================================================
# Category logic (for diversity)
# ==========================================================
# Merge categories for diversity:
LEISURE_RECODE = {
    "fitness_centre": "fitness",
    "fitness_station": "fitness",
    "swimming_area": "swimming",
    "paddling_pool": "swimming",
}
# (optional) if you want to unify other near-duplicates later, add here.

# Keep only “sport-ish” building values for diversity (avoid generic building=yes and noise)
ALLOWED_BUILDING = {
    "pavilion", "riding_hall", "sports_hall", "sports_centre", "stadium"
}
# If building value is not in this set, we treat it as NA for category purposes.

def build_primary_category(df: pd.DataFrame) -> pd.Series:
    """
    One category per feature, used for diversity:
    - precedence: leisure if present, else building
    - leisure recode: fitness_* -> fitness; swimming_* -> swimming
    - building filtered to allowed sport building values
    """
    leisure = norm_str(df["leisure"]) if "leisure" in df.columns else pd.Series(pd.NA, index=df.index)
    building = norm_str(df["building"]) if "building" in df.columns else pd.Series(pd.NA, index=df.index)

    leisure_rec = leisure.replace(LEISURE_RECODE)
    building_f = building.where(building.isin(ALLOWED_BUILDING), pd.NA)

    # precedence
    cat = leisure_rec.copy()
    cat = cat.fillna(building_f)

    return cat

# ==========================================================
# 1) Load traffic zones
# ==========================================================
print("📂 Loading traffic zones...")
tz = gpd.read_file(TZ_PATH)
tz = ensure_epsg(tz, TARGET_EPSG, "TZ")
if ZONE_ID_FIELD not in tz.columns:
    raise ValueError(f"❌ Column '{ZONE_ID_FIELD}' not found in TZ.gpkg.")
tz_min = tz[[ZONE_ID_FIELD, "geometry"]].copy()
print(f"   Traffic zones: {len(tz)}")

# ==========================================================
# 2) Load OSM sport, pointify, build categories
# ==========================================================
print("\n📂 Loading OSM Sport (all layers if needed)...")
osm_raw = read_gpkg_all_layers(OSM_SPORT_GPKG)
osm_raw = ensure_epsg(osm_raw, TARGET_EPSG, "OSM Sport (raw)")
osm_raw = clean_geom(osm_raw, "OSM Sport (raw)")
print(f"   OSM raw features: {len(osm_raw)}")

# ensure required columns exist
for col in ["name", "leisure", "building", "sport", "access"]:
    if col not in osm_raw.columns:
        osm_raw[col] = pd.NA

# pointify (OSM is almost all points already, but keep robust)
osm = pointify(osm_raw, "OSM Sport")

# print raw counts (before recode/filter)
print_vc("🏷️  OSM leisure value_counts (raw):", norm_str(osm["leisure"]))
print_vc("🏗️  OSM building value_counts (raw):", norm_str(osm["building"]))

# recoded leisure (for diversity purposes)
leisure_rec = norm_str(osm["leisure"]).replace(LEISURE_RECODE)
print_vc("🏷️  OSM leisure value_counts (recoded for diversity):", leisure_rec)

# filtered building (for diversity purposes)
building_f = norm_str(osm["building"]).where(norm_str(osm["building"]).isin(ALLOWED_BUILDING), pd.NA)
print_vc("🏗️  OSM building value_counts (filtered for diversity):", building_f)

# primary category (precedence leisure > building)
osm["sport_cat"] = build_primary_category(osm)
print_vc("🏷️  Final sport_cat used for diversity (precedence leisure>building, recoded):", osm["sport_cat"])

print(f"\n✅ OSM sport features used (pointified): {len(osm)}")
print(f"   Features with a non-missing sport_cat (for diversity): {osm['sport_cat'].notna().sum()}")

# ==========================================================
# 3) Assign OSM sport POIs to TZ and compute F5_sport_count / F5_sport_div
# ==========================================================
print("\n📌 Assigning OSM sport features to traffic zones (sjoin_nearest)...")
sj = gpd.sjoin_nearest(
    osm[["geometry", "sport_cat"]].copy(),
    tz_min,
    how="left",
    distance_col="dist_to_zone",
)

sj = sj.dropna(subset=[ZONE_ID_FIELD]).copy()
sj[ZONE_ID_FIELD] = sj[ZONE_ID_FIELD].astype("int32")
print(f"   OSM features assigned to a zone: {len(sj)}")

# count: all features
sport_count = sj.groupby(ZONE_ID_FIELD).size().rename("F5_sport_count")

# diversity: unique categories per zone (excluding NA)
sport_div = (
    sj.dropna(subset=["sport_cat"])
      .groupby(ZONE_ID_FIELD)["sport_cat"]
      .nunique()
      .rename("F5_sport_div")
)

tz = tz.set_index(ZONE_ID_FIELD)
tz["F5_sport_count"] = sport_count
tz["F5_sport_div"] = sport_div
tz[["F5_sport_count", "F5_sport_div"]] = tz[["F5_sport_count", "F5_sport_div"]].fillna(0).astype("int32")
tz = tz.reset_index()

print("\n🧮 Example F5_sport_* (OSM-only):")
print(tz[[ZONE_ID_FIELD, "F5_sport_count", "F5_sport_div"]].head())

# ==========================================================
# 4) OFFICIAL comparison: Bauten + SportLine + LeisureUse (4 types) -> F5_sport_offTLM_count
# ==========================================================
print("\n🏟️ Loading OFFICIAL sport facilities (Bauten + SportLine + Areal) for comparison...")

sport_bauten = gpd.read_file(SPORT_BAUTEN_PATH)
sport_bauten = ensure_epsg(sport_bauten, TARGET_EPSG, "OFF Sport (Bauten)")
sport_bauten = clean_geom(sport_bauten, "OFF Sport (Bauten)")
print(f"   OFF Sport (Bauten): {len(sport_bauten)}")

sport_line = gpd.read_file(SPORTLINE_PATH)
sport_line = ensure_epsg(sport_line, TARGET_EPSG, "OFF SportLine")
sport_line = clean_geom(sport_line, "OFF SportLine")
print(f"   OFF SportLine: {len(sport_line)}")

leisure_use = gpd.read_file(LEISURE_AREAL_PATH)
leisure_use = ensure_epsg(leisure_use, TARGET_EPSG, "OFF LeisureUse")
leisure_use = clean_geom(leisure_use, "OFF LeisureUse")
if "objektart" not in leisure_use.columns:
    raise ValueError("❌ OFF LeisureUse must contain column 'objektart'.")

leisure_sport = leisure_use[leisure_use["objektart"].isin(SPORT_AREAL_TYPES)].copy()
leisure_sport = clean_geom(leisure_sport, "OFF LeisureUse (selected sport areals)")
print(f"   OFF LeisureUse (4 sport areal types): {len(leisure_sport)}")

# minimal union + centroid for matching
def min_off(gdf, src):
    out = gdf[["geometry"]].copy()
    out["src"] = src
    return out

off_all = pd.concat([
    min_off(sport_bauten, "Bauten_Sport"),
    min_off(sport_line, "Bauten_SportLine"),
    min_off(leisure_sport, "Areal_LeisureUse"),
], ignore_index=True)
off_all = gpd.GeoDataFrame(off_all, geometry="geometry", crs=sport_bauten.crs)
print(f"   Total OFFICIAL objects (combined): {len(off_all)}")

off_cent = off_all.copy()
off_cent["geometry"] = off_cent.geometry.centroid
off_cent = clean_geom(off_cent, "OFF centroids")

join_off = gpd.sjoin_nearest(
    off_cent,
    tz_min,
    how="left",
    distance_col="dist_to_zone",
)

join_off = join_off.dropna(subset=[ZONE_ID_FIELD]).copy()
join_off[ZONE_ID_FIELD] = join_off[ZONE_ID_FIELD].astype("int32")

off_count = join_off.groupby(ZONE_ID_FIELD).size().rename("F5_sport_offTLM_count")

tz = tz.set_index(ZONE_ID_FIELD)
tz["F5_sport_offTLM_count"] = off_count
tz["F5_sport_offTLM_count"] = tz["F5_sport_offTLM_count"].fillna(0).astype("int32")
tz = tz.reset_index()

print("\n🧮 Example F5_sport_offTLM_count:")
print(tz[[ZONE_ID_FIELD, "F5_sport_offTLM_count"]].head())

# ==========================================================
# 5) Correlation OSM vs OFFICIAL (Swiss only)
# ==========================================================
if "N_KT" in tz.columns:
    df_ch = tz[~tz["N_KT"].isin(SPECIAL_LABELS)].copy()
else:
    df_ch = tz.copy()

print(f"\n🇨🇭 Swiss zones used for correlation (excluding LIE/Enk. if present): {len(df_ch)}")

pearson_all = df_ch["F5_sport_count"].corr(df_ch["F5_sport_offTLM_count"], method="pearson")
spearman_all = df_ch["F5_sport_count"].corr(df_ch["F5_sport_offTLM_count"], method="spearman")

print("\n📊 Correlation: F5_sport_count (OSM-only) vs F5_sport_offTLM_count (OFFICIAL)")
print(f"   Pearson : {pearson_all:.3f}")
print(f"   Spearman: {spearman_all:.3f}")

mask_nz = (df_ch["F5_sport_count"] > 0) & (df_ch["F5_sport_offTLM_count"] > 0)
df_nz = df_ch.loc[mask_nz].copy()
print(f"   Zones with both > 0: {len(df_nz)}")

if len(df_nz) > 1:
    pearson_nz = df_nz["F5_sport_count"].corr(df_nz["F5_sport_offTLM_count"], method="pearson")
    spearman_nz = df_nz["F5_sport_count"].corr(df_nz["F5_sport_offTLM_count"], method="spearman")
    print("\n📊 Correlation (only zones with both > 0):")
    print(f"   Pearson : {pearson_nz:.3f}")
    print(f"   Spearman: {spearman_nz:.3f}")

# ==========================================================
# 6) Save back to TZ.gpkg
# ==========================================================
os.makedirs(TZ_PATH.parent, exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with F5_sport_count, F5_sport_div, F5_sport_offTLM_count to: {TZ_PATH}")


# ==========================================================
# 7) Quick plots (optional)
# ==========================================================
import matplotlib.pyplot as plt

# --- helper: plot a TZ choropleth ---
def plot_tz(tz_gdf, col, title):
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    tz_gdf.plot(column=col, legend=True, ax=ax, edgecolor="black", linewidth=0.1)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

# Maps
plot_tz(tz, "F5_sport_count", "F5_sport_count — sport POIs (OSM-only)")
plot_tz(tz, "F5_sport_div", "F5_sport_div — category diversity (precedence leisure>building; recoded)")
plot_tz(tz, "F5_sport_offTLM_count", "F5_sport_offTLM_count — OFFICIAL sport facilities (comparison)")

# --- Scatter: OSM vs OFFICIAL (Swiss-only if N_KT present) ---
if "N_KT" in tz.columns:
    df_scatter = tz[~tz["N_KT"].isin(SPECIAL_LABELS)].copy()
else:
    df_scatter = tz.copy()

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(df_scatter["F5_sport_offTLM_count"], df_scatter["F5_sport_count"], s=8)
ax.set_xlabel("F5_sport_offTLM_count (OFFICIAL)")
ax.set_ylabel("F5_sport_count (OSM-only)")
ax.set_title("Sport supply: OSM vs OFFICIAL (zone level)")
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==========================================================
# PATHS
# ==========================================================
TZ_PATH   = "TZ/TZ.gpkg"
JOBS_PATH = "Jobs_2023/Jobs_2023.gpkg"

ZONE_ID_FIELD = "zone_id"
SPECIAL_LABELS = ["LIE", "Enk."]  # Swiss-only filter (if N_KT exists)

TARGET_EPSG = 2056

# ==========================================================
# CRS helper
# ==========================================================
def ensure_epsg_2056(gdf, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label} has no CRS defined.")
    if gdf.crs.to_epsg() != TARGET_EPSG:
        gdf = gdf.to_crs(TARGET_EPSG)
        print(f"↪ Reprojected {label} to EPSG:{TARGET_EPSG}")
    return gdf

# ==========================================================
# 1) Load TZ
# ==========================================================
print("📂 Loading traffic zones...")
tz = gpd.read_file(TZ_PATH)
print(f"   Traffic zones: {len(tz)}")

tz = ensure_epsg_2056(tz, "TZ")

if ZONE_ID_FIELD not in tz.columns:
    raise ValueError(f"❌ Column '{ZONE_ID_FIELD}' not found in TZ.gpkg.")

# NOTE: you requested F5 naming (sport factor)
if "F5_sport_count" not in tz.columns:
    raise ValueError(
        "❌ 'F5_sport_count' not found in TZ.gpkg. "
        "Run the script that creates F5_sport_count from OSM first."
    )

tz_min = tz[[ZONE_ID_FIELD, "geometry"]].copy()

# ==========================================================
# 2) Load Jobs_2023
# ==========================================================
print("\n📂 Loading Jobs_2023...")
jobs = gpd.read_file(JOBS_PATH)
print(f"   Jobs_2023 points: {len(jobs)}")

jobs = ensure_epsg_2056(jobs, "Jobs_2023")

print("   Available columns in Jobs_2023:")
print(list(jobs.columns))

# ==========================================================
# 3) Detect sector-93 columns (sport/amusement/recreation)
#    - FTE: B0893*VZA
#    - Establishments: another B0893* (Arbeitsstätten)
# ==========================================================
cand_93 = [c for c in jobs.columns if isinstance(c, str) and c.startswith("B0893")]
print("\n🔍 Candidate columns for sector 93 (start with 'B0893'):")
print(cand_93)

if not cand_93:
    raise ValueError(
        "❌ No columns starting with 'B0893' found in Jobs_2023. "
        "Check column names in your STATENT file."
    )

fte_cols = [c for c in cand_93 if str(c).upper().endswith("VZA")]
if len(fte_cols) == 0:
    raise ValueError(
        "❌ No B0893* column ending with 'VZA' found (FTE). "
        "Set the FTE column manually."
    )
if len(fte_cols) > 1:
    print("⚠️ Multiple B0893*VZA columns found; using the first one.")

FTE_COL = fte_cols[0]

count_candidates = [c for c in cand_93 if c != FTE_COL]
if not count_candidates:
    raise ValueError(
        "❌ No second B0893* column found for establishments (Arbeitsstätten). "
        "Set the COUNT column manually."
    )

COUNT_COL = count_candidates[0]

print(f"\n✅ FTE column (Vollzeitäquivalente): {FTE_COL}")
print(f"✅ COUNT column (Arbeitsstätten):     {COUNT_COL}")

jobs[FTE_COL] = pd.to_numeric(jobs[FTE_COL], errors="coerce").fillna(0)
jobs[COUNT_COL] = pd.to_numeric(jobs[COUNT_COL], errors="coerce").fillna(0)

# ==========================================================
# 4) Spatial join Jobs -> TZ (within)
# ==========================================================
print("\n📌 Assigning Jobs points to traffic zones (sjoin, predicate='within')...")

jobs_sj = gpd.sjoin(
    jobs[[FTE_COL, COUNT_COL, "geometry"]],
    tz_min,
    how="left",
    predicate="within",
)

missing_zones = int(jobs_sj[ZONE_ID_FIELD].isna().sum())
print(f"   Jobs points without assigned zone: {missing_zones}")

jobs_sj = jobs_sj[~jobs_sj[ZONE_ID_FIELD].isna()].copy()
jobs_sj[ZONE_ID_FIELD] = jobs_sj[ZONE_ID_FIELD].astype("int32")

print(f"   Jobs points with assigned zone: {len(jobs_sj)}")

# ==========================================================
# 5) Aggregate per zone
#    - F5_sport_FTE       = sum(FTE_COL)
#    - F5_sport_off_count = sum(COUNT_COL)
# ==========================================================
agg = (
    jobs_sj
    .groupby(ZONE_ID_FIELD)
    .agg(
        F5_sport_FTE=("{}".format(FTE_COL), "sum"),
        F5_sport_off_count=("{}".format(COUNT_COL), "sum"),
    )
    .fillna(0)
)

# Merge into TZ (do not drop other attributes; overwrite only these 2 if exist)
for c in ["F5_sport_FTE", "F5_sport_off_count"]:
    if c in tz.columns:
        tz = tz.drop(columns=[c])

tz = tz.merge(agg.reset_index(), on=ZONE_ID_FIELD, how="left")
tz["F5_sport_FTE"] = tz["F5_sport_FTE"].fillna(0).astype("float64")
tz["F5_sport_off_count"] = tz["F5_sport_off_count"].fillna(0).astype("float64")

print("\n🧮 Example F5_sport_FTE / F5_sport_off_count:")
print(tz[[ZONE_ID_FIELD, "F5_sport_FTE", "F5_sport_off_count"]].head())

# ==========================================================
# 6) Correlation: Jobs vs OSM (Swiss-only if possible)
# ==========================================================
if "N_KT" in tz.columns:
    df_ch = tz[~tz["N_KT"].isin(SPECIAL_LABELS)].copy()
else:
    df_ch = tz.copy()

print(f"\n🇨🇭 Swiss zones (excluding LIE/Enk. if available): {len(df_ch)}")

pearson_all = df_ch["F5_sport_off_count"].corr(df_ch["F5_sport_count"], method="pearson")
spearman_all = df_ch["F5_sport_off_count"].corr(df_ch["F5_sport_count"], method="spearman")

print("\n📊 Correlation F5_sport_off_count (Jobs, establishments) vs F5_sport_count (OSM):")
print(f"   Pearson : {pearson_all:.3f}")
print(f"   Spearman: {spearman_all:.3f}")

mask_nz = (df_ch["F5_sport_off_count"] > 0) & (df_ch["F5_sport_count"] > 0)
df_nz = df_ch.loc[mask_nz].copy()
print(f"   Zones with both > 0: {len(df_nz)}")

if len(df_nz) > 1:
    pearson_nz = df_nz["F5_sport_off_count"].corr(df_nz["F5_sport_count"], method="pearson")
    spearman_nz = df_nz["F5_sport_off_count"].corr(df_nz["F5_sport_count"], method="spearman")
    print("\n📊 Correlation (only zones with both > 0):")
    print(f"   Pearson : {pearson_nz:.3f}")
    print(f"   Spearman: {spearman_nz:.3f}")

# ==========================================================
# 7) Save TZ
# ==========================================================
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with F5_sport_FTE and F5_sport_off_count to: {TZ_PATH}")

# Optional: quick scatter
plt.figure(figsize=(6, 6))
plt.scatter(df_ch["F5_sport_count"], df_ch["F5_sport_off_count"], s=5, alpha=0.5)
plt.xlabel("F5_sport_count (OSM)")
plt.ylabel("F5_sport_off_count (Jobs, establishments)")
plt.title("F5_sport_off_count vs F5_sport_count")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# -------------------------------------------------------------
# Paths
# -------------------------------------------------------------
TZ_PATH            = "TZ/TZ.gpkg"
JOBS_PATH          = "Jobs_2023/Jobs_2023.gpkg"

SPORT_OSM_PATH     = "OSM/Sport/Sport.gpkg"  # OSM sport layer (may contain mixed geometries)
SPORT_BAUTEN_PATH  = "swissTLM3D_2025/Bauten/Sport.gpkg"
LEISURE_AREAL_PATH = "swissTLM3D_2025/Areal/LeisureUse.gpkg"

ZONE_ID_FIELD   = "zone_id"
KT_FIELD        = "N_KT"
SPECIAL_LABELS  = ["LIE", "Enk", "Enk."]   # Liechtenstein + enclaves labels

# Official sport areal classes of interest (LeisureUse.gpkg)
SPORT_AREAL_TYPES = [
    "Sportplatzareal",
    "Schwimmbadareal",
    "Golfplatzareal",
    "Pferderennbahnareal",
]

TARGET_EPSG = 2056
MAX_MATCH_DIST_M = 100

# Reproducible RNG for imputation
RNG = np.random.default_rng(123)

# -------------------------------------------------------------
# Helpers
# -------------------------------------------------------------
def ensure_epsg_2056(gdf, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label}: CRS is missing.")
    if gdf.crs.to_epsg() != TARGET_EPSG:
        gdf = gdf.to_crs(TARGET_EPSG)
        print(f"↪ Reprojected {label} to EPSG:{TARGET_EPSG}")
    return gdf

def clean_geom(gdf, label):
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def pointify(gdf, label):
    """Convert all geometries to points (centroid for non-points)."""
    gdf = clean_geom(gdf, label)
    is_point = gdf.geometry.geom_type == "Point"
    gdf.loc[~is_point, "geometry"] = gdf.loc[~is_point, "geometry"].centroid
    bad = gdf.geometry.geom_type != "Point"
    if bad.any():
        gdf.loc[bad, "geometry"] = gdf.loc[bad, "geometry"].representative_point()
    return gdf

def norm_tag(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"": np.nan, "nan": np.nan, "none": np.nan, "null": np.nan})
    )

def corr_report(df, x, y, label):
    valid = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    print(f"\n📈 Correlation — {label}")
    print(f"   Zones used: {len(valid)}")
    if len(valid) < 2:
        print("   ⚠ Not enough observations.")
        return
    if valid[x].std() == 0 or valid[y].std() == 0:
        print("   ⚠ std==0 -> correlation not meaningful.")
        return
    pear = valid[x].corr(valid[y], method="pearson")
    spear = valid[x].corr(valid[y], method="spearman")
    print(f"   Std {x}: {valid[x].std():.6f} | Std {y}: {valid[y].std():.6f}")
    print(f"   Pearson : {pear:.3f}")
    print(f"   Spearman: {spear:.3f}")

# -------------------------------------------------------------
# 1) Load traffic zones
# -------------------------------------------------------------
print("📂 Loading traffic zones...")
tz = gpd.read_file(TZ_PATH)
print(f"   Traffic zones: {len(tz)}")
tz = ensure_epsg_2056(tz, "TZ")
tz = clean_geom(tz, "TZ")

if ZONE_ID_FIELD not in tz.columns:
    raise ValueError(f"❌ Column '{ZONE_ID_FIELD}' not found in TZ.gpkg.")
if KT_FIELD not in tz.columns:
    print("⚠️ 'N_KT' not found in TZ. Swiss-only filtering will be skipped.")
tz_min = tz[[ZONE_ID_FIELD, "geometry"]].copy()

# -------------------------------------------------------------
# 2) Load OSM sport (points) and ensure it is point-only
# -------------------------------------------------------------
print("\n📂 Loading OSM Sport...")
sport_osm = gpd.read_file(SPORT_OSM_PATH)
print(f"   OSM Sport (raw): {len(sport_osm)}")
sport_osm = ensure_epsg_2056(sport_osm, "OSM Sport")
sport_osm = pointify(sport_osm, "OSM Sport (pointified)")

# Create stable POI id
sport_osm = sport_osm.reset_index(drop=True)
sport_osm["poi_id"] = sport_osm.index.astype("int64")

# Ensure columns exist
for col in ["name", "leisure", "building", "sport"]:
    if col not in sport_osm.columns:
        sport_osm[col] = pd.NA

# -------------------------------------------------------------
# 3) Load OFFICIAL sport geometries and compute official areas
# -------------------------------------------------------------
print("\n📂 Loading OFFICIAL Sport (Bauten)...")
sport_bauten = gpd.read_file(SPORT_BAUTEN_PATH)
sport_bauten = ensure_epsg_2056(sport_bauten, "OFF Sport (Bauten)")
sport_bauten = clean_geom(sport_bauten, "OFF Sport (Bauten)")
print(f"   OFF Sport (Bauten): {len(sport_bauten)}")

print("\n📂 Loading OFFICIAL LeisureUse (Areal)...")
leisure_use = gpd.read_file(LEISURE_AREAL_PATH)
leisure_use = ensure_epsg_2056(leisure_use, "OFF LeisureUse")
leisure_use = clean_geom(leisure_use, "OFF LeisureUse")
print(f"   OFF LeisureUse: {len(leisure_use)}")

if "objektart" not in leisure_use.columns:
    raise ValueError("❌ LeisureUse must contain column 'objektart'.")

leisure_sport = leisure_use[leisure_use["objektart"].isin(SPORT_AREAL_TYPES)].copy()
leisure_sport = clean_geom(leisure_sport, "OFF LeisureUse (sport areals)")
print(f"   OFF LeisureUse sport areals (4 types): {len(leisure_sport)}")

# Compute official area (m²) and combine OFF sources
sport_bauten_off = sport_bauten.copy()
sport_bauten_off["area_off_m2"] = sport_bauten_off.geometry.area.astype("float64")
sport_bauten_off["src_off"] = "Bauten"
sport_bauten_off = sport_bauten_off[["objektart", "src_off", "area_off_m2", "geometry"]].copy()

leisure_sport_off = leisure_sport.copy()
leisure_sport_off["area_off_m2"] = leisure_sport_off.geometry.area.astype("float64")
leisure_sport_off["src_off"] = "Areal"
leisure_sport_off = leisure_sport_off[["objektart", "src_off", "area_off_m2", "geometry"]].copy()

sport_off_all = pd.concat([sport_bauten_off, leisure_sport_off], ignore_index=True)
sport_off_all = gpd.GeoDataFrame(sport_off_all, geometry="geometry", crs=sport_bauten_off.crs)

print(f"\n✅ Total OFFICIAL objects (Bauten + selected Areal): {len(sport_off_all)}")

# -------------------------------------------------------------
# 4) Match OSM -> OFFICIAL to transfer area_off_m2 (within + nearest <= 100m)
# -------------------------------------------------------------
sport_osm["area_off_m2"] = np.nan
sport_osm["src_off"] = pd.NA
sport_osm["obj_off"] = pd.NA

left_pts = sport_osm[["poi_id", "geometry"]].copy()
right_off = sport_off_all.copy()

# 4.1 WITHIN (point inside polygon)
print("\n🧭 Matching OSM within OFFICIAL polygons...")
join_within = gpd.sjoin(left_pts, right_off, how="left", predicate="within")

within_best = (
    join_within.dropna(subset=["area_off_m2"])
    .sort_values(["poi_id", "area_off_m2"], ascending=[True, False])
    .drop_duplicates(subset=["poi_id"])
    .set_index("poi_id")
)

sport_osm.loc[sport_osm["poi_id"].isin(within_best.index), "area_off_m2"] = (
    sport_osm.loc[sport_osm["poi_id"].isin(within_best.index), "poi_id"]
    .map(within_best["area_off_m2"]).to_numpy()
)
sport_osm.loc[sport_osm["poi_id"].isin(within_best.index), "src_off"] = (
    sport_osm.loc[sport_osm["poi_id"].isin(within_best.index), "poi_id"]
    .map(within_best["src_off"]).to_numpy()
)
sport_osm.loc[sport_osm["poi_id"].isin(within_best.index), "obj_off"] = (
    sport_osm.loc[sport_osm["poi_id"].isin(within_best.index), "poi_id"]
    .map(within_best["objektart"]).to_numpy()
)

n_within = int(sport_osm["area_off_m2"].notna().sum())
print(f"   OSM POIs with OFF area via within: {n_within}")

# 4.2 NEAREST for remaining (<= 100 m)
missing = sport_osm["area_off_m2"].isna()
osm_missing = sport_osm.loc[missing, ["poi_id", "geometry"]].copy()
print(f"   OSM POIs still missing after within: {len(osm_missing)}")

if len(osm_missing) > 0:
    print(f"🧭 Matching OSM -> OFFICIAL nearest (max {MAX_MATCH_DIST_M} m)...")
    nearest = gpd.sjoin_nearest(
        osm_missing,
        right_off,
        how="left",
        distance_col="dist_to_off_m",
        max_distance=MAX_MATCH_DIST_M,
    )
    nearest_best = (
        nearest.dropna(subset=["area_off_m2"])
        .sort_values(["poi_id", "area_off_m2"], ascending=[True, False])
        .drop_duplicates(subset=["poi_id"])
        .set_index("poi_id")
    )

    sport_osm.loc[sport_osm["poi_id"].isin(nearest_best.index), "area_off_m2"] = (
        sport_osm.loc[sport_osm["poi_id"].isin(nearest_best.index), "poi_id"]
        .map(nearest_best["area_off_m2"]).to_numpy()
    )
    sport_osm.loc[sport_osm["poi_id"].isin(nearest_best.index), "src_off"] = (
        sport_osm.loc[sport_osm["poi_id"].isin(nearest_best.index), "poi_id"]
        .map(nearest_best["src_off"]).to_numpy()
    )
    sport_osm.loc[sport_osm["poi_id"].isin(nearest_best.index), "obj_off"] = (
        sport_osm.loc[sport_osm["poi_id"].isin(nearest_best.index), "poi_id"]
        .map(nearest_best["objektart"]).to_numpy()
    )

n_any = int(sport_osm["area_off_m2"].notna().sum())
print(f"✅ OSM POIs with OFF area (within + nearest<=100m): {n_any} / {len(sport_osm)}")

# -------------------------------------------------------------
# 5) Assign zone_id to OSM sport POIs (within + nearest<=100m)
# -------------------------------------------------------------
print("\n📌 Assigning zone_id to OSM sport POIs...")
left_pts = sport_osm[["poi_id", "geometry"]].copy()
right_tz = tz[[ZONE_ID_FIELD, KT_FIELD, "geometry"]].copy() if KT_FIELD in tz.columns else tz[[ZONE_ID_FIELD, "geometry"]].copy()

join_tz = gpd.sjoin(left_pts, right_tz, how="left", predicate="within")
tz_first = (
    join_tz.dropna(subset=[ZONE_ID_FIELD])
    .groupby("poi_id")
    .first()
)

sport_osm = sport_osm.merge(
    tz_first[[ZONE_ID_FIELD] + ([KT_FIELD] if KT_FIELD in tz_first.columns else [])],
    on="poi_id",
    how="left"
)

miss_zone = sport_osm[ZONE_ID_FIELD].isna()
print(f"   Missing zone after within: {int(miss_zone.sum())}")

if miss_zone.any():
    miss_pts = sport_osm.loc[miss_zone, ["poi_id", "geometry"]].copy()
    near_tz = gpd.sjoin_nearest(
        miss_pts,
        right_tz,
        how="left",
        distance_col="dist_to_zone_m",
        max_distance=MAX_MATCH_DIST_M,
    )
    near_first = (
        near_tz.dropna(subset=[ZONE_ID_FIELD])
        .groupby("poi_id")
        .first()
    )
    sport_osm = sport_osm.drop(columns=[c for c in [ZONE_ID_FIELD, KT_FIELD] if c in sport_osm.columns], errors="ignore")
    sport_osm = sport_osm.merge(
        pd.concat([tz_first, near_first], axis=0)[[ZONE_ID_FIELD] + ([KT_FIELD] if KT_FIELD in tz_first.columns else [])]
        .reset_index(),
        on="poi_id",
        how="left"
    )

print(f"✅ OSM POIs with assigned zone_id (within+nearest<=100m): {int(sport_osm[ZONE_ID_FIELD].notna().sum())} / {len(sport_osm)}")

# -------------------------------------------------------------
# 6) Build OSM category label (for category-specific imputation)
# -------------------------------------------------------------
sport_osm["leisure_norm"]  = norm_tag(sport_osm["leisure"])
sport_osm["building_norm"] = norm_tag(sport_osm["building"])
sport_osm["sport_norm"]    = norm_tag(sport_osm["sport"])

def make_osm_cat(row):
    if pd.notna(row["leisure_norm"]):
        return f"L:{row['leisure_norm']}"
    if pd.notna(row["sport_norm"]):
        return f"S:{row['sport_norm']}"
    if pd.notna(row["building_norm"]):
        return f"B:{row['building_norm']}"
    return "UNK"

sport_osm["osm_cat"] = sport_osm.apply(make_osm_cat, axis=1)

# -------------------------------------------------------------
# 7) Clean OFF areas using CH-only quantile range (1%–99%)
# -------------------------------------------------------------
if KT_FIELD in sport_osm.columns:
    valid_off = sport_osm[(~sport_osm[KT_FIELD].isin(SPECIAL_LABELS)) & sport_osm["area_off_m2"].notna()].copy()
else:
    valid_off = sport_osm[sport_osm["area_off_m2"].notna()].copy()

print(f"\nValid CH POIs with area_off_m2 (pre-filter): {len(valid_off)}")

if len(valid_off) > 0:
    q01 = float(valid_off["area_off_m2"].quantile(0.01))
    q99 = float(valid_off["area_off_m2"].quantile(0.99))
    print(f"   Using reasonable range [q01,q99] = [{q01:.1f}, {q99:.1f}] m²")

    in_rng = (sport_osm["area_off_m2"] >= q01) & (sport_osm["area_off_m2"] <= q99)
    sport_osm.loc[~in_rng, "area_off_m2"] = np.nan

    if KT_FIELD in sport_osm.columns:
        valid_off = sport_osm[(~sport_osm[KT_FIELD].isin(SPECIAL_LABELS)) & sport_osm["area_off_m2"].notna()].copy()
    else:
        valid_off = sport_osm[sport_osm["area_off_m2"].notna()].copy()

print(f"Valid CH POIs with area_off_m2 (post-filter): {len(valid_off)}")

# -------------------------------------------------------------
# 8) Fit lognormal models (per osm_cat, fallback per OFF src, fallback global)
# -------------------------------------------------------------
if len(valid_off) > 0:
    valid_off["log_area"] = np.log(valid_off["area_off_m2"].astype(float))

    mu_all = float(valid_off["log_area"].mean())
    sigma_all = float(valid_off["log_area"].std(ddof=1))

    cat_stats = (
        valid_off.groupby("osm_cat")["log_area"]
        .agg(["mean", "std", "count"])
        .rename(columns={"mean": "mu_cat", "std": "sigma_cat"})
    )
    off_stats = (
        valid_off.groupby("src_off")["log_area"]
        .agg(["mean", "std", "count"])
        .rename(columns={"mean": "mu_off", "std": "sigma_off"})
    )

    mu_cat = cat_stats["mu_cat"].to_dict()
    sigma_cat = cat_stats["sigma_cat"].to_dict()
    count_cat = cat_stats["count"].to_dict()

    mu_off = off_stats["mu_off"].to_dict()
    sigma_off = off_stats["sigma_off"].to_dict()
    count_off = off_stats["count"].to_dict()

    valid_min = float(valid_off["area_off_m2"].min())
    valid_max = float(valid_off["area_off_m2"].max())
else:
    # very generic fallback
    mu_all = float(np.log(5_000.0))
    sigma_all = 1.0
    mu_cat = sigma_cat = count_cat = {}
    mu_off = sigma_off = count_off = {}
    valid_min, valid_max = 50.0, 1_000_000.0

def sample_area_for_row(row, max_tries=30, min_n_cat=30, min_n_off=30):
    """
    Sample a plausible sport area for an OSM POI:
      1) category-specific lognormal (osm_cat) if enough matches,
      2) fallback to OFF source distribution (Bauten/Areal) if enough matches,
      3) fallback global lognormal.
    Truncated to [valid_min, valid_max].
    """
    cat = row.get("osm_cat", "UNK")
    mu = mu_all
    sigma = sigma_all

    # (1) category-specific
    if cat in count_cat and count_cat[cat] >= min_n_cat:
        if pd.notna(mu_cat.get(cat)) and pd.notna(sigma_cat.get(cat)) and float(sigma_cat.get(cat, 0)) > 0:
            mu = float(mu_cat[cat])
            sigma = float(sigma_cat[cat])
    else:
        # (2) fallback by guessed OFF source (Bauten if building tag exists, else Areal if leisure tag exists)
        src_guess = None
        if pd.notna(row.get("building_norm", np.nan)):
            src_guess = "Bauten"
        elif pd.notna(row.get("leisure_norm", np.nan)):
            src_guess = "Areal"

        if src_guess in count_off and count_off[src_guess] >= min_n_off:
            if pd.notna(mu_off.get(src_guess)) and pd.notna(sigma_off.get(src_guess)) and float(sigma_off.get(src_guess, 0)) > 0:
                mu = float(mu_off[src_guess])
                sigma = float(sigma_off[src_guess])

    for _ in range(max_tries):
        val = float(RNG.lognormal(mean=mu, sigma=sigma))
        if valid_min <= val <= valid_max:
            return val

    return float(np.clip(np.exp(mu), valid_min, valid_max))

# -------------------------------------------------------------
# 9) Final per-POI area + aggregate to zones (F5 naming)
# -------------------------------------------------------------
sport_osm["F5_sport_area_m2"] = sport_osm["area_off_m2"]
to_impute = sport_osm["F5_sport_area_m2"].isna()
print(f"\n🧪 POIs requiring area imputation: {int(to_impute.sum())}")

sport_osm.loc[to_impute, "F5_sport_area_m2"] = sport_osm.loc[to_impute].apply(sample_area_for_row, axis=1)

print(f"Remaining NaN in F5_sport_area_m2: {int(sport_osm['F5_sport_area_m2'].isna().sum())}")
print("\nF5_sport_area_m2 distribution (all POIs):")
print(sport_osm["F5_sport_area_m2"].describe(percentiles=[0.1, 0.5, 0.9]))

area_zone = (
    sport_osm.dropna(subset=[ZONE_ID_FIELD])
    .groupby(ZONE_ID_FIELD)["F5_sport_area_m2"]
    .sum()
    .reset_index(name="F5_sport_area_total")
)

if "F5_sport_area_total" in tz.columns:
    tz = tz.drop(columns=["F5_sport_area_total"])
tz = tz.merge(area_zone, on=ZONE_ID_FIELD, how="left")
tz["F5_sport_area_total"] = tz["F5_sport_area_total"].fillna(0).astype("float64")

print("\n✅ Example F5_sport_area_total:")
print(tz[[ZONE_ID_FIELD, "F5_sport_area_total"]].head())

# -------------------------------------------------------------
# 10) Add STATENT sport FTE (B0893VZA) and correlate with area
# -------------------------------------------------------------
print(f"\n📂 Loading Jobs_2023: {JOBS_PATH}")
jobs = gpd.read_file(JOBS_PATH)
jobs = ensure_epsg_2056(jobs, "Jobs_2023")
jobs = clean_geom(jobs, "Jobs_2023")

# Find sport FTE column (B0893VZA)
cand_93 = [c for c in jobs.columns if isinstance(c, str) and c.startswith("B0893")]
fte_cols = [c for c in cand_93 if str(c).upper().endswith("VZA")]
if not fte_cols:
    raise ValueError("❌ No B0893*VZA column found in Jobs_2023 (sport FTE).")
FTE_COL = fte_cols[0]
jobs[FTE_COL] = pd.to_numeric(jobs[FTE_COL], errors="coerce").fillna(0)

# Spatial join jobs -> TZ and sum FTE per zone
jobs_z = gpd.sjoin(jobs[[FTE_COL, "geometry"]], tz_min, how="inner", predicate="within")
fte_zone = jobs_z.groupby(ZONE_ID_FIELD)[FTE_COL].sum().reset_index(name="F5_sport_FTE")

if "F5_sport_FTE" in tz.columns:
    tz = tz.drop(columns=["F5_sport_FTE"])
tz = tz.merge(fte_zone, on=ZONE_ID_FIELD, how="left")
tz["F5_sport_FTE"] = tz["F5_sport_FTE"].fillna(0).astype("float64")

# Correlation (Swiss-only if possible)
if KT_FIELD in tz.columns:
    tz_ch = tz[~tz[KT_FIELD].isin(SPECIAL_LABELS)].copy()
else:
    tz_ch = tz.copy()

corr_report(
    tz_ch,
    "F5_sport_area_total",
    "F5_sport_FTE",
    "F5_sport_area_total (OSM+OFF proxy) vs F5_sport_FTE (STATENT B0893VZA), Swiss-only"
)

# -------------------------------------------------------------
# 11) Save TZ + quick maps
# -------------------------------------------------------------
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with F5_sport_area_total and F5_sport_FTE to: {TZ_PATH}")

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
tz.plot(column="F5_sport_area_total", legend=True, ax=ax, edgecolor="black", linewidth=0.1)
ax.set_title("F5_sport_area_total — total estimated sport area per zone (m²)", fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
tz.plot(column="F5_sport_FTE", legend=True, ax=ax, edgecolor="black", linewidth=0.1)
ax.set_title("F5_sport_FTE — STATENT sport FTE per zone (B0893VZA)", fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.show()


## F5 — Active sport routes (length + diversity)

### Objective
Compute zone-level indicators for **outdoor active-sport route supply** by combining:
- official **hiking trails** (filtered to *mountain + alpine* classes),
- official **ski routes**,
- official **snowshoe routes**,
- **via ferrata** routes from OSM (line geometries).

Aggregate everything to **NPVM 2017 traffic zones** (`TZ/TZ.gpkg`, EPSG:2056) and store:
- `F5_sport_out_length` — total route length per zone (km),
- `F5_sport_out_div` — diversity of route types per zone.

### Inputs
- Traffic zones: `TZ/TZ.gpkg` (must contain `zone_id`, EPSG:2056).
- Hiking trails: `GEOADMIN/Hiking_2023.gpkg` (must contain `wanderwege`).
- Ski routes: `GEOADMIN/Skiroutes_2024/Ski.gpkg`.
- Snowshoe routes: `GEOADMIN/Snowshow_routes_2024/Snowshoe.gpkg`.
- Via ferrata (OSM): `OSM/Sport/ferrata.geojson` (assumed WGS84 if CRS missing).

### Key choices
- **Hiking filter:** keep only `wanderwege ∈ {Bergwanderweg, Alpinwanderweg}` (mountain + alpine routes).
- **Geometry handling:** only `LineString` / `MultiLineString` are retained.
- **Zone assignment:** use a **line–polygon intersection** (`overlay`), which splits lines along zone borders and prevents double counting.
- **Length measure:** compute in EPSG:2056 and store length in **kilometres**.
- **Diversity rule:** a route type counts for diversity if:
  - for `hiking`, `ski`, `snowshoe`: length in zone ≥ 0.1 km (100 m threshold),
  - for `via_ferrata`: counts if length > 0 (no 100 m threshold), because ferrata segments are often short but meaningful.

### Outputs
Saved into `TZ/TZ.gpkg`:
- `F5_sport_out_length` (float, km)
- `F5_sport_out_div` (int, number of route types)


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# ==========================================================
# Paths
# ==========================================================
TZ_PATH       = "TZ/TZ.gpkg"
HIKING_PATH   = "GEOADMIN/Hiking_2023.gpkg"
SKI_PATH      = "GEOADMIN/Skiroutes_2024/Ski.gpkg"
SNOWSHOE_PATH = "GEOADMIN/Snowshow_routes_2024/Snowshoe.gpkg"

FERRATA_GEOJSON = "OSM/Sport/ferrata.geojson"
FERRATA_GPKG    = "OSM/Sport/ferrata.gpkg"

TARGET_EPSG = 2056

# Diversity threshold for hiking/ski/snowshoe only (100 m)
MIN_LEN_KM_FOR_DIVERSITY = 0.1

# ==========================================================
# 1) Load traffic zones
# ==========================================================
print("📂 Loading traffic zones...")
tz = gpd.read_file(TZ_PATH)
print(f"   Traffic zones: {len(tz)}")

if tz.crs is None:
    raise ValueError("❌ TZ has no CRS.")
if tz.crs.to_epsg() != TARGET_EPSG:
    tz = tz.to_crs(TARGET_EPSG)
    print("   ↪ Reprojected TZ to EPSG:2056")

if "zone_id" not in tz.columns:
    raise ValueError("❌ 'zone_id' not found in TZ.gpkg.")

tz_min = tz[["zone_id", "geometry"]].copy()

# ==========================================================
# 2) Load line datasets (hiking, ski, snowshoe)
# ==========================================================
def load_to_2056(path, label):
    print(f"\n📂 Loading {label}: {path}")
    g = gpd.read_file(path)
    print(f"   {label}: {len(g)} geometries")
    if g.crs is None:
        raise ValueError(f"❌ {label} has no CRS.")
    if g.crs.to_epsg() != TARGET_EPSG:
        g = g.to_crs(TARGET_EPSG)
        print(f"   ↪ Reprojected {label} to EPSG:2056")
    return g

hiking = load_to_2056(HIKING_PATH,   "Hiking_2023")
ski    = load_to_2056(SKI_PATH,      "Ski_2024")
snow   = load_to_2056(SNOWSHOE_PATH, "Snowshoe_2024")

# ==========================================================
# 3) Keep only line geometries + basic cleaning
# ==========================================================
def keep_lines(g, label):
    g = g[g.geometry.notna()].copy()
    g = g[~g.geometry.is_empty].copy()
    g = g[g.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    print(f"   {label}: {len(g)} valid line features")
    return g

hiking = keep_lines(hiking, "Hiking (lines)")
ski    = keep_lines(ski,    "Ski (lines)")
snow   = keep_lines(snow,   "Snowshoe (lines)")

# ==========================================================
# 4) Hiking filter: keep only mountain + alpine hiking routes
# ==========================================================
if "wanderwege" not in hiking.columns:
    raise ValueError("❌ Hiking_2023.gpkg is missing column 'wanderwege' (adjust column name if needed).")

print("\n🥾 Filtering hiking to Bergwanderweg + Alpinwanderweg...")
mask_sport = hiking["wanderwege"].isin(["Wanderweg", "Bergwanderweg", "Alpinwanderweg"])
hiking_sport = hiking[mask_sport].copy()
print(f"   Hiking sport (mountain+alpine): {len(hiking_sport)} lines")

hiking_sport["sport_out_type"] = "hiking"
ski["sport_out_type"]          = "ski"
snow["sport_out_type"]         = "snowshoe"

# ==========================================================
# 4b) Via ferrata (OSM): geojson -> gpkg (geometry only) -> keep lines
# ==========================================================
print(f"\n🧗 Loading via ferrata (OSM): {FERRATA_GEOJSON}")
ferrata = gpd.read_file(FERRATA_GEOJSON)
print(f"   Via ferrata (raw): {len(ferrata)} geometries")

# If CRS missing, assume WGS84 (standard OSM)
if ferrata.crs is None:
    ferrata = ferrata.set_crs(4326)
    print("   CRS missing -> set to EPSG:4326 (assumed OSM WGS84)")

# Reproject to EPSG:2056 for metric lengths
ferrata_2056 = ferrata.to_crs(TARGET_EPSG) if ferrata.crs.to_epsg() != TARGET_EPSG else ferrata

# Save GPKG (geometry only to avoid OSM field issues)
os.makedirs(os.path.dirname(FERRATA_GPKG), exist_ok=True)
ferrata_geom_only = ferrata_2056[["geometry"]].copy()
ferrata_geom_only.to_file(FERRATA_GPKG, driver="GPKG")
print(f"💾 Saved via ferrata as GPKG (geometry only): {FERRATA_GPKG}")

# Read back (clean layer)
ferrata_2056 = gpd.read_file(FERRATA_GPKG)
if ferrata_2056.crs is None or ferrata_2056.crs.to_epsg() != TARGET_EPSG:
    ferrata_2056 = ferrata_2056.to_crs(TARGET_EPSG)

ferrata_2056 = keep_lines(ferrata_2056, "Via ferrata (lines)")
ferrata_2056["sport_out_type"] = "via_ferrata"

# ==========================================================
# 5) Combine all route layers into one GeoDataFrame
# ==========================================================
routes_all = gpd.GeoDataFrame(
    pd.concat(
        [
            hiking_sport[["sport_out_type", "geometry"]],
            ski[["sport_out_type", "geometry"]],
            snow[["sport_out_type", "geometry"]],
            ferrata_2056[["sport_out_type", "geometry"]],
        ],
        ignore_index=True
    ),
    crs=tz.crs
)

print(f"\n🧩 Total route features (hiking + ski + snowshoe + via_ferrata): {len(routes_all)}")

# ==========================================================
# 6) Zone intersection (overlay) -> split lines by zone borders, then compute length
# ==========================================================
print("\n🔀 Intersect routes with zones (overlay intersection)...")
sj = gpd.overlay(routes_all, tz_min, how="intersection")
print(f"   Segments after overlay: {len(sj)}")

sj = sj[sj.geometry.notna() & ~sj.geometry.is_empty].copy()
sj = sj[sj.geometry.type.isin(["LineString", "MultiLineString"])].copy()
print(f"   Valid line segments: {len(sj)}")

sj["len_km"] = sj.length / 1000.0

# ==========================================================
# 7) Aggregate:
#   - by (zone_id, type): sum km
#   - by zone: total length
#   - diversity with type-specific threshold
# ==========================================================
print("\n📏 Aggregating lengths by (zone_id, sport_out_type)...")
len_by_zone_type = (
    sj.groupby(["zone_id", "sport_out_type"])["len_km"]
      .sum()
      .reset_index()
)

# Total per zone
len_by_zone = (
    len_by_zone_type
    .groupby("zone_id")["len_km"]
    .sum()
    .reset_index(name="F5_sport_out_length")
)

# Diversity rule:
def counts_for_diversity(row):
    t = row["sport_out_type"]
    L = row["len_km"]
    if t == "via_ferrata":
        return L > 0.0
    return L >= MIN_LEN_KM_FOR_DIVERSITY

len_by_zone_type["count_for_div"] = len_by_zone_type.apply(counts_for_diversity, axis=1)

div_df = (
    len_by_zone_type[len_by_zone_type["count_for_div"]]
    .groupby("zone_id")["sport_out_type"]
    .nunique()
    .reset_index(name="F5_sport_out_div")
)

print("\n   Example F5_sport_out_length:")
print(len_by_zone.head())

print("\n   Example F5_sport_out_div:")
print(div_df.head())

# ==========================================================
# 8) Merge into TZ and fill NA with 0
# ==========================================================
print("\n🔗 Merging F5_* indicators into TZ...")

for c in ["F5_sport_out_length", "F5_sport_out_div"]:
    if c in tz.columns:
        tz = tz.drop(columns=[c])

tz = tz.merge(len_by_zone, on="zone_id", how="left")
tz = tz.merge(div_df,      on="zone_id", how="left")

tz["F5_sport_out_length"] = tz["F5_sport_out_length"].fillna(0.0).astype("float64")
tz["F5_sport_out_div"]    = tz["F5_sport_out_div"].fillna(0).astype("int32")

print("\n✅ Example TZ columns:")
print(tz[["zone_id", "F5_sport_out_length", "F5_sport_out_div"]].head())

# ==========================================================
# 9) Save TZ
# ==========================================================
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with F5_sport_out_length and F5_sport_out_div to: {TZ_PATH}")

# ==========================================================
# 10) Plots
# ==========================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F5_sport_out_length", ax=ax, legend=True, edgecolor="black", linewidth=0.05)
ax.set_title("F5_sport_out_length (km) — hiking + ski + snowshoe + via_ferrata", fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F5_sport_out_div", ax=ax, legend=True, edgecolor="black", linewidth=0.05)
ax.set_title("F5_sport_out_div — #route types (via_ferrata counts without 100m threshold)", fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()


## F4 – Cultural opportunities

For the **cultural** dimension of the attractivity index \(F4\), I use points of interest derived from OpenStreetMap (OSM), filtered for Switzerland, Liechtenstein, Campione d’Italia and Büsingen am Hochrhein.

#### Source dataset

- **Source:** OSM export (Overpass) including all potentially cultural POIs.
- **Original format:** `Cultural.geojson`.
- **Geometric pre–processing:**
  - All geometries are converted to **points** (centroid of lines/polygons when needed).
  - Reprojection to **EPSG:2056 – CH1903+ / LV95**.
  - Saved as `OSM/Cultural/Cultural.gpkg`.

#### Selection of cultural POIs

On `Cultural.gpkg` I apply the following steps:

1. Normalize tag values:
   - Columns used: `tourism`, `amenity`, `leisure`;
   - Convert to lowercase;
   - Empty or pseudo-missing values (`""`, `"nan"`, `"none"`, `"null"`) are mapped to `"nan"`.

2. Define **strong tag sets** that identify a cultural POI:

   - `tourism` (kept):
     - `aquarium`, `gallery`, `museum`, `zoo`.

   - `amenity` (kept):
     - `arts_centre`, `cinema`, `conference_centre`, `events_venue`,
       `exhibition_centre`, `museum`, `music_venue`,
       `planetarium`, `theatre`, `library`.

3. Inclusion rule:

   - A record is **kept** if:
     - `tourism` is in the strong list **or**
     - `amenity` is in the strong list.

   - The `leisure` tag is *not* used as an inclusion criterion (only for cleaning).

4. Exclusion rule on `leisure`:

   - If `leisure = garden` and the POI is **not** already selected by strong `tourism`/`amenity`, the record is **excluded**.
   - This means:
     - generic gardens are not counted as cultural POIs;
     - gardens that are also museums/theatres/etc. are still kept.

5. Generic tags `tourism = attraction` and `tourism = yes` are **not** used as stand–alone criteria: a POI with these values is kept only if it also has an `amenity` in the strong list; otherwise it is discarded.

The filtered result is saved as  
`OSM/Cultural/Cultural_filtered.gpkg`.

#### Duplicate removal

To avoid counting the same facility multiple times:

1. Normalize the name:
   - build `name_clean` as a cleaned version of `name` (string, trimmed, empty → NA).

2. Build an approximate spatial key:
   - `x_round` and `y_round` = point coordinates rounded at 0.1 m.

3. Compute a simple **information score**:
   $$
   \text{info\_score}_i = \#\{\text{non-nan tags in } \{\text{tourism}, \text{amenity}, \text{leisure}\}\}_i
   $$
   Higher score = record with more tag information.

4. Deduplication procedure:
   - consider only records with non-null `name_clean`;
   - sort by \((\text{name\_clean}, x_{\text{round}}, y_{\text{round}}, \text{info\_score})\) in descending `info_score`;
   - for each combination \((\text{name\_clean}, x_{\text{round}}, y_{\text{round}})\) keep **only the first** record and drop the others;
   - records without a name are not deduplicated by name and are all kept.

5. As a final check I inspect duplicates on the OSM `@id` field and the tag distributions to ensure the result is consistent.

#### Aggregation at Traffic Zone level

Filtered cultural POIs are then aggregated to **Traffic Zones 2017** (NPVM):

- each point is spatially joined to the zone \(j\) containing it;
- for each zone \(j\) I compute a **count**:
  $$
  F4_{\text{cult\_count}}(j)
  = \sum_{i \in \text{cultural POIs}} \mathbf{1}\{\text{POI } i \in \text{zone } j\}
  $$
- the result is stored in the zone layer as variable `F4_cult_count`.

In parallel, for a structural comparison, I aggregate STATENT (Jobs\_2023) data for the following sectors:

- **B0890AS** – Creative, artistic and entertainment activities (NACE 90),
- **B0891AS** – Libraries, archives, museums, botanical and zoological gardens (NACE 91),

summed at Traffic Zone level (excluding Liechtenstein and enclaves) to obtain:
$$
F4_{\text{cult\_off\_count}}(j)
= \text{B0890AS}(j) + \text{B0891AS}(j)
$$

Finally, I compute the Pearson correlation between `F4_cult_count` (OSM) and `F4_cult_off_count` (STATENT) in order to evaluate the consistency between the “visible” cultural offer captured by OSM and the formal occupational structure in cultural sectors.



#### Cultural diversity indicator

Beyond simple counts, I also compute a **cultural diversity** indicator at Traffic Zone level, based on the variety of cultural types present in each zone.

1. For each POI \(i\) in `Cultural_filtered.gpkg`, I assign a **single cultural type** \(c_i\):

   - If `tourism` is in the strong tourism list (`aquarium`, `gallery`, `museum`, `zoo`), I use that value as \(c_i\);
   - Otherwise, if `amenity` is in the strong amenity list (`arts_centre`, `cinema`, `conference_centre`, `events_venue`, `exhibition_centre`, `museum`, `music_venue`, `planetarium`, `theatre`, `library`), I use that as \(c_i\).

   Facilities that have both a strong `tourism` and `amenity` tag (e.g. `tourism = museum`, `amenity = arts_centre`) are still treated as **one cultural POI**; the type \(c_i\) is chosen following the hierarchy above, so that they are not double counted.

2. After spatially joining the POIs to Traffic Zones, I define the **cultural diversity** of zone \(j\) as the number of distinct cultural types present in that zone:

$$
F4_{\text{cult\_diversity}}(j)
= \left| \left\{ c_i \;:\; i \in \text{cultural POIs with type } c_i \text{ in zone } j \right\} \right|
$$

where \(|\cdot|\) denotes set cardinality.

The resulting variable is stored in the zone layer as  
`F4_cult_diversity`.

---

#### Cultural floor area indicator

To better capture the **intensity** of the cultural offer, I construct a proxy for the **total cultural floor area** per zone, combining OSM cultural POIs with swissTLM3D building footprints.

1. **Building matching**

   - I load building footprints from swissTLM3D (`tlm_bauten_gebaeude_footprint`).
   - For each cultural POI \(i\), I:
     - first perform a spatial join where the POI lies **within** a building polygon;
     - for POIs not matched in this step, I perform a **nearest-neighbour** join to the closest building polygon.

   For matched buildings, I compute the footprint area in square metres:

$$
A^{\text{raw}}_i = \text{area of the matched building footprint (m}^2)
$$

2. **Range filtering**

Empirical inspection of the distribution of \(A^{\text{raw}}_i\) (and comparison with typical sizes of cultural facilities) shows that a small fraction of matches correspond either to very small artefacts or to excessively large complexes. I therefore restrict attention to a **plausible range**:

$$
A_i =
\begin{cases}
A^{\text{raw}}_i & \text{if } A_{\min} \le A^{\text{raw}}_i \le A_{\max} \\
\text{NA} & \text{otherwise}
\end{cases}
$$

where \(A_{\min}\) and \(A_{\max}\) are lower and upper thresholds chosen from the empirical distribution (e.g. based on quantiles and visual inspection). Values outside this range are treated as missing and will be imputed.

For Liechtenstein and the foreign enclaves (zones with `N_KT ∈ {LIE, Enk.}`), I do not trust the swissTLM3D footprints in the same way as for Switzerland; in these cases, \(A_i\) is also set to missing and handled via imputation.

3. **Lognormal imputation of missing areas**

To obtain a complete proxy for floor area, I impute missing \(A_i\) using a **lognormal model** calibrated on the **valid Swiss matches**:

- For each POI with a valid area in Switzerland, I compute:

$$
\ell_i = \log(A_i)
$$

- I then estimate:

  - a **global** lognormal distribution

  $$
  \mu_{\text{all}} = \mathbb{E}[\ell_i], \quad
  \sigma_{\text{all}} = \sqrt{\mathrm{Var}[\ell_i]}
  $$

  - and, where the data allow it, **per-amenity** lognormal parameters

  $$
  \mu_a = \mathbb{E}[\ell_i \mid \text{amenity} = a], \quad
  \sigma_a = \sqrt{\mathrm{Var}[\ell_i \mid \text{amenity} = a]}
  $$

  for each amenity type \(a\) with a sufficient number of observations.

For every cultural POI with missing area, I then draw an imputed value \(A^{\ast}_i\) from a lognormal distribution:

- if amenity \(a\) has enough valid observations and a well-defined \((\mu_a, \sigma_a)\), I use:

  $$
  A^{\ast}_i \sim \text{Lognormal}(\mu_a, \sigma_a)
  $$

- otherwise, I fall back to the global parameters:

  $$
  A^{\ast}_i \sim \text{Lognormal}(\mu_{\text{all}}, \sigma_{\text{all}})
  $$

All samples are truncated to the same plausible range \([A_{\min}, A_{\max}]\). If repeated sampling does not produce a value within the range (rare), I use \(\exp(\mu_a)\) or \(\exp(\mu_{\text{all}})\) clamped to \([A_{\min}, A_{\max}]\).

This yields a final per-POI area variable \(A^{\text{final}}_i\), which is equal to the observed \(A_i\) when available and to the imputed \(A^{\ast}_i\) otherwise.

4. **Aggregation at Traffic Zone level**

After assigning each cultural POI to its Traffic Zone \(j\), I define the **total cultural floor area** of zone \(j\) as:

$$
F4_{\text{cult\_area\_total}}(j)
= \sum_{i \in \text{cultural POIs in zone } j} A^{\text{final}}_i
$$

The result is stored in the zone layer as  
`F4_cult_area_total`.

---

#### Cultural employment (FTE) from STATENT

To complement the OSM-based measures with an official employment perspective, I also build a **cultural Full-Time Equivalent (FTE)** indicator based on STATENT (Jobs\_2023).

From the grid of workplaces, I extract the following variables:

- **B0890VZA** – Creative, artistic and entertainment activities (NACE 90), number of full-time equivalents;
- **B0891VZA** – Libraries, archives, museums, botanical and zoological gardens (NACE 91), number of full-time equivalents.

These are spatially aggregated from the STATENT grid to Traffic Zones (using a spatial join and summing values within each zone), excluding Liechtenstein and the foreign enclaves (zones with `N_KT ∈ {LIE, Enk.}`).

For each zone \(j\), I define:

$$
F4_{\text{cult\_FTE}}(j)
= \text{B0890VZA}(j) + \text{B0891VZA}(j)
$$

where \(\text{B0890VZA}(j)\) and \(\text{B0891VZA}(j)\) denote the STATENT full-time equivalent counts of sectors 90 and 91 assigned to zone \(j\).

The resulting variable is stored in the zone layer as  
`F4_cult_FTE`.

---

#### Consistency checks: correlation between OSM area and STATENT FTE

Finally, to evaluate how well the OSM-based indicator of cultural floor area aligns with official employment data in cultural sectors, I compute correlations between `F4_cult_area_total` and `F4_cult_FTE`.

- I restrict the analysis to:
  - Swiss zones only (`N_KT` not in `{LIE, Enk.}`),
  - zones with non-missing values for both indicators.

- On this set of zones, I compute:
  - the **Pearson correlation coefficient**:

    $$
    \rho_{\text{Pearson}}
    = \mathrm{corr}\big(F4_{\text{cult\_area\_total}}(j), F4_{\text{cult\_FTE}}(j)\big)
    $$

  - and the **Spearman rank correlation**:

    $$
    \rho_{\text{Spearman}}
    = \mathrm{corr}\big(\mathrm{rank}(F4_{\text{cult\_area\_total}}(j)),
                       \mathrm{rank}(F4_{\text{cult\_FTE}}(j))\big)
    $$

These correlations provide a quantitative check of the consistency between the **“visible” cultural offer** inferred from OSM (in terms of floor area) and the **underlying employment structure** in cultural sectors as captured by STATENT. The exact values are reported and interpreted in the results chapter.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import geopandas as gpd
import pandas as pd
from pathlib import Path

# ==========================
# PATHS
# ==========================
IN_PATH  = Path("OSM/Cultural/Cultural.gpkg")
OUT_PATH = Path("OSM/Cultural/Cultural_filtered.gpkg")

TARGET_EPSG = 2056

print(f"📂 Reading: {IN_PATH}")
gdf = gpd.read_file(IN_PATH)
print(f"🔎 Total records (raw): {len(gdf)}")
print(f"🧭 CRS (raw): {gdf.crs}")

# Ensure needed columns exist
for col in ["tourism", "amenity", "leisure"]:
    if col not in gdf.columns:
        gdf[col] = pd.NA

# ==========================
# 1) Normalize tag values
# ==========================
def norm_val(v) -> str:
    if pd.isna(v):
        return "nan"
    s = str(v).strip().lower()
    if s in ("", "nan", "none", "null"):
        return "nan"
    return s

for col in ["tourism", "amenity", "leisure"]:
    gdf[col] = gdf[col].apply(norm_val)

# ==========================
# 2) Selection rules
# ==========================
KEEP_TOURISM_STRICT = {"aquarium", "gallery", "museum", "zoo"}

KEEP_AMENITY_STRICT = {
    "arts_centre",
    "cinema",
    "conference_centre",
    "events_venue",
    "exhibition_centre",
    "museum",
    "music_venue",
    "planetarium",
    "theatre",
    "library",
}

EXCLUDE_LEISURE_IF_NOT_STRONG = {"garden"}

GENERIC_TOURISM = {"attraction", "yes"}  # never sufficient on their own

def decide_keep(row) -> bool:
    t = row["tourism"]
    a = row["amenity"]
    l = row["leisure"]

    strong = (t in KEEP_TOURISM_STRICT) or (a in KEEP_AMENITY_STRICT)

    # Strong always kept
    if strong:
        return True

    # Exclude generic gardens unless strong (handled above)
    if l in EXCLUDE_LEISURE_IF_NOT_STRONG:
        return False

    # tourism=attraction/yes is NOT a standalone criterion (needs strong amenity, but that would be strong anyway)
    if t in GENERIC_TOURISM:
        return False

    # Otherwise, not cultural by our strict definition
    return False

gdf["keep_cultural"] = gdf.apply(decide_keep, axis=1)

gdf_f = gdf[gdf["keep_cultural"]].copy()
print(f"✅ Kept after filter (before point conversion / dedup): {len(gdf_f)}")
print(f"❌ Dropped: {len(gdf) - len(gdf_f)}")

# ==========================
# 3) Geometry handling: convert ALL to points (centroids)
# ==========================
# Make sure we are in a metric CRS before computing centroids
if gdf_f.crs is None:
    # assume WGS84 if missing (OSM typical); adjust if your file differs
    gdf_f = gdf_f.set_crs(epsg=4326)
    print("ℹ️ CRS missing -> set to EPSG:4326 (assumed).")

if gdf_f.crs.to_epsg() != TARGET_EPSG:
    gdf_f = gdf_f.to_crs(TARGET_EPSG)
    print(f"↪ Reprojected filtered layer to EPSG:{TARGET_EPSG} for centroid/coords.")

# Clean geometries minimally
gdf_f = gdf_f[gdf_f.geometry.notna() & ~gdf_f.geometry.is_empty].copy()

# Convert: keep points; centroid for everything else
is_point = gdf_f.geometry.geom_type == "Point"
gdf_f.loc[~is_point, "geometry"] = gdf_f.loc[~is_point, "geometry"].centroid

# After conversion, enforce all Points
if not (gdf_f.geometry.geom_type == "Point").all():
    # fallback for any weird geometry: representative point
    bad = ~(gdf_f.geometry.geom_type == "Point")
    gdf_f.loc[bad, "geometry"] = gdf_f.loc[bad, "geometry"].representative_point()

print("📍 Geometry types after point conversion:")
print(gdf_f.geometry.geom_type.value_counts(dropna=False))

# ==========================
# 4) Deduplication by name + rounded coordinates
# ==========================
# Clean name
if "name" in gdf_f.columns:
    gdf_f["name_clean"] = gdf_f["name"].astype(str).str.strip()
    gdf_f["name_clean"] = gdf_f["name_clean"].replace("", pd.NA)
else:
    gdf_f["name_clean"] = pd.NA

# Rounded coordinates (now safe because all points)
gdf_f["x_round"] = gdf_f.geometry.x.round(1)
gdf_f["y_round"] = gdf_f.geometry.y.round(1)

def is_nan_like(v: str) -> bool:
    v = str(v).strip().lower()
    return v in ("", "nan", "none", "null")

def info_score(row) -> int:
    """Number of non-nan tags among tourism/amenity/leisure."""
    score = 0
    for c in ["tourism", "amenity", "leisure"]:
        if not is_nan_like(row[c]):
            score += 1
    return score

gdf_f["info_score"] = gdf_f.apply(info_score, axis=1)

named = gdf_f[gdf_f["name_clean"].notna()].copy()
unnamed = gdf_f[gdf_f["name_clean"].isna()].copy()

before_named = len(named)

# Keep most informative record per (name_clean, x_round, y_round)
named = named.sort_values(
    ["name_clean", "x_round", "y_round", "info_score"],
    ascending=[True, True, True, False]
)
dup_mask = named.duplicated(subset=["name_clean", "x_round", "y_round"], keep="first")
removed = int(dup_mask.sum())
named = named[~dup_mask].copy()

print("\n" + "=" * 60)
print("DEDUP BY NAME + POSITION")
print(f"Named records before: {before_named}")
print(f"Named records after : {len(named)}")
print(f"Removed duplicates  : {removed}")

# Recombine
gdf_f = pd.concat([named, unnamed], ignore_index=True)
print(f"🔁 Total records after dedup: {len(gdf_f)}")

# Drop helper columns (keep keep_cultural optional)
gdf_f = gdf_f.drop(columns=["name_clean", "x_round", "y_round", "info_score"], errors="ignore")

# ==========================
# 5) Save
# ==========================
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
gdf_f.to_file(OUT_PATH, driver="GPKG")
print(f"💾 Saved filtered file: {OUT_PATH}")

# ==========================
# 6) Distributions
# ==========================
def print_distribution(df, col):
    print("\n" + "=" * 60)
    print(f"{col.upper()} — distribution (after filter+dedup)")
    vals = df[col].fillna("nan").astype(str).str.strip().str.lower()
    vals = vals.replace({"": "nan", "none": "nan", "null": "nan"})
    vc = vals.value_counts()
    print(f"Total records: {len(vals)}")
    for v, c in vc.items():
        print(f"  {v!r:<25} -> {c}")

for col in ["tourism", "amenity", "leisure"]:
    print_distribution(gdf_f, col)

# ==========================
# 7) Duplicate check on @id
# ==========================
print("\n" + "=" * 60)
print("DUPLICATES BY OSM ID (@id) AFTER FILTER+DEDUP")

if "@id" in gdf_f.columns:
    dup = gdf_f[gdf_f["@id"].duplicated(keep=False)]
    print(f"Total filtered records: {len(gdf_f)}")
    print(f"Records part of a duplicated @id: {len(dup)}")
    print(f"Distinct duplicated @id values: {dup['@id'].nunique()}")
else:
    print("⚠️ Column '@id' not present.")

# ==========================
# 8) Tag combinations (top 20)
# ==========================
print("\n" + "=" * 60)
print("TOP 20 tourism/amenity/leisure combinations (after filter+dedup)")

def clean_for_combo(v):
    return "nan" if is_nan_like(v) else str(v)

comb = (
    "tourism=" + gdf_f["tourism"].apply(clean_for_combo) + "," +
    "amenity=" + gdf_f["amenity"].apply(clean_for_combo) + "," +
    "leisure=" + gdf_f["leisure"].apply(clean_for_combo)
)

vc_comb = comb.value_counts()
for k, v in vc_comb.head(20).items():
    print(f"  {k} -> {v}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# PATHS & PARAMETERS
# ==========================================================
TZ_PATH   = "TZ/TZ.gpkg"                          # Traffic zones (must contain zone_id)
CULT_PATH = "OSM/Cultural/Cultural_filtered.gpkg" # Filtered cultural POIs (points, EPSG:2056)
JOBS_PATH = "Jobs_2023/Jobs_2023.gpkg"            # STATENT jobs grid (EPSG:2056)

ID_FIELD = "zone_id"
TARGET_EPSG = 2056

# STATENT sectors used for the "official" cultural proxy (counts)
STATENT_COLS = ["B0890AS", "B0891AS"]

# ==========================================================
# 1) LOAD TRAFFIC ZONES
# ==========================================================
print(f"📂 Reading TZ from: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
print(f"🔎 TZ loaded: {len(tz)}")

if tz.crs is None:
    raise ValueError("❌ TZ has no CRS defined.")
if tz.crs.to_epsg() != TARGET_EPSG:
    tz = tz.to_crs(TARGET_EPSG)
    print(f"↪ Reprojected TZ to EPSG:{TARGET_EPSG}")

if ID_FIELD not in tz.columns:
    raise ValueError(f"❌ '{ID_FIELD}' not found in TZ. Columns: {list(tz.columns)}")

# Minimal GeoDataFrame for spatial joins
tz_geom = tz[[ID_FIELD, "geometry"]].copy()

# ==========================================================
# 2) LOAD OSM CULTURAL POIs
# ==========================================================
print(f"\n📂 Reading filtered cultural POIs from: {CULT_PATH}")
cult = gpd.read_file(CULT_PATH)
print(f"🔎 Cultural POIs (filtered): {len(cult)}")

if cult.crs is None:
    raise ValueError("❌ Cultural POIs have no CRS defined.")
if cult.crs.to_epsg() != TARGET_EPSG:
    cult = cult.to_crs(TARGET_EPSG)
    print(f"↪ Reprojected CULT to EPSG:{TARGET_EPSG}")

# Deduplicate by OSM @id (same OSM object counted once)
if "@id" in cult.columns:
    before = len(cult)
    cult = cult.drop_duplicates(subset="@id")
    print(f"✅ Deduplicated by @id: removed {before - len(cult)} duplicates (remaining {len(cult)})")
else:
    print("⚠️ No '@id' column found -> cannot deduplicate by OSM id.")

# ==========================================================
# 3) SPATIAL JOIN POIs -> TZ AND COMPUTE F4_cult_count
# ==========================================================
print("\n🔗 Spatial join cultural POIs -> TZ (within)...")

try:
    cult_zones = gpd.sjoin(cult, tz_geom, how="left", predicate="within")
except TypeError:
    # compatibility with older geopandas
    cult_zones = gpd.sjoin(cult, tz_geom, how="left", op="within")

assigned = int(cult_zones[ID_FIELD].notna().sum())
print(f"🔎 POIs assigned to a TZ: {assigned} / {len(cult_zones)}")

cult_counts = (
    cult_zones
    .dropna(subset=[ID_FIELD])
    .groupby(ID_FIELD)
    .size()
    .reset_index(name="F4_cult_count")
)

print(f"✅ TZ with at least 1 cultural POI: {len(cult_counts)}")

# ==========================================================
# 4) LOAD STATENT JOBS AND COMPUTE F4_cult_off_count
# ==========================================================
print(f"\n📂 Reading STATENT Jobs from: {JOBS_PATH}")
jobs = gpd.read_file(JOBS_PATH)
print(f"🔎 Jobs records: {len(jobs)}")

if jobs.crs is None:
    raise ValueError("❌ Jobs layer has no CRS defined.")
if jobs.crs.to_epsg() != TARGET_EPSG:
    jobs = jobs.to_crs(TARGET_EPSG)
    print(f"↪ Reprojected JOBS to EPSG:{TARGET_EPSG}")

# Exclude Liechtenstein + enclaves if the column exists
if "N_KT" in jobs.columns:
    before_jobs = len(jobs)
    jobs = jobs[~jobs["N_KT"].isin(["LIE", "Enk"])].copy()
    print(f"✅ Excluded N_KT in ['LIE','Enk']: removed {before_jobs - len(jobs)} records")
else:
    print("⚠️ 'N_KT' not found in Jobs -> cannot exclude LIE/Enk here.")

# Ensure the STATENT columns exist and are numeric
for col in STATENT_COLS:
    if col not in jobs.columns:
        raise ValueError(f"❌ Column '{col}' not found in Jobs. Columns: {list(jobs.columns)}")
    jobs[col] = pd.to_numeric(jobs[col], errors="coerce").fillna(0)

jobs["cult_off"] = jobs[STATENT_COLS].sum(axis=1)

print("\n🔗 Spatial join STATENT jobs -> TZ (within)...")
try:
    jobs_zones = gpd.sjoin(jobs, tz_geom, how="inner", predicate="within")
except TypeError:
    jobs_zones = gpd.sjoin(jobs, tz_geom, how="inner", op="within")

print(f"🔎 Jobs points assigned to TZ: {len(jobs_zones)}")

jobs_zone_sum = (
    jobs_zones
    .groupby(ID_FIELD)["cult_off"]
    .sum()
    .reset_index(name="F4_cult_off_count")
)

print(f"✅ TZ with at least 1 cultural job (sum>0 possible): {len(jobs_zone_sum)}")

# ==========================================================
# 5) MERGE RESULTS BACK INTO TZ (without dropping other attributes)
# ==========================================================
# Remove old columns if they exist (only these two)
for c in ["F4_cult_count", "F4_cult_off_count"]:
    if c in tz.columns:
        tz = tz.drop(columns=[c])

tz = tz.merge(cult_counts, on=ID_FIELD, how="left")
tz["F4_cult_count"] = tz["F4_cult_count"].fillna(0).astype("int32")

tz = tz.merge(jobs_zone_sum, on=ID_FIELD, how="left")
tz["F4_cult_off_count"] = tz["F4_cult_off_count"].fillna(0).astype("float64")

# ==========================================================
# 6) CORRELATION (OSM vs STATENT)
# ==========================================================
print("\n📊 Correlation between F4_cult_count (OSM) and F4_cult_off_count (STATENT)")

valid = tz[["F4_cult_count", "F4_cult_off_count"]].replace([np.inf, -np.inf], np.nan).dropna()
n_valid = len(valid)

if n_valid > 1:
    pearson = valid["F4_cult_count"].corr(valid["F4_cult_off_count"], method="pearson")
    spearman = valid["F4_cult_count"].corr(valid["F4_cult_off_count"], method="spearman")
    print(f"   Zones used: {n_valid}")
    print(f"   Pearson : {pearson:.3f}")
    print(f"   Spearman: {spearman:.3f}")
else:
    print(f"⚠️ Not enough valid observations for correlation (n={n_valid}).")

# ==========================================================
# 7) SAVE UPDATED TZ
# ==========================================================
print(f"\n💾 Writing updated TZ (adds F4_cult_count and F4_cult_off_count): {TZ_PATH}")
tz.to_file(TZ_PATH, driver="GPKG")

# ==========================================================
# 8) MAP (simple continuous choropleth, no mapclassify)
# ==========================================================
print("\n🗺️ Plotting map of F4_cult_count...")

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(
    column="F4_cult_count",
    ax=ax,
    legend=True,
    edgecolor="none",
    linewidth=0,
)
ax.set_title("F4_cult_count — OSM cultural POIs per traffic zone")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TZ_PATH   = "TZ/TZ.gpkg"
CULT_PATH = "OSM/Cultural/Cultural_filtered.gpkg"

TARGET_EPSG = 2056
ID_FIELD = "zone_id"

# -------------------------------------------------------------
# 1) Read data
# -------------------------------------------------------------
tz = gpd.read_file(TZ_PATH)
print(f"Traffic zones: {len(tz)}")

gdf_cult = gpd.read_file(CULT_PATH)
print(f"Cultural POIs (filtered): {len(gdf_cult)}")

# CRS -> EPSG:2056
if tz.crs is None:
    raise ValueError("TZ has no CRS.")
if tz.crs.to_epsg() != TARGET_EPSG:
    tz = tz.to_crs(TARGET_EPSG)

if gdf_cult.crs is None:
    raise ValueError("Cultural POIs have no CRS.")
if gdf_cult.crs.to_epsg() != TARGET_EPSG:
    gdf_cult = gdf_cult.to_crs(TARGET_EPSG)

if ID_FIELD not in tz.columns:
    raise ValueError(f"'{ID_FIELD}' not found in TZ columns.")

# Safety: deduplicate by OSM id if present
if "@id" in gdf_cult.columns:
    before = len(gdf_cult)
    gdf_cult = gdf_cult.drop_duplicates(subset="@id")
    print(f"✅ Deduplicated by @id: removed {before - len(gdf_cult)} duplicates.")

# -------------------------------------------------------------
# 2) Normalize tags + assign ONE cultural type per POI
# -------------------------------------------------------------
def norm_val(v):
    if pd.isna(v):
        return "nan"
    s = str(v).strip().lower()
    if s in ("", "nan", "none", "null"):
        return "nan"
    return s

for col in ["tourism", "amenity"]:
    if col not in gdf_cult.columns:
        gdf_cult[col] = "nan"
    gdf_cult[col] = gdf_cult[col].apply(norm_val)

KEEP_TOURISM_STRICT = {"aquarium", "gallery", "museum", "zoo"}
KEEP_AMENITY = {
    "arts_centre", "cinema", "conference_centre", "events_venue",
    "exhibition_centre", "museum", "music_venue",
    "planetarium", "theatre", "library",
}

def choose_cult_type(row):
    t = row.get("tourism", "nan")
    a = row.get("amenity", "nan")
    if t in KEEP_TOURISM_STRICT:
        return t
    if a in KEEP_AMENITY:
        return a
    return np.nan

gdf_cult["cult_type"] = gdf_cult.apply(choose_cult_type, axis=1)

print("\nSample cult_type counts:")
print(gdf_cult["cult_type"].value_counts(dropna=False).head(15))

# Keep only POIs with a valid type
gdf_cult = gdf_cult.dropna(subset=["cult_type"]).copy()

# -------------------------------------------------------------
# 3) Spatial join POIs -> TZ
# -------------------------------------------------------------
left = gpd.GeoDataFrame(
    gdf_cult[["cult_type", "geometry"]].copy(),
    geometry="geometry",
    crs=gdf_cult.crs,
)
right = tz[[ID_FIELD, "geometry"]].copy()

join = gpd.sjoin(left, right, how="left", predicate="within")

print(f"\nJoin result rows: {len(join)}")
print("POIs without zone_id:", int(join[ID_FIELD].isna().sum()))

# -------------------------------------------------------------
# 4) Diversity per zone (number of distinct types)
# -------------------------------------------------------------
join_valid = join.dropna(subset=[ID_FIELD, "cult_type"]).copy()
join_valid[ID_FIELD] = join_valid[ID_FIELD].astype(int)

div_per_zone = (
    join_valid
    .groupby(ID_FIELD)["cult_type"]
    .nunique()
    .reset_index(name="F4_cult_diversity")
)

print("\nExample diversity per zone:")
print(div_per_zone.head())

# Merge into TZ (do not drop other attributes)
if "F4_cult_diversity" in tz.columns:
    tz = tz.drop(columns=["F4_cult_diversity"])

tz = tz.merge(div_per_zone, on=ID_FIELD, how="left")
tz["F4_cult_diversity"] = tz["F4_cult_diversity"].fillna(0).astype("int32")

# -------------------------------------------------------------
# 5) Save TZ
# -------------------------------------------------------------
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n✅ Updated TZ.gpkg with F4_cult_diversity saved to: {TZ_PATH}")

# -------------------------------------------------------------
# 6) Plot (NO mapclassify / NO scheme_kwargs)
# -------------------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(
    column="F4_cult_diversity",
    ax=ax,
    legend=True,
    edgecolor="none",
    linewidth=0,
)
ax.set_title("F4_cult_diversity — number of cultural POI types per Traffic Zone", fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# ==========================================================
# PATHS
# ==========================================================
TZ_PATH        = "TZ/TZ.gpkg"
CULT_PATH      = "OSM/Cultural/Cultural_filtered.gpkg"
TLM_PATH       = "swissTLM3D_2025/SWISSTLM3D_2025.gpkg"
TLM_BLD_LAYER  = "tlm_bauten_gebaeude_footprint"
JOBS_PATH      = "Jobs_2023/Jobs_2023.gpkg"  # STATENT

TARGET_EPSG = 2056
ID_FIELD = "zone_id"
KT_FIELD = "N_KT"

# Special zones to exclude from "Swiss-only" evaluation (and building-trust logic)
SPECIAL_KT = {"LIE", "Enk", "Enk."}

# Reproducibility for imputation
RNG = np.random.default_rng(123)

# Building area plausibility range for cultural facilities (m²)
MIN_AREA = 30.0
MAX_AREA = 5000.0

# Per-type lognormal used only if enough observations
MIN_N_PER_TYPE = 30

# STATENT FTE columns (culture)
FTE_COLS = ["B0890VZA", "B0891VZA"]

# ==========================================================
# UTILS
# ==========================================================
def ensure_epsg(gdf, epsg, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label}: CRS is missing.")
    if gdf.crs.to_epsg() != epsg:
        gdf = gdf.to_crs(epsg)
        print(f"↪ Reprojected {label} to EPSG:{epsg}")
    return gdf

def clean_geom(gdf, label):
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def norm_val(v) -> str:
    if pd.isna(v):
        return "nan"
    s = str(v).strip().lower()
    if s in ("", "nan", "none", "null"):
        return "nan"
    return s

# Strict cultural type hierarchy (match your doc)
KEEP_TOURISM_STRICT = {"aquarium", "gallery", "museum", "zoo"}
KEEP_AMENITY_STRICT = {
    "arts_centre", "cinema", "conference_centre", "events_venue",
    "exhibition_centre", "museum", "music_venue",
    "planetarium", "theatre", "library",
}

def choose_cult_type(row):
    t = row.get("tourism", "nan")
    a = row.get("amenity", "nan")
    if t in KEEP_TOURISM_STRICT:
        return t
    if a in KEEP_AMENITY_STRICT:
        return a
    return np.nan

def corr_report(df, x, y, label):
    valid = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    print(f"\n📈 Correlation — {label}")
    print(f"   Zones used: {len(valid)}")
    if len(valid) < 2:
        print("   ⚠ Not enough observations.")
        return
    if valid[x].std() == 0 or valid[y].std() == 0:
        print("   ⚠ std==0 -> correlation not meaningful.")
        return
    pear = valid[x].corr(valid[y], method="pearson")
    spear = valid[x].corr(valid[y], method="spearman")
    print(f"   Std {x}: {valid[x].std():.6f} | Std {y}: {valid[y].std():.6f}")
    print(f"   Pearson : {pear:.3f}")
    print(f"   Spearman: {spear:.3f}")

# ==========================================================
# 1) LOAD TZ
# ==========================================================
print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
print(f"   Traffic zones: {len(tz)}")

tz = ensure_epsg(tz, TARGET_EPSG, "TZ")
tz = clean_geom(tz, "TZ")

if ID_FIELD not in tz.columns:
    raise ValueError(f"❌ TZ must contain '{ID_FIELD}'. Columns: {list(tz.columns)}")
if KT_FIELD not in tz.columns:
    raise ValueError(f"❌ TZ must contain '{KT_FIELD}'. Columns: {list(tz.columns)}")

# Use only ID + geometry for joins
tz_geom = tz[[ID_FIELD, KT_FIELD, "geometry"]].copy()

# ==========================================================
# 2) LOAD CULTURAL POIs (FILTERED)
# ==========================================================
print(f"\n📂 Reading cultural POIs: {CULT_PATH}")
cult = gpd.read_file(CULT_PATH)
print(f"   Cultural POIs (filtered): {len(cult)}")

cult = ensure_epsg(cult, TARGET_EPSG, "Cultural POIs")
cult = clean_geom(cult, "Cultural POIs")

# Safety: dedup by OSM id if available
if "@id" in cult.columns:
    before = len(cult)
    cult = cult.drop_duplicates(subset="@id")
    print(f"✅ Deduplicated by @id: removed {before - len(cult)} duplicates.")
else:
    print("⚠️ No '@id' column -> dedup skipped.")

# Stable POI id
cult = cult.reset_index(drop=True)
cult["poi_id"] = cult.index.astype("int64")

# Normalize tags + assign a single cult_type
for col in ["tourism", "amenity"]:
    if col not in cult.columns:
        cult[col] = "nan"
    cult[col] = cult[col].apply(norm_val)

cult["cult_type"] = cult.apply(choose_cult_type, axis=1)

print("\n📊 cult_type distribution (top 15):")
print(cult["cult_type"].value_counts(dropna=False).head(15))

# ==========================================================
# 3) LOAD swissTLM3D BUILDING FOOTPRINTS
# ==========================================================
print(f"\n📂 Reading swissTLM3D building footprints: {TLM_PATH} | layer={TLM_BLD_LAYER}")
bld = gpd.read_file(TLM_PATH, layer=TLM_BLD_LAYER)
print(f"   Buildings (raw): {len(bld)}")

bld = ensure_epsg(bld, TARGET_EPSG, "Buildings")
bld = clean_geom(bld, "Buildings")

# Building footprint area (m²)
bld["bldg_area_m2"] = bld.geometry.area.astype("float64")

# Keep only columns we need (+ geometry)
keep_bld_cols = [c for c in ["uuid", "objektart", "nutzung", "bldg_area_m2", "geometry"] if c in bld.columns]
bld = bld[keep_bld_cols].copy()

print("\n📊 Building area stats (m²):")
print(bld["bldg_area_m2"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

# ==========================================================
# 4) MATCH EACH POI TO A BUILDING (within + nearest)
# ==========================================================
cult_bld = cult.copy()
cult_bld["bldg_area_m2_raw"] = np.nan

# WITHIN join
left = cult_bld[["poi_id", "geometry"]].copy()
right = bld[["bldg_area_m2", "geometry"]].copy()

print("\n🔗 Matching POIs to buildings (within)...")
join_within = gpd.sjoin(left, right, how="left", predicate="within")
within_first = (
    join_within.dropna(subset=["bldg_area_m2"])
    .groupby("poi_id")["bldg_area_m2"]
    .first()
)

cult_bld.loc[cult_bld["poi_id"].isin(within_first.index), "bldg_area_m2_raw"] = (
    cult_bld.loc[cult_bld["poi_id"].isin(within_first.index), "poi_id"].map(within_first).to_numpy()
)

n_within = int(cult_bld["bldg_area_m2_raw"].notna().sum())
print(f"   POIs matched by within: {n_within} / {len(cult_bld)}")

# NEAREST join for the remaining
missing = cult_bld["bldg_area_m2_raw"].isna()
cult_missing = cult_bld.loc[missing, ["poi_id", "geometry"]].copy()

print(f"   POIs still missing after within: {len(cult_missing)}")
if len(cult_missing) > 0:
    print("🔗 Matching POIs to buildings (nearest)...")
    nearest = gpd.sjoin_nearest(
        cult_missing,
        right,
        how="left",
        distance_col="dist_to_bldg_m"
    )
    nearest_first = (
        nearest.dropna(subset=["bldg_area_m2"])
        .groupby("poi_id")["bldg_area_m2"]
        .first()
    )
    cult_bld.loc[cult_bld["poi_id"].isin(nearest_first.index), "bldg_area_m2_raw"] = (
        cult_bld.loc[cult_bld["poi_id"].isin(nearest_first.index), "poi_id"].map(nearest_first).to_numpy()
    )

n_any = int(cult_bld["bldg_area_m2_raw"].notna().sum())
print(f"   POIs matched (within+nearest): {n_any} / {len(cult_bld)}")

# ==========================================================
# 5) ASSIGN POIs TO ZONES (within + nearest fallback)
# ==========================================================
print("\n🔗 Assigning POIs to traffic zones (within, fallback nearest)...")

left_tz = cult_bld[["poi_id", "geometry"]].copy()
right_tz = tz_geom[[ID_FIELD, KT_FIELD, "geometry"]].copy()

join_tz = gpd.sjoin(left_tz, right_tz, how="left", predicate="within")
tz_first = (
    join_tz.dropna(subset=[ID_FIELD])
    .groupby("poi_id")[[ID_FIELD, KT_FIELD]]
    .first()
)

cult_bld = cult_bld.merge(tz_first, on="poi_id", how="left")

miss_zone = cult_bld[ID_FIELD].isna()
if miss_zone.any():
    miss_pts = cult_bld.loc[miss_zone, ["poi_id", "geometry"]].copy()
    near_tz = gpd.sjoin_nearest(miss_pts, right_tz, how="left", distance_col="dist_to_zone")
    near_first = (
        near_tz.dropna(subset=[ID_FIELD])
        .groupby("poi_id")[[ID_FIELD, KT_FIELD]]
        .first()
    )
    cult_bld.loc[cult_bld["poi_id"].isin(near_first.index), [ID_FIELD, KT_FIELD]] = (
        cult_bld.loc[cult_bld["poi_id"].isin(near_first.index), "poi_id"]
        .map(near_first[ID_FIELD]).to_numpy(),
        cult_bld.loc[cult_bld["poi_id"].isin(near_first.index), "poi_id"]
        .map(near_first[KT_FIELD]).to_numpy()
    )

print(f"   POIs with assigned zone_id: {int(cult_bld[ID_FIELD].notna().sum())} / {len(cult_bld)}")

# ==========================================================
# 6) RANGE FILTER + IMPUTATION (THIS IS WHERE IMPUTATION HAPPENS)
# ==========================================================
# Do not trust swissTLM3D buildings for LIE/enclaves
cult_bld.loc[cult_bld[KT_FIELD].isin(SPECIAL_KT), "bldg_area_m2_raw"] = np.nan

# Apply plausible range
in_range = (
    cult_bld["bldg_area_m2_raw"].notna() &
    (cult_bld["bldg_area_m2_raw"] >= MIN_AREA) &
    (cult_bld["bldg_area_m2_raw"] <= MAX_AREA)
)
cult_bld["bldg_area_m2"] = np.where(in_range, cult_bld["bldg_area_m2_raw"], np.nan)

valid_ch = cult_bld[
    (~cult_bld[KT_FIELD].isin(SPECIAL_KT)) &
    cult_bld["bldg_area_m2"].notna()
].copy()

print(f"\n✅ Valid CH POIs with building area in range [{MIN_AREA},{MAX_AREA}] m²: {len(valid_ch)}")

# Fit lognormal on CH-valid matches
if len(valid_ch) > 0:
    valid_ch["log_area"] = np.log(valid_ch["bldg_area_m2"])
    mu_all = float(valid_ch["log_area"].mean())
    sigma_all = float(valid_ch["log_area"].std(ddof=1))

    type_stats = (
        valid_ch
        .groupby("cult_type")["log_area"]
        .agg(["mean", "std", "count"])
        .rename(columns={"mean": "mu", "std": "sigma"})
    )

    mu_dict = type_stats["mu"].to_dict()
    sigma_dict = type_stats["sigma"].to_dict()
    count_dict = type_stats["count"].to_dict()

    print("\n📌 Per-type log-area stats (head):")
    print(type_stats.sort_values("count", ascending=False).head(10))
else:
    # fallback if no valid CH areas exist (should not happen normally)
    mu_all = float(np.log((MIN_AREA + MAX_AREA) / 2.0))
    sigma_all = 0.5
    mu_dict, sigma_dict, count_dict = {}, {}, {}

def sample_area(row, max_tries=30):
    """
    Sample an imputed area (m²) from a lognormal distribution,
    per-type if count>=MIN_N_PER_TYPE else global.
    Always truncated to [MIN_AREA, MAX_AREA].
    """
    t = row.get("cult_type", None)
    mu = mu_all
    sigma = sigma_all

    if t in count_dict and count_dict[t] >= MIN_N_PER_TYPE:
        if (not pd.isna(mu_dict.get(t))) and (not pd.isna(sigma_dict.get(t))) and (sigma_dict.get(t, 0) > 0):
            mu = float(mu_dict[t])
            sigma = float(sigma_dict[t])

    for _ in range(max_tries):
        val = float(RNG.lognormal(mean=mu, sigma=sigma))
        if MIN_AREA <= val <= MAX_AREA:
            return val

    # fallback if sampling keeps missing the range
    return float(np.clip(np.exp(mu), MIN_AREA, MAX_AREA))

# Final per-POI area: observed if available, else imputed
cult_bld["F4_cult_area_m2"] = cult_bld["bldg_area_m2"]

need_imp = cult_bld["F4_cult_area_m2"].isna()
print(f"\n🧪 POIs needing imputation (incl. LIE/enclaves and range-outliers): {int(need_imp.sum())}")

cult_bld.loc[need_imp, "F4_cult_area_m2"] = cult_bld.loc[need_imp].apply(sample_area, axis=1)

print(f"   Remaining NaN in F4_cult_area_m2: {int(cult_bld['F4_cult_area_m2'].isna().sum())}")

# ==========================================================
# 7) AGGREGATE TO ZONES: F4_cult_area_total
# ==========================================================
area_zone = (
    cult_bld
    .dropna(subset=[ID_FIELD])
    .groupby(ID_FIELD)["F4_cult_area_m2"]
    .sum()
    .reset_index(name="F4_cult_area_total")
)

# Do NOT drop other TZ attributes; overwrite only if column exists
if "F4_cult_area_total" in tz.columns:
    tz = tz.drop(columns=["F4_cult_area_total"])

tz = tz.merge(area_zone, on=ID_FIELD, how="left")
tz["F4_cult_area_total"] = tz["F4_cult_area_total"].fillna(0).astype("float64")

print("\n✅ Example F4_cult_area_total:")
print(tz[[ID_FIELD, "F4_cult_area_total"]].head())

# ==========================================================
# 8) STATENT FTE: F4_cult_FTE (B0890VZA + B0891VZA) aggregated to TZ
# ==========================================================
print(f"\n📂 Reading STATENT Jobs: {JOBS_PATH}")
jobs = gpd.read_file(JOBS_PATH)
print(f"   Jobs records: {len(jobs)}")

jobs = ensure_epsg(jobs, TARGET_EPSG, "Jobs")
jobs = clean_geom(jobs, "Jobs")

# Ensure numeric and create cult_fte per job cell
for c in FTE_COLS:
    if c not in jobs.columns:
        raise ValueError(f"❌ Jobs missing column '{c}'. Columns: {list(jobs.columns)}")
    jobs[c] = pd.to_numeric(jobs[c], errors="coerce").fillna(0)

jobs["cult_fte"] = jobs[FTE_COLS].sum(axis=1)

# Spatial join jobs -> TZ (Swiss-only evaluation later; for aggregation we keep all TZ)
print("\n🔗 Spatial join Jobs -> TZ (within)...")
jobs_z = gpd.sjoin(jobs, tz_geom[[ID_FIELD, KT_FIELD, "geometry"]], how="inner", predicate="within")

fte_zone = (
    jobs_z
    .groupby(ID_FIELD)["cult_fte"]
    .sum()
    .reset_index(name="F4_cult_FTE")
)

if "F4_cult_FTE" in tz.columns:
    tz = tz.drop(columns=["F4_cult_FTE"])

tz = tz.merge(fte_zone, on=ID_FIELD, how="left")
tz["F4_cult_FTE"] = tz["F4_cult_FTE"].fillna(0).astype("float64")

print("\n✅ Example F4_cult_FTE:")
print(tz[[ID_FIELD, "F4_cult_FTE"]].head())

# ==========================================================
# 9) CORRELATION: F4_cult_area_total vs F4_cult_FTE (Swiss only)
# ==========================================================
mask_swiss = ~tz[KT_FIELD].isin(SPECIAL_KT)
tz_swiss = tz.loc[mask_swiss].copy()

corr_report(
    tz_swiss,
    x="F4_cult_area_total",
    y="F4_cult_FTE",
    label="F4_cult_area_total (OSM+TLM proxy) vs F4_cult_FTE (STATENT B0890VZA+B0891VZA), Swiss-only"
)

# ==========================================================
# 10) SAVE TZ
# ==========================================================
os.makedirs(os.path.dirname(TZ_PATH), exist_ok=True)
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved updated TZ (F4_cult_area_total + F4_cult_FTE) to: {TZ_PATH}")

# ==========================================================
# 11) PLOTS (no mapclassify)
# ==========================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F4_cult_area_total", ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title("F4_cult_area_total — cultural floor area proxy per traffic zone (m²)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F4_cult_FTE", ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title("F4_cult_FTE — STATENT cultural FTE per traffic zone (B0890VZA + B0891VZA)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import geopandas as gpd
import pandas as pd
import numpy as np

TZ_PATH   = "TZ/TZ.gpkg"
CULT_PATH = "OSM/Cultural/Cultural_filtered.gpkg"

TARGET_EPSG = 2056
ID_FIELD = "zone_id"

KEEP_TOURISM_STRICT = {"aquarium", "gallery", "museum", "zoo"}
KEEP_AMENITY_STRICT = {
    "arts_centre", "cinema", "conference_centre", "events_venue",
    "exhibition_centre", "museum", "music_venue",
    "planetarium", "theatre", "library",
}

def norm_val(v) -> str:
    if pd.isna(v):
        return "nan"
    s = str(v).strip().lower()
    if s in ("", "nan", "none", "null"):
        return "nan"
    return s

def choose_cult_type(row):
    t = row.get("tourism", "nan")
    a = row.get("amenity", "nan")
    if t in KEEP_TOURISM_STRICT:
        return t
    if a in KEEP_AMENITY_STRICT:
        return a
    return np.nan

def safe_colname(x: str) -> str:
    # keep letters, numbers, underscore
    x = str(x).strip().lower()
    x = x.replace(" ", "_")
    x = "".join(ch if (ch.isalnum() or ch == "_") else "_" for ch in x)
    while "__" in x:
        x = x.replace("__", "_")
    return x

print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
if tz.crs is None or tz.crs.to_epsg() != TARGET_EPSG:
    tz = tz.to_crs(TARGET_EPSG)

if ID_FIELD not in tz.columns:
    raise ValueError(f"❌ '{ID_FIELD}' not found in TZ.")

tz_geom = tz[[ID_FIELD, "geometry"]].copy()

print(f"\n📂 Reading cultural POIs: {CULT_PATH}")
cult = gpd.read_file(CULT_PATH)
if cult.crs is None or cult.crs.to_epsg() != TARGET_EPSG:
    cult = cult.to_crs(TARGET_EPSG)

# Dedup by OSM id if present
if "@id" in cult.columns:
    before = len(cult)
    cult = cult.drop_duplicates(subset="@id")
    print(f"✅ Deduplicated by @id: removed {before - len(cult)} duplicates.")

# Normalize tags
for col in ["tourism", "amenity"]:
    if col not in cult.columns:
        cult[col] = "nan"
    cult[col] = cult[col].apply(norm_val)

# Assign type
cult["cult_type"] = cult.apply(choose_cult_type, axis=1)
cult = cult.dropna(subset=["cult_type"]).copy()

print("\n📊 cult_type distribution (top 15):")
print(cult["cult_type"].value_counts().head(15))

# Spatial join POIs -> TZ
print("\n🔗 Spatial join cultural POIs -> TZ (within)...")
try:
    j = gpd.sjoin(cult[["cult_type", "geometry"]], tz_geom, how="left", predicate="within")
except TypeError:
    j = gpd.sjoin(cult[["cult_type", "geometry"]], tz_geom, how="left", op="within")

j = j.dropna(subset=[ID_FIELD]).copy()
j[ID_FIELD] = j[ID_FIELD].astype(int)

# Pivot to per-type counts
counts = (
    j.groupby([ID_FIELD, "cult_type"])
     .size()
     .unstack(fill_value=0)
)

# Rename columns -> F4_cult_<type>_count
rename_map = {c: f"F4_cult_{safe_colname(c)}_count" for c in counts.columns}
counts = counts.rename(columns=rename_map).reset_index()

# Drop existing per-type columns if they already exist (only those we overwrite)
new_cols = [c for c in counts.columns if c != ID_FIELD]
drop_existing = [c for c in new_cols if c in tz.columns]
if drop_existing:
    tz = tz.drop(columns=drop_existing)

# Merge into TZ
tz = tz.merge(counts, on=ID_FIELD, how="left")

# Fill NaNs with 0 and cast to int
for c in new_cols:
    tz[c] = tz[c].fillna(0).astype("int32")

# Optional: recompute total from subtypes (keeps consistency)
# (comment out if you prefer to keep your previous F4_cult_count untouched)
subtype_cols = new_cols
tz["F4_cult_count"] = tz[subtype_cols].sum(axis=1).astype("int32")

# Save
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with per-type cultural counts to: {TZ_PATH}")
print(f"✅ Added columns: {new_cols}")


## F3 Outdoor


#### Source datasets

- **Traffic zones:** `TZ/TZ.gpkg` (NPVM 2017)
- **Official hiking trails:** `GEOADMIN/Hiking_2023.gpkg`
- **Protected areas:**
  - National and regional parks – `GEOADMIN/Parks_2025/Parks.gpkg`
  - Pro Natura nature reserves – `GEOADMIN/Reserves_2024.gpkg`

All layers are projected to **EPSG:2056 – CH1903+ / LV95** and geometrically validated using a zero–width buffer.

---

### F3_outdoor_walk – Length of official hiking trails

1. From the hiking dataset, only segments with `wanderwege = "Wanderweg"` are selected exluding therefore Bergwanderweg and Alpinwanderweg.  
2. Each segment is spatially intersected with the traffic zones.  
3. The total length of hiking trails (in km) is then aggregated per zone as:

   \[
   F3_{\text{outdoor\_walk}}(j) = 
   \sum_{i \in \text{Wanderweg}} \text{length}_{ij} \text{ [km]}
   \]

4. The resulting values are stored in `TZ/TZ.gpkg` as `F3_outdoor_walk`.

---

### F3_outdoor_protected – Presence of protected areas

1. The polygons from **Parks.gpkg** and **Reserves_2024.gpkg** are merged into a single layer of protected areas.  
2. For each traffic zone \(j\), a binary indicator is assigned:

   \[
   F3_{\text{outdoor\_protected}}(j) =
   \begin{cases}
   1 & \text{if zone intersects a park or reserve,} \\
   0 & \text{otherwise.}
   \end{cases}
   \]

3. The result is stored in `TZ/TZ.gpkg` as `F3_outdoor_protected`.

---

### Output

Both variables are saved into the updated zone layer `TZ/TZ.gpkg` and mapped for visualization:

- **F3_outdoor_walk:** total kilometers of official hiking trails per zone.  
- **F3_outdoor_protected:** binary indicator for the presence of a protected area.
- **F3_outdoor_protected_area** total area in km^2 of protected area per traffic zone


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
base = Path(".")

tz_path       = base / "TZ" / "TZ.gpkg"
hiking_path   = base / "GEOADMIN" / "Hiking_2023.gpkg"
parks_path    = base / "GEOADMIN" / "Parks_2025" / "Parks.gpkg"
reserves_path = base / "GEOADMIN" / "Reserves_2024.gpkg"

CRS_EPSG = 2056

# =========================================================
# 1) Load Traffic Zones (NPVM 2017)
# =========================================================
tz = gpd.read_file(tz_path)

if "zone_id" not in tz.columns:
    raise ValueError("TZ.gpkg: column 'zone_id' not found.")

tz["zone_id"] = tz["zone_id"].astype(int)

# CRS harmonisation + polygon fix (buffer(0) ok for polygons)
if tz.crs is None or tz.crs.to_epsg() != CRS_EPSG:
    tz = tz.to_crs(CRS_EPSG)

tz = tz[tz.geometry.notna() & (~tz.geometry.is_empty)].copy()
tz["geometry"] = tz.geometry.buffer(0)

print(f"Traffic zones: {len(tz)}")

# =========================================================
# 2) F3_outdoor_walk — total length of official hiking trails per zone (km)
#    Keep only 'Wanderweg' (exclude Bergwanderweg, Alpinwanderweg)
# =========================================================
hike = gpd.read_file(hiking_path)
print(f"Hiking features (raw): {len(hike)}")

if hike.crs is None:
    raise ValueError("Hiking_2023.gpkg has no CRS defined.")
if hike.crs.to_epsg() != CRS_EPSG:
    hike = hike.to_crs(CRS_EPSG)

if "wanderwege" not in hike.columns:
    raise ValueError("Hiking_2023.gpkg: column 'wanderwege' not found.")

# IMPORTANT: do NOT apply buffer(0) to (possibly) 3D lines
hike = hike[hike.geometry.notna() & (~hike.geometry.is_empty)].copy()
hike = hike[hike.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

hike = hike[hike["wanderwege"] == "Wanderweg"].copy()
print(f"Wanderweg segments: {len(hike)}")

tz_min = tz[["zone_id", "geometry"]].copy()

# Intersect hiking lines with traffic zones
hike_int = gpd.overlay(
    hike[["geometry"]],
    tz_min,
    how="intersection"
)

print(f"Segments after intersection (Wanderweg ∩ TZ): {len(hike_int)}")

# Keep only linear geometries (safety)
hike_int = hike_int[hike_int.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

# Length in km (EPSG:2056 is metric)
hike_int["length_km"] = hike_int.geometry.length / 1000.0

walk_per_zone = (
    hike_int
    .groupby("zone_id")["length_km"]
    .sum()
    .reindex(tz["zone_id"], fill_value=0)
)

tz["F3_outdoor_walk"] = walk_per_zone.values

print("\nExample F3_outdoor_walk:")
print(tz[["zone_id", "F3_outdoor_walk"]].head())

# =========================================================
# 3) Protected areas — official parks + Pro Natura reserves
#    - F3_outdoor_protected: 0/1 if zone intersects any protected polygon
#    - F3_outdoor_protected_area: area of (protected ∩ zone) in km²
# =========================================================
parks = gpd.read_file(parks_path)
res   = gpd.read_file(reserves_path)

if parks.crs is None:
    raise ValueError("Parks.gpkg has no CRS defined.")
if res.crs is None:
    raise ValueError("Reserves_2024.gpkg has no CRS defined.")

if parks.crs.to_epsg() != CRS_EPSG:
    parks = parks.to_crs(CRS_EPSG)
if res.crs.to_epsg() != CRS_EPSG:
    res = res.to_crs(CRS_EPSG)

# Keep only valid polygonal geometries
parks = parks[parks.geometry.notna() & (~parks.geometry.is_empty)].copy()
res   = res[res.geometry.notna() & (~res.geometry.is_empty)].copy()

# buffer(0) is OK here (polygons)
parks["geometry"] = parks.geometry.buffer(0)
res["geometry"]   = res.geometry.buffer(0)

parks = parks[parks.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
res   = res[res.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

protected = pd.concat([parks[["geometry"]], res[["geometry"]]], ignore_index=True)
protected = gpd.GeoDataFrame(protected, geometry="geometry", crs=CRS_EPSG)

print(f"\nProtected polygons (parks + reserves): {len(protected)}")

# ---- 3a) Binary indicator: intersects yes/no
tz_for_join = tz[["zone_id", "geometry"]].copy().reset_index(drop=True)

join = gpd.sjoin(
    tz_for_join,
    protected,
    how="left",
    predicate="intersects"
)

zones_protected = join.loc[join["index_right"].notna(), "zone_id"].unique()
tz["F3_outdoor_protected"] = tz["zone_id"].isin(zones_protected).astype(int)

print("\nExample F3_outdoor_protected:")
print(tz[["zone_id", "F3_outdoor_protected"]].head())
print(f"Zones with protected areas: {int(tz['F3_outdoor_protected'].sum())} / {len(tz)}")

# ---- 3b) Protected area within zone (km²)
# Dissolve protected to reduce overlay complexity (one multipolygon)
protected_union = protected.dissolve()

prot_int = gpd.overlay(
    tz_min,                       # zone_id + geometry
    protected_union[["geometry"]],
    how="intersection"
)

# Area in km²
prot_int["prot_area_km2"] = prot_int.geometry.area / 1_000_000.0

protected_area_per_zone = (
    prot_int
    .groupby("zone_id")["prot_area_km2"]
    .sum()
    .reindex(tz["zone_id"], fill_value=0)
)

tz["F3_outdoor_protected_area"] = protected_area_per_zone.values

print("\nExample F3_outdoor_protected_area (km²):")
print(tz[["zone_id", "F3_outdoor_protected_area"]].head())

# =========================================================
# 4) Save updated TZ (overwrite)
# =========================================================
tz.to_file(tz_path, driver="GPKG")
print(f"\nSaved updated TZ.gpkg to: {tz_path}")
print("Added/updated columns: F3_outdoor_walk, F3_outdoor_protected, F3_outdoor_protected_area")

# =========================================================
# 5) Quick maps
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

tz.plot(
    column="F3_outdoor_walk",
    ax=axes[0],
    legend=True,
    linewidth=0,
    edgecolor="none"
)
axes[0].set_title("F3_outdoor_walk — Wanderweg length per zone (km)")
axes[0].axis("off")

tz.plot(
    column="F3_outdoor_protected",
    ax=axes[1],
    legend=True,
    linewidth=0,
    edgecolor="none"
)
axes[1].set_title("F3_outdoor_protected — Protected area present (0/1)")
axes[1].axis("off")

tz.plot(
    column="F3_outdoor_protected_area",
    ax=axes[2],
    legend=True,
    linewidth=0,
    edgecolor="none"
)
axes[2].set_title("F3_outdoor_protected_area — Protected area within zone (km²)")
axes[2].axis("off")

plt.tight_layout()
plt.show()


### Water-related outdoor environment (rivers & lakes)

For the **water-related** part of the outdoor domain I combine official lake geometries from swissTLM3D with river networks from OpenStreetMap, and aggregate everything to traffic zones.

---

#### 1. Lake polygons and shoreline length (F3_outdoor_lake, F3_outdoor_lake_bin)

**Source (official):**

- `swissTLM3D_2025/gewaesser/StillWater.gpkg`  
  - Contains **linear** segments of still water (mainly `objektart = "See"` and `Seeinsel`).

**Pre-processing:**

1. Load `StillWater.gpkg`, keep valid line geometries.
2. Group segments by lake identity (using `name` / id),  
   merge all segments of the same lake into one geometry and **reconstruct polygons**:
   - dissolve/union linework per lake,
   - polygonize to obtain closed lake polygons.
3. Save the resulting polygons as  
   `swissTLM3D_2025/gewaesser/StillWater_polygons.gpkg`  
   (fields: `group_id`, `name`, `objektart`, `geometry`).

**Shoreline length per zone (F3_outdoor_lake):**

4. Load traffic zones `TZ/TZ.gpkg` (CRS **EPSG:2056**, polygons).
5. Load `StillWater_polygons.gpkg` and ensure the same CRS (EPSG:2056).
6. Compute **lake boundaries** with `geometry.boundary` (shoreline as lines).
7. Intersect zones and shoreline (overlay) and compute the length of each resulting line segment in **meters**.
8. Aggregate shoreline length per `zone_id` and store as:

- `F3_outdoor_lake` = total length of lake shoreline (m) in each traffic zone.

**Binary lake presence (F3_outdoor_lake_bin):**

9. Create a binary indicator:

- `F3_outdoor_lake_bin = 1` if `F3_outdoor_lake > 0`, else `0`.

Both variables are written back to `TZ/TZ.gpkg` and mapped for inspection.

---

#### 2. River length per zone (F3_outdoor_river) and lake area (F3_lake_area)

**Rivers (OSM-based):**

- Input: `Rivers/Rivers.geojson` (downloaded via Overpass) with:
  - `waterway = river | stream | canal`
  - in **Switzerland, Liechtenstein, Campione d’Italia, Büsingen am Hochrhein**.

**Conversion to GeoPackage and cleaning:**

1. Load `Rivers.geojson` and keep only valid **LineString/MultiLineString** features.
2. Drop problematic / overly verbose attribute columns (e.g. `fixme`, long tag columns) to make the layer writable.
3. Reproject to **EPSG:2056**.
4. Save as `Rivers/Rivers.gpkg`.

**Remove river segments inside lakes:**

5. Load `Rivers/Rivers.gpkg` and `StillWater_polygons.gpkg` (both in EPSG:2056).
6. Build a single lake geometry (`union_all` / `unary_union` of lake polygons).
7. For each river geometry, compute `difference(river, lakes_union)` to obtain only the parts **outside** lakes.

**River length per zone (F3_outdoor_river):**

8. Intersect the cleaned river network (outside lakes) with traffic zones (overlay).
9. Compute the length of the intersected river segments in **kilometres**.
10. Aggregate by `zone_id`:

- `F3_outdoor_river` = total length of rivers/streams/canals (km) per zone, excluding sections inside lakes.

This variable is merged back into `TZ/TZ.gpkg` and visualised as a choropleth.

**Lake area per zone (F3_lake_area):**

11. Intersect lake polygons (`StillWater_polygons.gpkg`) with traffic zones.
12. Compute the area of the intersected polygons in **square metres**.
13. Aggregate by `zone_id`:

- `F3_lake_area` = total lake area (m²) inside each traffic zone.

`F3_lake_area` is also saved into `TZ/TZ.gpkg` and can be used as an additional water-related attribute (e.g. for the outdoor or landscape dimension).


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import unary_union, polygonize
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
base = Path(".")
TZ_PATH        = base / "TZ" / "TZ.gpkg"
STILL_PATH     = base / "swissTLM3D_2025" / "gewaesser" / "StillWater.gpkg"
LAKES_POLY_OUT = base / "swissTLM3D_2025" / "gewaesser" / "StillWater_polygons.gpkg"

CRS_EPSG = 2056

# =========================================================
# PARAMETERS
# =========================================================
WINSOR_Q = 0.98  # cap at 98th percentile (set to 0.99 if you really want 99th)

# =========================================================
# 1) Build lake polygons from swissTLM3D StillWater linework
# =========================================================
print(f"Reading StillWater from: {STILL_PATH}")
still = gpd.read_file(STILL_PATH)
print(f"StillWater features (raw): {len(still)}")

if still.crs is None:
    raise ValueError("StillWater.gpkg has no CRS defined.")
if still.crs.to_epsg() != CRS_EPSG:
    still = still.to_crs(CRS_EPSG)
    print("Reprojected StillWater to EPSG:2056")

# Keep only valid linear geometries
still = still[still.geometry.notna() & (~still.geometry.is_empty)].copy()
still = still[still.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
print(f"Valid line features: {len(still)}")

# Robust grouping key
# - use 'name' if present
# - otherwise fall back to 'id' if present
# - otherwise fall back to the row index
if "name" in still.columns:
    name_series = still["name"]
else:
    name_series = pd.Series([None] * len(still), index=still.index)

if "id" in still.columns:
    id_series = still["id"].astype(str)
else:
    id_series = still.index.astype(str)

still["group_id"] = name_series.fillna(id_series)

print(f"Distinct lake groups (name/id): {still['group_id'].nunique()}")

rows = []
for gid, sub in still.groupby("group_id"):
    lake_name = None
    if "name" in sub.columns:
        vals = sub["name"].dropna().unique()
        lake_name = vals[0] if len(vals) > 0 else None

    objektart = None
    if "objektart" in sub.columns:
        vals = sub["objektart"].dropna().unique()
        objektart = vals[0] if len(vals) > 0 else None

    merged = unary_union(sub.geometry.values)

    # polygonize needs iterable of linework
    if hasattr(merged, "geoms"):
        linework = [g for g in merged.geoms if g.geom_type in ("LineString", "MultiLineString")]
    else:
        linework = [merged] if merged.geom_type in ("LineString", "MultiLineString") else []

    if not linework:
        continue

    polys = list(polygonize(linework))
    if not polys:
        continue

    poly_union = unary_union(polys)

    # Keep only polygonal results
    if poly_union.geom_type not in ("Polygon", "MultiPolygon"):
        continue

    rows.append(
        {"group_id": gid, "name": lake_name, "objektart": objektart, "geometry": poly_union}
    )

lakes_poly = gpd.GeoDataFrame(rows, geometry="geometry", crs=still.crs)
print(f"Lake groups successfully polygonized: {len(lakes_poly)}")

LAKES_POLY_OUT.parent.mkdir(parents=True, exist_ok=True)
lakes_poly.to_file(LAKES_POLY_OUT, driver="GPKG")
print(f"Saved polygonized lakes to: {LAKES_POLY_OUT}")

# Quick plot (optional)
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
lakes_poly.plot(ax=ax, linewidth=0, edgecolor="none")
ax.set_title("Polygonized lakes from swissTLM3D StillWater")
ax.set_axis_off()
plt.tight_layout()
plt.show()

# =========================================================
# 2) Load traffic zones
# =========================================================
print(f"\nReading traffic zones from: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
print(f"Traffic zones: {len(tz)}")

if "zone_id" not in tz.columns:
    raise ValueError("TZ.gpkg: column 'zone_id' not found.")
tz["zone_id"] = tz["zone_id"].astype(int)

if tz.crs is None or tz.crs.to_epsg() != CRS_EPSG:
    tz = tz.to_crs(CRS_EPSG)
    print("Reprojected TZ to EPSG:2056")

tz = tz[tz.geometry.notna() & (~tz.geometry.is_empty)].copy()
tz["geometry"] = tz.geometry.buffer(0)

tz_min = tz[["zone_id", "geometry"]].copy()

# =========================================================
# 3) F3_outdoor_lake: shoreline length per zone (meters)
#    + F3_outdoor_lake_bin: presence/absence
# =========================================================
print(f"\nReading lake polygons from: {LAKES_POLY_OUT}")
lakes = gpd.read_file(LAKES_POLY_OUT)
print(f"Lake polygons: {len(lakes)}")

if lakes.crs is None:
    raise ValueError("StillWater_polygons.gpkg has no CRS defined.")
if lakes.crs.to_epsg() != CRS_EPSG:
    lakes = lakes.to_crs(CRS_EPSG)
    print("Reprojected lakes to EPSG:2056")

lakes = lakes[lakes.geometry.notna() & (~lakes.geometry.is_empty)].copy()
lakes["geometry"] = lakes.geometry.buffer(0)
lakes = lakes[lakes.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# Shoreline as boundary lines
shore = lakes[["geometry"]].copy()
shore["geometry"] = shore.geometry.boundary
shore = shore[shore.geometry.notna() & (~shore.geometry.is_empty)].copy()
shore = shore[shore.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
print(f"Shoreline line features: {len(shore)}")

# Intersect TZ with shoreline
shore_int = gpd.overlay(
    tz_min,
    shore,
    how="intersection",
    keep_geom_type=False
)

shore_int = shore_int[shore_int.geometry.notna() & (~shore_int.geometry.is_empty)].copy()
shore_int = shore_int[shore_int.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
print(f"Shoreline segments after TZ intersection: {len(shore_int)}")

shore_int["length_m"] = shore_int.geometry.length

lake_len_per_zone = (
    shore_int.groupby("zone_id")["length_m"]
    .sum()
    .reindex(tz["zone_id"], fill_value=0.0)
)

tz["F3_outdoor_lake_raw"] = lake_len_per_zone.values

# Winsorize / cap upper tail (to reduce dominance of very large shoreline zones)
cap = tz["F3_outdoor_lake_raw"].quantile(WINSOR_Q)
tz["F3_outdoor_lake"] = tz["F3_outdoor_lake_raw"].clip(upper=cap)

tz["F3_outdoor_lake_bin"] = (tz["F3_outdoor_lake"] > 0).astype(int)

print("\n--- Lake shoreline summary ---")
print(f"Zones with shoreline > 0: {int((tz['F3_outdoor_lake_raw'] > 0).sum())} / {len(tz)}")
print(f"Raw max shoreline (m): {tz['F3_outdoor_lake_raw'].max():.2f}")
print(f"Cap at q={WINSOR_Q:.2f}: {cap:.2f} m")
print(f"Capped max shoreline (m): {tz['F3_outdoor_lake'].max():.2f}")

# =========================================================
# 4) F3_lake_area: lake area per zone (m²)
# =========================================================
lake_tz = gpd.overlay(
    lakes[["geometry"]],
    tz_min,
    how="intersection"
)

lake_tz = lake_tz[lake_tz.geometry.notna() & (~lake_tz.geometry.is_empty)].copy()
lake_tz = lake_tz[lake_tz.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

lake_tz["area_m2"] = lake_tz.geometry.area

area_per_zone = (
    lake_tz.groupby("zone_id")["area_m2"]
    .sum()
    .reindex(tz["zone_id"], fill_value=0.0)
)

tz["F3_lake_area"] = area_per_zone.values

print("\n--- Lake area summary ---")
print(f"Zones with lake area > 0: {int((tz['F3_lake_area'] > 0).sum())} / {len(tz)}")
print(f"Max lake area (m²): {tz['F3_lake_area'].max():.0f}")

# =========================================================
# 5) Save TZ
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\nSaved updated TZ.gpkg to: {TZ_PATH}")
print("Added/updated columns: F3_outdoor_lake_raw, F3_outdoor_lake, F3_outdoor_lake_bin, F3_lake_area")

# =========================================================
# 6) Maps (optional)
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

tz.plot(column="F3_outdoor_lake", ax=axes[0], legend=True, linewidth=0, edgecolor="none")
axes[0].set_title("F3_outdoor_lake — shoreline length (m, capped)")
axes[0].axis("off")

tz.plot(column="F3_outdoor_lake_bin", ax=axes[1], legend=True, linewidth=0, edgecolor="none")
axes[1].set_title("F3_outdoor_lake_bin — lake present (0/1)")
axes[1].axis("off")

tz.plot(column="F3_lake_area", ax=axes[2], legend=True, linewidth=0, edgecolor="none")
axes[2].set_title("F3_lake_area — lake area within zone (m²)")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

base = Path(".")

TZ_PATH        = base / "TZ" / "TZ.gpkg"
RIVERS_GPKG    = base / "OSM" / "Outdoor"  / "Rivers" / "Rivers.gpkg"
LAKES_POLY_GPK = base / "swissTLM3D_2025" / "gewaesser" / "StillWater_polygons.gpkg"

CRS_EPSG = 2056


# ----------------------------------------------------------
# Helper: force 2D (drops Z if present)
# ----------------------------------------------------------
def force_2d(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Drops Z dimension (LineStringZ -> LineString, etc.) to make overlay more robust.
    Uses shapely.force_2d if available, otherwise uses WKB roundtrip.
    """
    try:
        from shapely import force_2d as _force_2d
        gdf = gdf.copy()
        gdf["geometry"] = gdf.geometry.apply(lambda geom: _force_2d(geom) if geom is not None else None)
        return gdf
    except Exception:
        # fallback: WKB roundtrip with output_dimension=2
        try:
            import shapely.wkb
            gdf = gdf.copy()
            gdf["geometry"] = gdf.geometry.apply(
                lambda geom: shapely.wkb.loads(shapely.wkb.dumps(geom, output_dimension=2))
                if geom is not None else None
            )
            return gdf
        except Exception:
            # If even that fails, just return unchanged
            return gdf


# ----------------------------------------------------------
# Helper: read the correct rivers layer
# ----------------------------------------------------------
def read_rivers_lines(path: Path) -> gpd.GeoDataFrame:
    """
    Reads rivers from a GPKG created by your Overpass converter.
    Typically it has layers: points / lines / polygons / other.
    We want the 'lines' layer if present.
    """
    # Try common layer names
    for layer in ["lines", "line", "rivers_lines", "waterway_lines"]:
        try:
            gdf = gpd.read_file(path, layer=layer)
            if len(gdf) > 0:
                print(f"✅ Loaded rivers from layer: '{layer}' ({len(gdf)} rows)")
                return gdf
        except Exception:
            pass

    # If no layer found, fall back to default read
    gdf = gpd.read_file(path)
    print(f"⚠️ Loaded rivers from default layer ({len(gdf)} rows).")
    return gdf


# ==========================================================
# 1) Load TZ
# ==========================================================
print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
print(f"   Traffic zones: {len(tz)}")

if "zone_id" not in tz.columns:
    raise ValueError("TZ.gpkg: 'zone_id' not found.")

tz["zone_id"] = tz["zone_id"].astype(int)

if tz.crs is None or tz.crs.to_epsg() != CRS_EPSG:
    tz = tz.to_crs(CRS_EPSG)
    print("   Reprojected TZ to EPSG:2056")

# polygons -> buffer(0) ok
tz = tz[tz.geometry.notna() & (~tz.geometry.is_empty)].copy()
tz["geometry"] = tz.geometry.buffer(0)

tz_min = tz[["zone_id", "geometry"]].copy().reset_index(drop=True)


# ==========================================================
# 2) Load Rivers (LINES) and clean
# ==========================================================
print(f"\n📂 Reading Rivers: {RIVERS_GPKG}")
rivers = read_rivers_lines(RIVERS_GPKG)

print(f"   Rivers CRS: {rivers.crs}")
if rivers.crs is None:
    raise ValueError("Rivers.gpkg has no CRS defined.")
if rivers.crs.to_epsg() != CRS_EPSG:
    rivers = rivers.to_crs(CRS_EPSG)
    print("   Reprojected Rivers to EPSG:2056")

rivers = rivers[rivers.geometry.notna() & (~rivers.geometry.is_empty)].copy()

# IMPORTANT: accept LineStringZ / MultiLineStringZ too
gt = rivers.geometry.geom_type
rivers = rivers[gt.str.contains("LineString", na=False)].copy()

# Optional but recommended: drop Z dimension
rivers = force_2d(rivers)

print(f"   Rivers line features kept: {len(rivers)}")
if len(rivers) == 0:
    raise RuntimeError("No river line geometries found. You likely read the wrong layer in the GPKG.")


# ==========================================================
# 3) Load Lakes polygons
# ==========================================================
print(f"\n📂 Reading lake polygons: {LAKES_POLY_GPK}")
lakes = gpd.read_file(LAKES_POLY_GPK)

print(f"   Lakes CRS: {lakes.crs}")
if lakes.crs is None:
    raise ValueError("StillWater_polygons.gpkg has no CRS defined.")
if lakes.crs.to_epsg() != CRS_EPSG:
    lakes = lakes.to_crs(CRS_EPSG)
    print("   Reprojected Lakes to EPSG:2056")

lakes = lakes[lakes.geometry.notna() & (~lakes.geometry.is_empty)].copy()
lakes = lakes[lakes.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
lakes["geometry"] = lakes.geometry.buffer(0)


# ==========================================================
# 4) Rivers ∩ TZ  (total river length in each zone)
# ==========================================================
print("\n🔀 Overlay: Rivers ∩ TZ ...")
rz = gpd.overlay(
    rivers[["geometry"]],
    tz_min,
    how="intersection",
    keep_geom_type=False
)

rz = rz[rz.geometry.notna() & (~rz.geometry.is_empty)].copy()
rz = rz[rz.geometry.geom_type.str.contains("LineString", na=False)].copy()

print(f"   Segments after Rivers∩TZ: {len(rz)}")

if len(rz) == 0:
    # Debug prints to confirm extents overlap
    print("DEBUG tz bounds   :", tz.total_bounds)
    print("DEBUG rivers bounds:", rivers.total_bounds)
    raise RuntimeError("No Rivers∩TZ segments. If bounds overlap, geometries may be invalid or empty after cleaning.")

rz["len_km"] = rz.length / 1000.0

len_tot = (
    rz.groupby("zone_id")["len_km"]
      .sum()
      .reset_index(name="len_river_total")
)


# ==========================================================
# 5) Rivers ∩ TZ ∩ Lakes (river segments inside lakes)
# ==========================================================
print("\n🌊 Overlay: (Rivers∩TZ) ∩ Lakes ...")

rz_lakes = gpd.overlay(
    rz[["zone_id", "geometry"]],
    lakes[["geometry"]],
    how="intersection",
    keep_geom_type=False
)

rz_lakes = rz_lakes[rz_lakes.geometry.notna() & (~rz_lakes.geometry.is_empty)].copy()
rz_lakes = rz_lakes[rz_lakes.geometry.geom_type.str.contains("LineString", na=False)].copy()

print(f"   Segments inside lakes: {len(rz_lakes)}")

if len(rz_lakes) > 0:
    rz_lakes["len_km_lake"] = rz_lakes.length / 1000.0
    len_in_lake = (
        rz_lakes.groupby("zone_id")["len_km_lake"]
                .sum()
                .reset_index(name="len_river_in_lake")
    )
else:
    len_in_lake = pd.DataFrame({"zone_id": [], "len_river_in_lake": []})


# ==========================================================
# 6) Combine: river length outside lakes
# ==========================================================
len_merged = len_tot.merge(len_in_lake, on="zone_id", how="left")
len_merged["len_river_in_lake"] = len_merged["len_river_in_lake"].fillna(0.0)

len_merged["F3_outdoor_river"] = (
    len_merged["len_river_total"] - len_merged["len_river_in_lake"]
).clip(lower=0.0)

print("\n✅ Example F3_outdoor_river:")
print(len_merged[["zone_id", "len_river_total", "len_river_in_lake", "F3_outdoor_river"]].head())


# ==========================================================
# 7) Merge into TZ
# ==========================================================
tz = tz.merge(len_merged[["zone_id", "F3_outdoor_river"]], on="zone_id", how="left")
tz["F3_outdoor_river"] = tz["F3_outdoor_river"].fillna(0.0)

print(f"\nZones with F3_outdoor_river > 0: {(tz['F3_outdoor_river'] > 0).sum()} / {len(tz)}")


# ==========================================================
# 8) Save + plot
# ==========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with F3_outdoor_river to: {TZ_PATH}")

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(
    column="F3_outdoor_river",
    ax=ax,
    legend=True,
    edgecolor="none",
    linewidth=0,
)
ax.set_title("F3_outdoor_river (km of rivers per zone, excluding segments inside lakes)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


### EDA features outdoor (POI)

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
BASE = Path(".")  # run from your Data folder
MAX_ROWS_DEFAULT = 5

# =========================================================
# HELPERS
# =========================================================
def list_gpkg_layers(path: Path):
    """
    Return list of layers in a GPKG, if any. If reading fails, return [].
    """
    try:
        import fiona
        return list(fiona.listlayers(path))
    except Exception:
        return []


def explore_gpkg(
    path,
    layer=None,
    max_rows=MAX_ROWS_DEFAULT,
    show_value_counts_col=None,
    title=None,
    head_cols=None,
):
    """
    Lightweight EDA for a GeoPackage.

    - Prints: layer(s), row count, CRS, geometry type distribution, columns, head().
    - Optionally prints value_counts for one categorical column (e.g., 'objektart', 'type').
    """
    path = Path(path)

    print("\n" + "=" * 90)
    print(f"File: {path}" + (f"  |  {title}" if title else ""))
    print("=" * 90)

    if not path.exists():
        print("⚠️  File NOT found. Check the path.")
        return None

    layers = list_gpkg_layers(path)
    if layers:
        print(f"Available layers ({len(layers)}): {layers}")
    else:
        print("Available layers: (not listed / single-layer / listing failed)")

    try:
        if layer is None:
            gdf = gpd.read_file(path)
            used_layer = "(default)"
        else:
            gdf = gpd.read_file(path, layer=layer)
            used_layer = layer
    except Exception as e:
        print(f"❌ Error reading {path} (layer={layer}):\n{e}")
        return None

    print(f"\nLoaded layer: {used_layer}")
    print(f"Number of features: {len(gdf)}")
    print(f"CRS: {gdf.crs}")

    # geometry type distribution
    if "geometry" in gdf.columns:
        gt = gdf.geometry.geom_type.value_counts(dropna=False)
        print("\nGeometry types:")
        print(gt)

    # column list
    print("\nColumns:")
    print(list(gdf.columns))

    # head
    if head_cols is not None:
        cols = [c for c in head_cols if c in gdf.columns] + (["geometry"] if "geometry" in gdf.columns else [])
        print(f"\nHead (selected columns): {cols}")
        print(gdf[cols].head(max_rows))
    else:
        print("\nHead:")
        print(gdf.head(max_rows))

    # value counts
    if show_value_counts_col is not None:
        if show_value_counts_col in gdf.columns:
            print(f"\nValue counts for '{show_value_counts_col}' (top 30):")
            print(gdf[show_value_counts_col].value_counts(dropna=False).head(30))
        else:
            print(f"\n⚠️  Column '{show_value_counts_col}' not found.")

    return gdf


# =========================================================
# 1) DATA INVENTORY / EDA
# =========================================================
glacier_path   = BASE / "GEOADMIN" / "Glacier_2016" / "AAGlacier.gpkg"
cableways_path = BASE / "GEOADMIN" / "Cableways_2024" / "Cableways.gpkg"
parks_path     = BASE / "GEOADMIN" / "Parks_2025" / "Parks.gpkg"
reserves_path  = BASE / "GEOADMIN" / "Reserves_2024.gpkg"
hiking_path    = BASE / "GEOADMIN" / "Hiking_2023.gpkg"

tlm_river_path = BASE / "swissTLM3D_2025" / "gewaesser" / "River.gpkg"
tlm_still_path = BASE / "swissTLM3D_2025" / "gewaesser" / "StillWater.gpkg"
divuse_path    = BASE / "swissTLM3D_2025" / "Areal" / "DiverseUse.gpkg"
leisuse_path   = BASE / "swissTLM3D_2025" / "Areal" / "LeisureUse.gpkg"
singleobj_path = BASE / "swissTLM3D_2025" / "EO" / "SingleObjects.gpkg"

# --- GEOADMIN
explore_gpkg(glacier_path,   title="Glaciers 2016", head_cols=["gid","name","area_km2","length_km"])
explore_gpkg(cableways_path, title="Cableways winter 2024", show_value_counts_col="type", head_cols=["name","type"])
explore_gpkg(parks_path,     title="Parks 2025", head_cols=["ObjNummer","Name","Kategorie","Status","Shape_Area","Shape_Leng"])
explore_gpkg(reserves_path,  title="Pro Natura reserves 2024", head_cols=["Objektnummer","Name"])
explore_gpkg(hiking_path,    title="Hiking trails 2023", show_value_counts_col="wanderwege", head_cols=["id","objektart","wanderwege","befahrbarkeit"])

# --- swissTLM3D_2025
explore_gpkg(tlm_river_path, title="swissTLM3D River.gpkg", show_value_counts_col="objektart", head_cols=["id","objektart","name","verlauf"])
explore_gpkg(tlm_still_path, title="swissTLM3D StillWater.gpkg", show_value_counts_col="objektart", head_cols=["id","objektart","name","wasserstand_wechselnd"])
explore_gpkg(divuse_path,    title="swissTLM3D DiverseUse.gpkg", show_value_counts_col="objektart", head_cols=["id","objektart","name"])
explore_gpkg(leisuse_path,   title="swissTLM3D LeisureUse.gpkg", show_value_counts_col="objektart", head_cols=["id","objektart","name"])
explore_gpkg(singleobj_path, title="swissTLM3D SingleObjects.gpkg", show_value_counts_col="objektart", head_cols=["id","objektart","name"])


# =========================================================
# 2) QUICK SUBSETS (optional, but useful)
#    - Cableways: keep non-ski types
#    - SingleObjects: keep "hard" outdoor POIs
# =========================================================
print("\n" + "=" * 90)
print("QUICK SUBSETS")
print("=" * 90)

# ---- 2.1 Cableways non-ski subset
cable = gpd.read_file(cableways_path)

if "type" not in cable.columns:
    raise ValueError("Cableways.gpkg: column 'type' not found.")

non_ski_types = ["aerial_cableway", "cable_car"]
cable_non_ski = cable[cable["type"].isin(non_ski_types)].copy()

print(f"\nCableways total: {len(cable)}")
print(f"Cableways non-ski (aerial_cableway + cable_car): {len(cable_non_ski)}")
print(cable_non_ski[["name", "type"]].head(10))

out_cable_non_ski = BASE / "GEOADMIN" / "Cableways_2024" / "Cableways_non_ski.gpkg"
out_cable_non_ski.parent.mkdir(parents=True, exist_ok=True)
cable_non_ski.to_file(out_cable_non_ski, driver="GPKG")
print(f"Saved: {out_cable_non_ski}")

# ---- 2.2 SingleObjects: waterfalls + caves
single = gpd.read_file(singleobj_path)

if "objektart" not in single.columns:
    raise ValueError("SingleObjects.gpkg: column 'objektart' not found.")

wanted_obj = ["Wasserfall", "Grotte, Hoehle"]
single_sel = single[single["objektart"].isin(wanted_obj)].copy()

print(f"\nSingleObjects total: {len(single)}")
print(f"Selected (Wasserfall, Grotte/Hoehle): {len(single_sel)}")
print(single_sel["objektart"].value_counts(dropna=False))
print(single_sel[["objektart", "name"]].head(10))

out_single_hard = BASE / "swissTLM3D_2025" / "EO" / "SingleObjects_outdoor_hard.gpkg"
out_single_hard.parent.mkdir(parents=True, exist_ok=True)
single_sel.to_file(out_single_hard, driver="GPKG")
print(f"Saved: {out_single_hard}")


## Outdoor hard features — OSM extraction, zonal indicators, and validation

This step constructs **Outdoor Hard** supply indicators from OpenStreetMap and aggregates them to **NPVM 2017 traffic zones** (TZ).  
It focuses on **supply-side presence/counts** only.

### OSM feature selection

We extract OSM objects in Switzerland (plus nearby enclaves) using the following tag filters:

- `aerialway ∈ {zip_line, gondola, cable_car}`
- `tourism ∈ {alpine_hut, wilderness_hut, viewpoint}`
- `natural ∈ {glacier, cave_entrance}`
- `waterway = waterfall`

### Zonal aggregation (counts)

For each traffic zone, we compute **counts of unique OSM features intersecting the zone polygon**.

- Spatial join predicate: `intersects`
- Only true matches are counted (`how="inner"`, avoiding the “always ≥ 1” counting bug)
- Uniqueness is enforced using OSM `@id` when available (fallback: feature index)

This produces per-tag zonal counts (e.g., glaciers, caves, waterfalls, huts, aerialways, viewpoints).

### Aggregated indicators (the three final outputs)

We derive three summary indicators:

1. **`F3_outdoor_hard_count` (OSM total count)**  
   Total number of Outdoor Hard features per zone (sum of all included OSM categories).

2. **`F3_outdoor_hard_diversity` (OSM diversity)**  
   Number of **macro-categories** present in the zone. Diversity is computed over:
   - **Aerialway** (zip_line + gondola + cable_car) as **one** macro-category  
   - **Huts** (alpine_hut + wilderness_hut) as **one** macro-category  
   - Viewpoint, glacier, cave entrance, waterfall as separate macro-categories

3. **`F3_outdoor_count_off` (official total count)**  
   Total number of official Outdoor Hard objects per zone, computed as the sum of the official zonal-count columns already created earlier (e.g., official glaciers, caves, waterfalls, cableways, huts), used as a benchmark.

### Validation against official datasets (correlations)

To assess consistency between OSM and official sources, we correlate **official zonal counts vs OSM zonal counts** across Switzerland, excluding Liechtenstein and enclaves (filtered via `N_KT`). The resulting correlations are:

- **Glaciers (official vs OSM, counts)**
  - Pearson: **0.955**
  - Spearman: **0.939**

- **Waterfalls (official vs OSM, counts)**
  - Pearson: **0.665**
  - Spearman: **0.523**

- **Caves (official vs OSM, counts)**
  - Pearson: **0.400**
  - Spearman: **0.439**

- **Non-ski cableways (official vs OSM, counts)**
  - Pearson: **0.876**
  - Spearman: **0.859**

- **Huts (official Schützhütte vs OSM `tourism=alpine_hut`, counts)**
  - Pearson: **0.346**
  - Spearman: **0.324**

These statistics serve as a sanity check of spatial agreement between OSM-derived and official inventories at the traffic-zone level.


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

base = Path(".")

# =========================================================
# INPUT / OUTPUT
# =========================================================
tlm_path      = base / "swissTLM3D_2025" / "SWISSTLM3D_2025.gpkg"
tlm_bld_layer = "tlm_bauten_gebaeude_footprint"

out_path = base / "swissTLM3D_2025" / "EO" / "TLM_schuetzhuette.gpkg"
out_path.parent.mkdir(parents=True, exist_ok=True)

TARGET_EPSG = 2056

# =========================================================
# HELPERS
# =========================================================
def ensure_epsg(gdf: gpd.GeoDataFrame, epsg: int, label: str) -> gpd.GeoDataFrame:
    """Ensure CRS is defined and reproject to the target EPSG if needed."""
    if gdf.crs is None:
        raise ValueError(f"❌ {label} has no CRS defined.")
    if gdf.crs.to_epsg() != epsg:
        gdf = gdf.to_crs(epsg)
        print(f"   ↪ Reprojected {label} to EPSG:{epsg}")
    return gdf


def drop_empty_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Drop missing/empty geometries."""
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf


def normalize_german_text(series: pd.Series) -> pd.Series:
    """
    Robust normalization for German strings:
    - lowercase
    - strip
    - replace umlauts/ß
    - also collapse ae/oe/ue variants to a/o/u
    - collapse whitespace
    """
    x = series.fillna("").astype(str).str.lower().str.strip()

    # umlauts -> base letters, ß -> ss
    x = (x.str.replace("ä", "a", regex=False)
           .str.replace("ö", "o", regex=False)
           .str.replace("ü", "u", regex=False)
           .str.replace("ß", "ss", regex=False))

    # ae/oe/ue -> a/o/u (covers alternative spellings like "Schuetzhuette")
    x = (x.str.replace("ae", "a", regex=False)
           .str.replace("oe", "o", regex=False)
           .str.replace("ue", "u", regex=False))

    x = x.str.replace(r"\s+", " ", regex=True)
    return x


# =========================================================
# MAIN
# =========================================================
print("📂 Reading swissTLM3D building footprints...")
bld = gpd.read_file(tlm_path, layer=tlm_bld_layer)

print(f"   Building features: {len(bld)}")
print(f"   Columns: {list(bld.columns)}")

bld = ensure_epsg(bld, TARGET_EPSG, "swissTLM3D buildings")
bld = drop_empty_geometries(bld)

if "nutzung" not in bld.columns:
    raise ValueError("❌ Column 'nutzung' not found in the swissTLM3D building layer.")

# Robust filter on 'nutzung' for "Schutzhütte" (incl. spelling variants)
nutz_norm = normalize_german_text(bld["nutzung"])

# after normalization, variants should match "schutzhutte"
mask_hut = nutz_norm.str.contains(r"\bschutzhutte\b", regex=True)

huts = bld[mask_hut].copy()
print(f"\n🏠 Buildings with 'nutzung' = Schutzhütte (robust match): {len(huts)}")

# Inspect original values captured by the filter
if len(huts) > 0:
    print("\nOriginal 'nutzung' values found (unique):")
    print(
        huts["nutzung"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .to_string(index=False)
    )

# Save as POINTS (centroids) — convenient for zonal counts / joins
if len(huts) > 0:
    huts_pts = huts.copy()
    huts_pts["geometry"] = huts_pts.geometry.centroid

    # keep only a small, stable set of fields (if present)
    keep_cols = [c for c in ["uuid", "name", "objektart", "nutzung"] if c in huts_pts.columns]
    huts_pts = huts_pts[keep_cols + ["geometry"]].copy()

    huts_pts.to_file(out_path, driver="GPKG")
    print(f"\n💾 Saved hut centroids to: {out_path}")
else:
    print("\n(No features matched — nothing written.)")


In [ ]:
import geopandas as gpd
from pathlib import Path

base = Path(".")

outdoor_gpkg = base / "OSM" / "Outdoor" / "outdoor_hard.gpkg"

print(f"📂 Leggo outdoor_hard da: {outdoor_gpkg}")
gdf = gpd.read_file(outdoor_gpkg)
print(f"   Features: {len(gdf)}")
print(f"   Colonne disponibili: {list(gdf.columns)}\n")

cols_to_check = ["aerialway", "tourism", "natural", "waterway"]

for col in cols_to_check:
    if col in gdf.columns:
        print("=" * 70)
        print(f"🔎 Distribuzione valori per '{col}':")
        vc = gdf[col].value_counts(dropna=False).sort_values(ascending=False)
        print(vc)
        print()
    else:
        print("=" * 70)
        print(f"⚠️ Colonna '{col}' NON presente nel GeoPackage.")
        print()


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
base = Path(".")

TZ_PATH = base / "TZ" / "TZ.gpkg"

# OFFICIAL
GLACIER_OFF_PATH = base / "GEOADMIN" / "Glacier_2016" / "AAGlacier.gpkg"
CABLE_OFF_PATH   = base / "GEOADMIN" / "Cableways_2024" / "Cableways_non_ski.gpkg"
SINGLE_HARD_PATH = base / "swissTLM3D_2025" / "EO" / "SingleObjects_outdoor_hard.gpkg"
HUTS_OFF_PATH    = base / "swissTLM3D_2025" / "EO" / "TLM_schuetzhuette.gpkg"

# OSM
OUTDOOR_OSM_PATH = base / "OSM" / "Outdoor" / "outdoor_hard.gpkg"

CRS_EPSG = 2056

# Regions to exclude from correlations (Liechtenstein + enclaves)
EXCLUDE_N_KT = {
    "Liechtenstein",
    "Campione d'Italia",
    "Büsingen am Hochrhein",
    "Enk.",
    "LIE",
}

# =========================================================
# UTILITIES
# =========================================================
def ensure_epsg_2056(gdf: gpd.GeoDataFrame, label: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError(f"❌ {label} has no CRS defined.")
    if gdf.crs.to_epsg() != CRS_EPSG:
        gdf = gdf.to_crs(CRS_EPSG)
        print(f"   ↪ Reprojected {label} to EPSG:{CRS_EPSG}")
    return gdf

def clean_geometries(gdf: gpd.GeoDataFrame, label: str) -> gpd.GeoDataFrame:
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"   🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def choose_id_col(gdf: gpd.GeoDataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in gdf.columns:
            return c
    return None

def safe_filter_eq(gdf: gpd.GeoDataFrame, col: str, value) -> gpd.GeoDataFrame:
    """Return gdf where gdf[col] == value; if col missing -> empty slice."""
    if col not in gdf.columns:
        return gdf.iloc[0:0].copy()
    return gdf[gdf[col] == value].copy()

def safe_filter_in(gdf: gpd.GeoDataFrame, col: str, values: list) -> gpd.GeoDataFrame:
    """Return gdf where gdf[col] in values; if col missing -> empty slice."""
    if col not in gdf.columns:
        return gdf.iloc[0:0].copy()
    return gdf[gdf[col].isin(values)].copy()

def count_unique_features_per_zone(
    tz: gpd.GeoDataFrame,
    g: gpd.GeoDataFrame,
    label: str,
    id_col: str | None = None,
    predicate: str = "intersects",
) -> pd.Series:
    """
    Count, for each zone_id, how many UNIQUE features intersect the zone polygon.
    - uses spatial join with how='inner'
    - uniqueness via id_col (if provided) else via feature index
    """
    if len(g) == 0:
        print(f"   ⚠ {label}: no features -> all zeros.")
        return pd.Series(0, index=tz["zone_id"], dtype="int32")

    tz_min = tz[["zone_id", "geometry"]].copy()

    if id_col is not None and id_col in g.columns:
        g_min = g[[id_col, "geometry"]].copy()
        g_min["_fid"] = g_min[id_col].astype(str)
    else:
        g_min = g[["geometry"]].copy()
        g_min["_fid"] = g_min.index.astype("int64").astype(str)

    sj = gpd.sjoin(tz_min, g_min[["_fid", "geometry"]], how="inner", predicate=predicate)
    counts = sj.groupby("zone_id")["_fid"].nunique()

    return counts.reindex(tz["zone_id"], fill_value=0).astype("int32")

def print_corr_pair(tz: gpd.GeoDataFrame, col_off: str, col_osm: str, label: str) -> None:
    """Pearson & Spearman correlations, excluding Liechtenstein + enclaves via N_KT."""
    if col_off not in tz.columns or col_osm not in tz.columns:
        print(f"⚠ {label}: missing columns: {col_off}, {col_osm}")
        return
    if "N_KT" not in tz.columns:
        raise ValueError("❌ TZ has no 'N_KT' column (needed to filter LIE/enclaves).")

    df = tz[[col_off, col_osm, "N_KT"]].copy()
    df = df[~df["N_KT"].isin(EXCLUDE_N_KT)].copy()
    df = df.dropna(subset=[col_off, col_osm])

    if len(df) == 0:
        print(f"⚠ {label}: no valid zones after filtering.")
        return

    std_off = df[col_off].std()
    std_osm = df[col_osm].std()

    print(f"\n📈 Correlation — {label} (CH, excl. LIE/enclaves)")
    print(f"   Zones used: {len(df)}")
    print(f"   Std {col_off}: {std_off:.6f}, Std {col_osm}: {std_osm:.6f}")

    if std_off == 0 or std_osm == 0:
        print("   ⚠ Constant series -> correlation undefined.")
        return

    pear = df[col_off].corr(df[col_osm], method="pearson")
    spear = df[col_off].corr(df[col_osm], method="spearman")
    print(f"   Pearson : {pear:.3f}")
    print(f"   Spearman: {spear:.3f}")

# =========================================================
# 1) LOAD TRAFFIC ZONES (TZ)  — IMPORTANT: DO NOT DROP OLD F3
# =========================================================
print(f"📂 Reading TZ from: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
print(f"   Traffic zones: {len(tz)}")

if "zone_id" not in tz.columns:
    raise ValueError("❌ TZ.gpkg missing column 'zone_id'.")
if "N_KT" not in tz.columns:
    raise ValueError("❌ TZ.gpkg missing column 'N_KT'.")

tz["zone_id"] = tz["zone_id"].astype(int)
tz = ensure_epsg_2056(tz, "TZ")
tz["geometry"] = tz.geometry.buffer(0)  # polygon fix
tz = clean_geometries(tz, "TZ")

print("✅ TZ loaded. NOTE: I will only OVERWRITE the columns computed below; I won't drop other existing F3_* columns.")

# =========================================================
# 2) LOAD OSM OUTDOOR HARD
# =========================================================
print(f"\n📂 Reading OSM outdoor_hard from: {OUTDOOR_OSM_PATH}")
osm = gpd.read_file(OUTDOOR_OSM_PATH)
print(f"   OSM features (raw): {len(osm)} | CRS: {osm.crs}")

if "@id" in osm.columns:
    n_before = len(osm)
    osm = osm.drop_duplicates(subset="@id")
    print(f"   Dedup @id: removed {n_before - len(osm)} duplicates.")

osm = ensure_epsg_2056(osm, "OSM outdoor_hard")
osm = clean_geometries(osm, "OSM outdoor_hard")
print(f"   Valid OSM outdoor_hard: {len(osm)}")

OSM_ID = "@id" if "@id" in osm.columns else None

# =========================================================
# 3) OFFICIAL DATASETS (COUNTS PER TZ) — suffix _off_*
# =========================================================
print("\n================ OFFICIAL COUNTS ================")

# --- GLACIERS (OFF)
print(f"\n📂 Reading official glaciers from: {GLACIER_OFF_PATH}")
gla_off = gpd.read_file(GLACIER_OFF_PATH)
print(f"   Official glaciers (raw): {len(gla_off)}")
gla_off = ensure_epsg_2056(gla_off, "Official glaciers")
gla_off = clean_geometries(gla_off, "Official glaciers")

gla_off_id = choose_id_col(gla_off, ["gid", "id", "uuid", "name"])
if gla_off_id is not None:
    n_before = len(gla_off)
    gla_off = gla_off.drop_duplicates(subset=gla_off_id)
    print(f"   Dedup OFF on '{gla_off_id}': removed {n_before - len(gla_off)}")

tz["F3_glacier_off_count"] = count_unique_features_per_zone(
    tz, gla_off, "Glaciers OFF", id_col=gla_off_id
).to_numpy()

# --- SINGLE OBJECTS (OFF) -> waterfalls + caves
print(f"\n📂 Reading official single objects from: {SINGLE_HARD_PATH}")
so = gpd.read_file(SINGLE_HARD_PATH)
print(f"   Official SingleObjects (raw): {len(so)}")
so = ensure_epsg_2056(so, "SingleObjects_outdoor_hard")
so = clean_geometries(so, "SingleObjects_outdoor_hard")

so_id = choose_id_col(so, ["id", "uuid", "objectid", "name"])
if so_id is not None:
    n_before = len(so)
    so = so.drop_duplicates(subset=so_id)
    print(f"   Dedup SingleObjects on '{so_id}': removed {n_before - len(so)}")

wfall_off = safe_filter_eq(so, "objektart", "Wasserfall")
cave_off  = safe_filter_eq(so, "objektart", "Grotte, Hoehle")
wfall_off = clean_geometries(wfall_off, "Official waterfalls")
cave_off  = clean_geometries(cave_off, "Official caves")

tz["F3_waterfall_off_count"] = count_unique_features_per_zone(
    tz, wfall_off, "Waterfalls OFF", id_col=so_id
).to_numpy()

tz["F3_cave_off_count"] = count_unique_features_per_zone(
    tz, cave_off, "Caves OFF", id_col=so_id
).to_numpy()

# --- CABLEWAYS (OFF) non-ski
print(f"\n📂 Reading official cableways from: {CABLE_OFF_PATH}")
cable_off = gpd.read_file(CABLE_OFF_PATH)
print(f"   Official cableways (raw): {len(cable_off)}")
cable_off = ensure_epsg_2056(cable_off, "Official cableways")
cable_off = clean_geometries(cable_off, "Official cableways")

cable_off_id = choose_id_col(cable_off, ["id", "uuid", "objectid", "name"])
if cable_off_id is not None:
    n_before = len(cable_off)
    cable_off = cable_off.drop_duplicates(subset=cable_off_id)
    print(f"   Dedup cableways on '{cable_off_id}': removed {n_before - len(cable_off)}")

tz["F3_cable_off_count"] = count_unique_features_per_zone(
    tz, cable_off, "Cableways OFF", id_col=cable_off_id
).to_numpy()

# --- HUTS (OFF) TLM_schuetzhuette
print(f"\n📂 Reading official huts from: {HUTS_OFF_PATH}")
huts_off = gpd.read_file(HUTS_OFF_PATH)
print(f"   Official huts (raw): {len(huts_off)}")
huts_off = ensure_epsg_2056(huts_off, "Official huts")
huts_off = clean_geometries(huts_off, "Official huts")

huts_off_id = choose_id_col(huts_off, ["uuid", "id", "egid", "objectid", "fid", "name"])
if huts_off_id is not None:
    n_before = len(huts_off)
    huts_off = huts_off.drop_duplicates(subset=huts_off_id)
    print(f"   Dedup huts on '{huts_off_id}': removed {n_before - len(huts_off)}")

tz["F3_hut_off_count"] = count_unique_features_per_zone(
    tz, huts_off, "Huts OFF", id_col=huts_off_id
).to_numpy()

# =========================================================
# 4) OSM SUBSETS (COUNTS PER TZ) — macro hut is ALWAYS F3_outdoor_hut_count; total is "hard"
# =========================================================
print("\n================ OSM COUNTS ================")

# aerialway subtypes
zipline  = safe_filter_eq(osm, "aerialway", "zip_line")
gondola  = safe_filter_eq(osm, "aerialway", "gondola")
cablecar = safe_filter_eq(osm, "aerialway", "cable_car")

# huts
alpine_hut = safe_filter_eq(osm, "tourism", "alpine_hut")
wild_hut   = safe_filter_eq(osm, "tourism", "wilderness_hut")

# other outdoor hard
viewpoint  = safe_filter_eq(osm, "tourism", "viewpoint")
glacier    = safe_filter_eq(osm, "natural", "glacier")
cave       = safe_filter_eq(osm, "natural", "cave_entrance")
waterfall  = safe_filter_eq(osm, "waterway", "waterfall")

# per-category columns (OSM)
osm_categories = {
    "F3_outdoor_zipline_count":            zipline,
    "F3_outdoor_gondola_count":            gondola,
    "F3_outdoor_cablecar_count":           cablecar,
    "F3_outdoor_alpine_hut_count":         alpine_hut,
    "F3_outdoor_wilderness_hut_count":     wild_hut,
    "F3_outdoor_viewpoint_count":          viewpoint,
    "F3_outdoor_glacier_count":            glacier,
    "F3_outdoor_cave_count":               cave,
    "F3_outdoor_waterfall_count":          waterfall,
}

print("📏 Computing OSM zonal counts (overwriting only these columns)...")
for col, gdf_cat in osm_categories.items():
    tz[col] = count_unique_features_per_zone(tz, gdf_cat, col, id_col=OSM_ID).to_numpy()

# macro-categories (OSM)
tz["F3_outdoor_aerialway_count"] = (
    tz["F3_outdoor_zipline_count"]
    + tz["F3_outdoor_gondola_count"]
    + tz["F3_outdoor_cablecar_count"]
).astype("int32")

# IMPORTANT: macro hut name fixed as requested
tz["F3_outdoor_hut_count"] = (
    tz["F3_outdoor_alpine_hut_count"]
    + tz["F3_outdoor_wilderness_hut_count"]
).astype("int32")

macro_cols = [
    "F3_outdoor_aerialway_count",
    "F3_outdoor_hut_count",
    "F3_outdoor_viewpoint_count",
    "F3_outdoor_glacier_count",
    "F3_outdoor_cave_count",
    "F3_outdoor_waterfall_count",
]

# total "hard" (OSM) + diversity
tz["F3_outdoor_hard_count"] = tz[macro_cols].sum(axis=1).astype("int32")
tz["F3_outdoor_hard_diversity"] = (tz[macro_cols] > 0).sum(axis=1).astype("int32")

# =========================================================
# 5) OFFICIAL AGGREGATE (suffix _off)
# =========================================================
official_cols = [
    "F3_glacier_off_count",
    "F3_cable_off_count",
    "F3_waterfall_off_count",
    "F3_cave_off_count",
    "F3_hut_off_count",
]

for c in official_cols:
    if c not in tz.columns:
        tz[c] = 0

tz["F3_outdoor_hard_count_off"] = tz[official_cols].sum(axis=1).astype("int32")

# =========================================================
# 6) CORRELATIONS (optional but useful)
# =========================================================
print("\n================ CORRELATIONS (OFF vs OSM) ================")
print_corr_pair(tz, "F3_glacier_off_count",   "F3_outdoor_glacier_count",   "Glaciers")
print_corr_pair(tz, "F3_waterfall_off_count", "F3_outdoor_waterfall_count", "Waterfalls")
print_corr_pair(tz, "F3_cave_off_count",      "F3_outdoor_cave_count",      "Caves")
# OFF cableways vs OSM aerialways (macro)
print_corr_pair(tz, "F3_cable_off_count",     "F3_outdoor_aerialway_count", "Cableways/Aerialways")
print_corr_pair(tz, "F3_hut_off_count",       "F3_outdoor_hut_count",       "Huts")

# =========================================================
# 7) SAVE (keeps all existing columns; only overwrites computed ones)
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved updated TZ.gpkg to: {TZ_PATH}")
print("✅ Done (no mass deletion of previous F3 columns).")

# =========================================================
# 8) QUICK MAPS
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F3_outdoor_hard_count", ax=ax, legend=True, edgecolor="none")
ax.set_title("F3_outdoor_hard_count (OSM)")
ax.set_axis_off()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F3_outdoor_hard_diversity", ax=ax, legend=True, edgecolor="none")
ax.set_title("F3_outdoor_hard_diversity (OSM)")
ax.set_axis_off()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F3_outdoor_hard_count_off", ax=ax, legend=True, edgecolor="none")
ax.set_title("F3_outdoor_hard_count_off (official)")
ax.set_axis_off()
plt.show()


## F3_outdoor_soft (OSM) — documentation (counts, diversity, and area)

This chunk documents the construction of **soft outdoor supply indicators** derived exclusively from **OpenStreetMap (OSM)** and aggregated to **NPVM 2017 Traffic Zones** (TZ). The resulting indicators represent a *supply-side* proxy of leisure opportunities, suitable as input factors \(F_h(j)\) in the attractivity framework.

### Scope and OSM extraction
The OSM layer `OSM/Outdoor/outdoor_soft` is extracted from Overpass and converted to GeoPackage. The selection targets **low-intensity outdoor leisure** elements, using the following key/value families:

- `leisure`: `park`, `playground`, `dog_park`, `beach_resort`, `marina`, `picnic_table`, `firepit`, `bird_hide`
- `tourism`: `picnic_site`

Note: OSM is community-maintained; attributes are not consistent across objects and can include many sparse columns.

### Pre-processing
All layers are harmonised to **EPSG:2056 (CH1903+ / LV95)**, and cleaned by removing null/empty geometries. OSM objects are deduplicated using `@id` when available.

Additionally, two rare mixed / ambiguous tags are excluded from the soft layer:
- `leisure = firepit;playground` (1 record)
- `leisure = common` (1 record)

### Attribute distributions (quality context)
In the current extraction (n = 35’895), the attribute distributions are dominated by two categories:

**Leisure distribution (value counts):**
- `playground`: 13’789  
- `picnic_table`: 7’861  
- `park`: 4’212  
- `firepit`: 3’954  
- `marina`: 449  
- `dog_park`: 132  
- `bird_hide`: 106  
- `beach_resort`: 45  
- `None` (missing / untagged): 5’345  

**Tourism distribution (value counts):**
- `picnic_site`: 5’586  
- minor residuals (e.g., `attraction`, `zoo`, `viewpoint`, …): very small counts  
- `None`: 30’280  

These distributions are used as a sanity check (coverage, dominance, and unexpected residual tags).

---

## Indicators engineered

### 1) `F3_outdoor_soft_count` (count)
For each TZ \(j\), we count the number of **unique OSM features** intersecting the zone polygon:
- Intersection predicate: `intersects`
- Uniqueness: by `@id` if available, otherwise by row index
- Objects intersecting multiple zones are counted in **each** intersected zone (intended behaviour)

**Important rule (no double counting of picnic sites):**  
`tourism=picnic_site` is treated as an additional soft category **only when it does not overlap any leisure feature**. Operationally:
- Compute the subset of `picnic_site`
- Remove any `picnic_site` geometries that intersect at least one object with `leisure ∈ SOFT_LEISURE_SET`
- Count the remaining `picnic_site` features

This avoids inflating counts where picnic sites are mapped as an “annotation” inside already mapped leisure places (parks, playgrounds, etc.).

### 2) `F3_outdoor_soft_diversity` (diversity)
For each TZ \(j\), diversity is the number of **soft categories present** (presence/absence), not the number of objects.

A category counts as “present” if the zone has at least one intersecting feature in that category.  
Diversity is computed as:
\[
F3\_outdoor\_soft\_diversity(j)=\sum_{c \in C} \mathbf{1}\{count_c(j) > 0\}
\]
where \(C\) is the set of soft categories used in this indicator.

**Note on category definition:**  
Diversity is computed across the **final soft categories**, not across subtypes or mixed tags.

### 3) `F3_soft_area` (area, OSM only)
This indicator measures the **total soft outdoor surface** (m²) per TZ from OSM, computed from **area-capable geometries only**.

**Included categories for the area version (restricted set):**
- `leisure`: `park`, `playground`, `dog_park`, `beach_resort`, `marina`
- `tourism`: `picnic_site` (after the “exclude if intersects leisure” rule, same as above)

**Geometry-type handling:**
- Only **(Multi)Polygon** geometries contribute to area.
- Points/lines are not force-buffered into polygons (to avoid arbitrary area inflation).  
  They are reported in QC (counts / distributions), but excluded from area aggregation.

**Avoiding overlap double counting (critical):**
OSM polygons can overlap (e.g., a playground polygon inside a park polygon). To avoid counting the same surface twice:
1. Build a single “soft area mask” by **unioning** all included polygons (unary union / dissolve).
2. Intersect this unioned geometry with each TZ polygon.
3. Compute area in **EPSG:2056** and sum:
\[
F3\_soft\_area(j)=\text{Area}\left( TZ_j \cap \bigcup_{k \in \text{soft polygons}} geom_k \right)
\]

This yields a conservative “unique surface” estimate.

---

## Output and storage
All derived indicators are joined back to the TZ layer by `zone_id` and persisted to:
- `TZ/TZ.gpkg`

The workflow produces:
- `F3_outdoor_soft_count`
- `F3_outdoor_soft_diversity`
- `F3_soft_area`

---

## Minimal QC (recommended)
Because ground-truth validation for OSM soft features is limited, QC focuses on:
- attribute distributions (leisure/tourism value_counts)
- zero-inflation and max values per indicator
- map inspection (spatial plausibility)

(Additional QC against official “public parks” polygons can be done separately, using `Oeffentliches Parkareal` from swissTLM3D Areal, but the soft-area indicator itself remains **OSM-only**.)


In [ ]:
import geopandas as gpd
from pathlib import Path

base = Path(".")

# Use the file you already prepared elsewhere (no CRS/format conversions here)
OUTDOOR_SOFT_PATH = base / "OSM" / "Outdoor" / "outdoor_soft.gpkg"
# Alternatively:
# OUTDOOR_SOFT_PATH = base / "OSM" / "Outdoor" / "outdoor_soft.geojson"

print(f"📂 Reading OSM outdoor_soft from: {OUTDOOR_SOFT_PATH}")

# Robust read
try:
    gdf = gpd.read_file(OUTDOOR_SOFT_PATH, on_invalid="ignore")
    print("   ✅ Read with pyogrio (on_invalid='ignore').")
except Exception as e:
    print(f"   ⚠️ pyogrio failed ({type(e).__name__}: {e}). Trying engine='fiona'...")
    gdf = gpd.read_file(OUTDOOR_SOFT_PATH, engine="fiona")
    print("   ✅ Read with engine='fiona'.")

print(f"   Features (raw): {len(gdf)}")
print(f"   CRS (as-is): {gdf.crs}")
print(f"   Columns: {list(gdf.columns)}")

# Value counts BEFORE cleaning/dedup
for col in ["leisure", "tourism"]:
    if col in gdf.columns:
        print(f"\n📊 '{col}' distribution (value_counts incl. NaN):")
        print(gdf[col].value_counts(dropna=False))
    else:
        print(f"\n⚠️ Column '{col}' not found.")

# Drop an optional column (attribute cleanup only)
if "fixme" in gdf.columns:
    gdf = gdf.drop(columns=["fixme"])
    print("\n🧹 Dropped column 'fixme'.")

# Deduplicate by OSM id if present
if "@id" in gdf.columns:
    n_before = len(gdf)
    gdf = gdf.drop_duplicates(subset="@id")
    print(f"🧽 Deduplicated by '@id': removed {n_before - len(gdf)} duplicates.")
else:
    print("ℹ️ No '@id' column -> deduplication skipped.")

# Geometry QC (light cleaning only)
before = len(gdf)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"🔍 Removed {before - len(gdf)} rows with NaN geometry.")

before = len(gdf)
gdf = gdf[~gdf.geometry.is_empty].copy()
print(f"🔍 Removed {before - len(gdf)} rows with empty geometry.")

print(f"✅ Features after QC cleaning: {len(gdf)}")


In [ ]:
# ==============================
# CHUNK — OSM outdoor_soft -> F3_outdoor_soft_* (per-category counts + total count + diversity)
# Rules:
# - exclude leisure values: "firepit;playground" and "common"
# - tourism=picnic_site: if intersects ANY leisure feature, drop it from the count
# - features intersecting multiple TZ are counted in each intersected TZ
# - save to TZ.gpkg + plot ONLY the total + diversity
# ==============================

import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

base = Path(".")

TZ_PATH           = base / "TZ" / "TZ.gpkg"
OUTDOOR_SOFT_GPKG  = base / "OSM" / "Outdoor" / "outdoor_soft.gpkg"

LEISURE_VALUES = {
    "beach_resort",
    "bird_hide",
    "dog_park",
    "firepit",
    "park",
    "picnic_table",
    "playground",
    "marina",
}
EXCLUDE_LEISURE_VALUES = {"firepit;playground", "common"}
TOURISM_VALUE = "picnic_site"

# ------------------------------
# Utilities
# ------------------------------
def ensure_2056(gdf, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label} has no CRS.")
    if gdf.crs.to_epsg() != 2056:
        gdf = gdf.to_crs(2056)
        print(f"   ↪ Reprojected {label} to EPSG:2056")
    return gdf

def clean_geom(gdf, label):
    before = len(gdf)
    gdf = gdf[~gdf.geometry.isna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"   🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def drop_cols_by_prefix(tz, prefixes):
    cols = []
    for p in prefixes:
        cols += [c for c in tz.columns if c.startswith(p)]
    cols = sorted(set(cols))
    if cols:
        tz = tz.drop(columns=cols)
        print(f"🧹 Dropped {len(cols)} existing columns: {cols}")
    return tz

def count_unique_features_per_zone(tz, g, label, id_col=None, predicate="intersects"):
    """Count UNIQUE features intersecting each traffic zone."""
    if len(g) == 0:
        print(f"   ⚠ {label}: no features -> all 0")
        return pd.Series(0, index=tz["zone_id"], dtype="int32")

    tz_min = tz[["zone_id", "geometry"]].copy()

    if id_col is not None and id_col in g.columns:
        g_min = g[[id_col, "geometry"]].copy()
        g_min["_fid"] = g_min[id_col].astype(str)
    else:
        g_min = g[["geometry"]].copy()
        g_min["_fid"] = g_min.index.astype("int64").astype(str)

    sj = gpd.sjoin(tz_min, g_min[["_fid", "geometry"]], how="inner", predicate=predicate)
    counts = sj.groupby("zone_id")["_fid"].nunique()
    return counts.reindex(tz["zone_id"], fill_value=0).astype("int32")

# ------------------------------
# 1) Load TZ
# ------------------------------
print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)

if "zone_id" not in tz.columns:
    raise ValueError("❌ TZ.gpkg must have 'zone_id'.")

tz["zone_id"] = tz["zone_id"].astype(int)
tz = ensure_2056(tz, "TZ")

# polygon fix (safe for polygons)
tz["geometry"] = tz.geometry.buffer(0)
tz = clean_geom(tz, "TZ")

# Optional: remove previous soft columns (only F3 soft-related)
tz = drop_cols_by_prefix(
    tz,
    prefixes=[
        "F3_outdoor_soft_",      # per-category + totals + diversity
        "F3_soft_",              # if you used this prefix somewhere
    ],
)

# ------------------------------
# 2) Load OSM outdoor_soft
# ------------------------------
print(f"\n📂 Reading OSM outdoor_soft: {OUTDOOR_SOFT_GPKG}")
out = gpd.read_file(OUTDOOR_SOFT_GPKG)
print(f"   Features (raw): {len(out)} | CRS: {out.crs}")

out = ensure_2056(out, "OSM outdoor_soft")
out = clean_geom(out, "OSM outdoor_soft")

# ID column
OSM_ID = "@id" if "@id" in out.columns else None

# Deduplicate by OSM id if available
if OSM_ID is not None:
    n_before = len(out)
    out = out.drop_duplicates(subset=OSM_ID)
    print(f"   Dedup {OSM_ID}: removed {n_before - len(out)} duplicates.")
else:
    print("   ⚠ No '@id' column -> using row index as unique feature id.")

# Normalize text columns (avoid literal "None" strings)
for c in ["leisure", "tourism"]:
    if c in out.columns:
        out[c] = out[c].replace("None", pd.NA)

# ------------------------------
# 3) Build subsets (apply exclusions)
# ------------------------------
# Leisure subset (target leisure values only)
leisure_all = out[out.get("leisure").isin(LEISURE_VALUES)].copy()

# Explicitly drop weird values if they exist
if "leisure" in leisure_all.columns:
    leisure_all = leisure_all[~leisure_all["leisure"].isin(EXCLUDE_LEISURE_VALUES)].copy()

leisure_all = clean_geom(leisure_all, "Leisure (filtered)")

# Tourism picnic_site subset
picnic_site = out[out.get("tourism") == TOURISM_VALUE].copy()
picnic_site = clean_geom(picnic_site, "Tourism picnic_site (raw)")

# ------------------------------
# 4) Exclude picnic_site that intersects ANY leisure feature
# ------------------------------
if len(picnic_site) > 0 and len(leisure_all) > 0:
    picnic_min = picnic_site[[OSM_ID, "geometry"]].copy() if (OSM_ID and OSM_ID in picnic_site.columns) else picnic_site[["geometry"]].copy()
    leisure_min = leisure_all[[OSM_ID, "geometry"]].copy() if (OSM_ID and OSM_ID in leisure_all.columns) else leisure_all[["geometry"]].copy()

    sj = gpd.sjoin(
        picnic_min,
        leisure_min[["geometry"]],
        how="left",
        predicate="intersects",
    )

    # For each picnic feature: does it intersect ANY leisure feature?
    has_match = sj.groupby(sj.index)["index_right"].apply(lambda s: s.notna().any())
    keep_idx = has_match[~has_match].index  # keep those with NO match

    picnic_site_filtered = picnic_site.loc[keep_idx].copy()
else:
    picnic_site_filtered = picnic_site.copy()

print(f"\n✅ Leisure features kept: {len(leisure_all)}")
print(f"✅ picnic_site kept AFTER overlap removal: {len(picnic_site_filtered)} (from {len(picnic_site)})")

# ------------------------------
# 5) Per-category counts + totals + diversity
# Naming requested:
# - per category: F3_outdoor_soft_<category>_count
# - total plotted: F3_outdoor_soft_count
# - diversity plotted: F3_outdoor_soft_diversity
# ------------------------------
cat_gdfs = {}

# Leisure categories -> one column each
for v in sorted(LEISURE_VALUES):
    # Column name like: F3_outdoor_soft_firepit_count
    col = f"F3_outdoor_soft_{v}_count"
    cat_gdfs[col] = leisure_all[leisure_all.get("leisure") == v].copy()

# Tourism picnic_site (filtered) -> one column
cat_gdfs["F3_outdoor_soft_picnic_site_count"] = picnic_site_filtered.copy()

soft_cols = []
print("\n📏 Counting features per zone (predicate=intersects)...")
for col, gdf_cat in cat_gdfs.items():
    gdf_cat = clean_geom(gdf_cat, col)
    tz[col] = count_unique_features_per_zone(tz, gdf_cat, col, id_col=OSM_ID).to_numpy()
    soft_cols.append(col)

# Total count (sum of per-category)
tz["F3_outdoor_soft_count"] = tz[soft_cols].sum(axis=1).astype("int32")

# Diversity (number of categories with count > 0)
tz["F3_outdoor_soft_diversity"] = (tz[soft_cols] > 0).sum(axis=1).astype("int32")

print("\nSanity check:")
print("   zones with soft_count==0:", int((tz["F3_outdoor_soft_count"] == 0).sum()))
print("   zones with soft_diversity==0:", int((tz["F3_outdoor_soft_diversity"] == 0).sum()))
print("   max soft_count:", int(tz["F3_outdoor_soft_count"].max()))
print("   max soft_diversity:", int(tz["F3_outdoor_soft_diversity"].max()))

# ------------------------------
# 6) Save TZ
# ------------------------------
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with F3_outdoor_soft_* to: {TZ_PATH}")

# ------------------------------
# 7) Plot maps (ONLY total + diversity)
# ------------------------------
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F3_outdoor_soft_count", ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title("F3_outdoor_soft_count (OSM soft outdoor features per zone)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F3_outdoor_soft_diversity", ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title("F3_outdoor_soft_diversity (number of soft outdoor categories per zone)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
base = Path(".")

TZ_PATH = base / "TZ" / "TZ.gpkg"
LEISURE_USE_PATH = base / "swissTLM3D_2025" / "Areal" / "LeisureUse.gpkg"
DIVERSE_USE_PATH = base / "swissTLM3D_2025" / "Areal" / "DiverseUse.gpkg"
OUTDOOR_SOFT_OSM_PATH = base / "OSM" / "Outdoor" / "outdoor_soft.gpkg"

# =========================================================
# UTILS
# =========================================================
def ensure_2056(gdf, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label}: CRS is missing.")
    if gdf.crs.to_epsg() != 2056:
        gdf = gdf.to_crs(2056)
        print(f"   ↪ Reprojected {label} to EPSG:2056")
    return gdf

def clean_geom(gdf, label):
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"   🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def pick_id_col(gdf, candidates):
    for c in candidates:
        if c in gdf.columns:
            return c
    return None

def count_unique_features_per_zone(tz, g, label, id_col=None, predicate="intersects"):
    """Return Series indexed by zone_id with 0 fill."""
    if len(g) == 0:
        print(f"   ⚠ {label}: no features -> all zeros.")
        return pd.Series(0, index=tz["zone_id"], dtype="int32")

    tz_min = tz[["zone_id", "geometry"]].copy()

    if id_col is not None and id_col in g.columns:
        g_min = g[[id_col, "geometry"]].copy()
        g_min["_fid"] = g_min[id_col].astype(str)
    else:
        g_min = g[["geometry"]].copy()
        g_min["_fid"] = g_min.index.astype("int64").astype(str)

    sj = gpd.sjoin(tz_min, g_min[["_fid", "geometry"]], how="inner", predicate=predicate)
    counts = sj.groupby("zone_id")["_fid"].nunique()
    return counts.reindex(tz["zone_id"], fill_value=0).astype("int32")

def make_zoneid_mask_ch(tz):
    """
    Boolean mask indexed by zone_id (no NaNs).
    Prefer N_KT if available; otherwise fall back to ISO-like columns.
    """
    zone_ids = pd.Index(tz["zone_id"].astype(int), name="zone_id")

    if "N_KT" in tz.columns:
        exclude = {"Liechtenstein", "Campione d'Italia", "Büsingen am Hochrhein", "Enk.", "LIE"}
        keep = ~tz["N_KT"].isin(exclude)
        return pd.Series(keep.to_numpy(dtype=bool), index=zone_ids)

    for col in ["ISO", "iso", "country", "CNTR", "cntr", "NUTS0", "nuts0"]:
        if col in tz.columns:
            s = tz[col].astype(str)
            keep = s.str.contains("CH", na=False)
            return pd.Series(keep.to_numpy(dtype=bool), index=zone_ids)

    return pd.Series(True, index=zone_ids)

def corr_report(a, b, label, mask=None):
    a, b = a.align(b, join="inner")
    if mask is None:
        m = (~a.isna()) & (~b.isna())
    else:
        mask = mask.reindex(a.index, fill_value=False)
        m = mask & (~a.isna()) & (~b.isna())

    a2, b2 = a[m], b[m]
    print(f"\n📈 Correlation — {label}")
    print(f"   Zones used: {len(a2)}")
    if len(a2) == 0:
        print("   No zones after masking -> nothing to correlate.")
        return
    if a2.std() == 0 or b2.std() == 0:
        print("   std==0 -> correlation not meaningful.")
        print(f"   Std A: {a2.std()} | Std B: {b2.std()}")
        return

    pearson = a2.corr(b2, method="pearson")
    spearman = a2.corr(b2, method="spearman")
    print(f"   Std A: {a2.std():.6f} | Std B: {b2.std():.6f}")
    print(f"   Pearson : {pearson:.3f}")
    print(f"   Spearman: {spearman:.3f}")

# =========================================================
# 1) LOAD TZ
# =========================================================
print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)

if "zone_id" not in tz.columns:
    raise ValueError("❌ TZ.gpkg must contain 'zone_id'.")

tz["zone_id"] = tz["zone_id"].astype(int)
tz = ensure_2056(tz, "TZ")
tz["geometry"] = tz.geometry.buffer(0)
tz = clean_geom(tz, "TZ")

mask_ch = make_zoneid_mask_ch(tz)

# Drop old OFF columns (recompute)
off_cols_to_drop = [
    "F3_outdoor_soft_off_count",
    "F3_outdoor_soft_leisureuse_off_count",
    "F3_outdoor_soft_public_parks_off_count",
]
drop_existing = [c for c in off_cols_to_drop if c in tz.columns]
if drop_existing:
    tz = tz.drop(columns=drop_existing)
    print(f"🧹 Dropped existing OFF columns (recomputing): {drop_existing}")

# =========================================================
# 2) OFFICIAL POLYGONS (swissTLM3D Areal)
# =========================================================
print(f"\n📂 Reading OFF LeisureUse: {LEISURE_USE_PATH}")
lei = gpd.read_file(LEISURE_USE_PATH)
lei = ensure_2056(lei, "OFF LeisureUse")
lei = clean_geom(lei, "OFF LeisureUse")
lei_id = pick_id_col(lei, ["uuid", "UUID", "id", "ID", "OBJECTID"])
print(f"   OFF LeisureUse features (valid): {len(lei)} | id_col: {lei_id}")

print(f"\n📂 Reading OFF DiverseUse: {DIVERSE_USE_PATH}")
div = gpd.read_file(DIVERSE_USE_PATH)
div = ensure_2056(div, "OFF DiverseUse")
div = clean_geom(div, "OFF DiverseUse")

if "objektart" not in div.columns:
    raise ValueError("❌ DiverseUse must contain column 'objektart'.")

parks_off = div[div["objektart"] == "Oeffentliches Parkareal"].copy()
parks_off = clean_geom(parks_off, "OFF Public Parks (Oeffentliches Parkareal)")
parks_off_id = pick_id_col(parks_off, ["uuid", "UUID", "id", "ID", "OBJECTID"])
print(f"   OFF Public Parks features (valid): {len(parks_off)} | id_col: {parks_off_id}")

# Counts per zone -> Series indexed by zone_id
off_leisureuse_count = count_unique_features_per_zone(tz, lei, "OFF LeisureUse count", id_col=lei_id)
off_public_parks_count = count_unique_features_per_zone(tz, parks_off, "OFF Public Parks count", id_col=parks_off_id)

# ===== CRITICAL FIX: assign by zone_id, not by tz.index =====
zone_id_index = pd.Index(tz["zone_id"], name="zone_id")

tz["F3_outdoor_soft_leisureuse_off_count"] = (
    off_leisureuse_count.reindex(zone_id_index).fillna(0).astype("int32").to_numpy()
)
tz["F3_outdoor_soft_public_parks_off_count"] = (
    off_public_parks_count.reindex(zone_id_index).fillna(0).astype("int32").to_numpy()
)

tz["F3_outdoor_soft_off_count"] = (
    tz["F3_outdoor_soft_leisureuse_off_count"] + tz["F3_outdoor_soft_public_parks_off_count"]
).fillna(0).astype("int32")

print("\nSanity check OFF soft:")
print("   F3_outdoor_soft_off_count == 0 zones:", int((tz["F3_outdoor_soft_off_count"] == 0).sum()))
print("   max F3_outdoor_soft_off_count:", int(tz["F3_outdoor_soft_off_count"].max()))

# =========================================================
# 3) OSM PARKS ONLY (QC + correlation)
# =========================================================
print(f"\n📂 Reading OSM outdoor_soft: {OUTDOOR_SOFT_OSM_PATH}")
osm = gpd.read_file(OUTDOOR_SOFT_OSM_PATH)
osm = ensure_2056(osm, "OSM outdoor_soft")

if "leisure" in osm.columns:
    osm["leisure"] = osm["leisure"].replace("None", pd.NA)

osm_parks = osm[osm.get("leisure") == "park"].copy()
osm_parks = clean_geom(osm_parks, "OSM parks (leisure=park)")

osm_id = "@id" if "@id" in osm_parks.columns else None
if osm_id is not None:
    n0 = len(osm_parks)
    osm_parks = osm_parks.drop_duplicates(subset=osm_id)
    print(f"   Dedup {osm_id}: removed {n0 - len(osm_parks)} duplicates.")

osm_parks_count = count_unique_features_per_zone(tz, osm_parks, "OSM parks count", id_col=osm_id).astype("float64")

# =========================================================
# 4) CORRELATION: OFF public parks vs OSM parks (CH only if mask available)
# =========================================================
corr_report(
    off_public_parks_count.astype("float64"),
    osm_parks_count,
    label="Public parks (OFF: 'Oeffentliches Parkareal') vs OSM parks (leisure=park)",
    mask=mask_ch,
)

# =========================================================
# 5) SAVE TZ
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with OFF columns to: {TZ_PATH}")

# =========================================================
# 6) PLOT
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F3_outdoor_soft_off_count", ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title("F3_outdoor_soft_off_count (swissTLM3D Areal: LeisureUse + Public Parks)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================================================
# PATHS
# =========================================================
base = Path(".")
TZ_PATH = base / "TZ" / "TZ.gpkg"
OUTDOOR_SOFT_OSM_PATH = base / "OSM" / "Outdoor" / "outdoor_soft.gpkg"

# =========================================================
# PARAMETERS
# =========================================================
# "Area-like" soft leisure categories (used for imputed area)
SOFT_LEISURE_AREA = {"park", "playground", "dog_park", "beach_resort", "marina"}

# For the polygon-only area (poly), we keep the same selection you used before:
# - leisure in SOFT_LEISURE_AREA
# - tourism = picnic_site
INCLUDE_PICNIC_SITE_IN_POLY = True          # as in your current script
EXCLUDE_PICNIC_SITE_FROM_IMP = True         # as requested ("NO picnic_site" for imputed)

# Output columns (ALL under F3 naming)
COL_AREA_POLY = "F3_outdoor_soft_area_poly"   # area from REAL polygons only
COL_AREA_IMP  = "F3_outdoor_soft_area_imp"    # area from real polygons + imputed buffers for points (no picnic_site)

# Deterministic imputation rule:
# For each leisure category, assign each point the category MEDIAN polygon area (m²),
# capped to P95 polygon area to avoid extreme buffers.
USE_MEDIAN = True
CAP_AT_P95 = True

# =========================================================
# UTILS
# =========================================================
def ensure_2056(gdf, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label} has no CRS.")
    if gdf.crs.to_epsg() != 2056:
        gdf = gdf.to_crs(2056)
        print(f"   ↪ Reprojected {label} to EPSG:2056")
    return gdf

def clean_geom(gdf, label):
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"   🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def normalize_none_strings(gdf, cols):
    for c in cols:
        if c in gdf.columns:
            gdf[c] = gdf[c].replace("None", pd.NA)
    return gdf

def safe_union(geoms, label):
    """Union a GeoSeries/iterable; return None if empty."""
    if geoms is None or len(geoms) == 0:
        return None
    try:
        return geoms.unary_union
    except Exception as e:
        print(f"⚠ Union failed for {label}: {type(e).__name__}: {e}")
        return None

# =========================================================
# 1) LOAD TZ (do NOT drop existing attributes)
# =========================================================
print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)

if "zone_id" not in tz.columns:
    raise ValueError("❌ TZ.gpkg must contain 'zone_id'.")

tz["zone_id"] = tz["zone_id"].astype(int)
tz = ensure_2056(tz, "TZ")
tz["geometry"] = tz.geometry.buffer(0)  # fix self-intersections
tz = clean_geom(tz, "TZ")

# =========================================================
# 2) LOAD OSM outdoor_soft
# =========================================================
print(f"\n📂 Reading OSM outdoor_soft: {OUTDOOR_SOFT_OSM_PATH}")
osm = gpd.read_file(OUTDOOR_SOFT_OSM_PATH)
osm = ensure_2056(osm, "OSM outdoor_soft")
osm = clean_geom(osm, "OSM outdoor_soft")
osm = normalize_none_strings(osm, ["leisure", "tourism"])

print(f"   OSM features (valid): {len(osm)}")

# =========================================================
# 3) BUILD SOFT SELECTIONS
# =========================================================
mask_poly = pd.Series(False, index=osm.index)

if "leisure" in osm.columns:
    mask_poly |= osm["leisure"].isin(SOFT_LEISURE_AREA)

if INCLUDE_PICNIC_SITE_IN_POLY and "tourism" in osm.columns:
    mask_poly |= (osm["tourism"] == "picnic_site")

soft_poly_sel = osm[mask_poly].copy()
soft_poly_sel = clean_geom(soft_poly_sel, "OSM soft selection (for poly area)")

print(f"\n✅ Selected features for POLY-area selection (all geometries): {len(soft_poly_sel)}")
print("📊 Geometry types (poly selection):")
print(soft_poly_sel.geometry.geom_type.value_counts(dropna=False))

# For imputed area: only leisure categories, exclude picnic_site entirely
mask_imp = pd.Series(False, index=osm.index)
if "leisure" in osm.columns:
    mask_imp |= osm["leisure"].isin(SOFT_LEISURE_AREA)

soft_imp_sel = osm[mask_imp].copy()
soft_imp_sel = clean_geom(soft_imp_sel, "OSM soft selection (for imputed area; leisure only)")

print(f"\n✅ Selected features for IMP-area selection (all geometries): {len(soft_imp_sel)}")
print("📊 Geometry types (imp selection):")
print(soft_imp_sel.geometry.geom_type.value_counts(dropna=False))

# =========================================================
# 4) POLYGON-ONLY AREA (REAL POLYGONS ONLY)
#    F3_outdoor_soft_area_poly = Area(TZ ∩ union(real polygons))
# =========================================================
poly_real = soft_poly_sel[soft_poly_sel.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
poly_real = clean_geom(poly_real, "REAL soft polygons (poly area)")
poly_real["geometry"] = poly_real.geometry.buffer(0)

print(f"\n🧩 Real polygons used for {COL_AREA_POLY}: {len(poly_real)}")

if len(poly_real) == 0:
    tz[COL_AREA_POLY] = 0.0
    print(f"⚠ No polygons -> {COL_AREA_POLY}=0 everywhere.")
else:
    union_poly = safe_union(poly_real.geometry, label=f"{COL_AREA_POLY} union")
    if union_poly is None:
        tz[COL_AREA_POLY] = 0.0
        print(f"⚠ Union failed -> {COL_AREA_POLY}=0 everywhere.")
    else:
        print(f"📐 Computing {COL_AREA_POLY} as Area(TZ ∩ union(polygons)) ...")
        tz[COL_AREA_POLY] = tz.geometry.intersection(union_poly).area.astype("float64")

# =========================================================
# 5) IMPUTED AREA (REAL POLYGONS + IMPUTED BUFFERS FOR POINTS; NO picnic_site)
#    F3_outdoor_soft_area_imp = Area(TZ ∩ union(real polygons + imputed buffers))
# =========================================================
# Split leisure-only selection into polygons and points (ignore lines)
imp_polys = soft_imp_sel[soft_imp_sel.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
imp_polys = clean_geom(imp_polys, "IMP real polygons")
imp_polys["geometry"] = imp_polys.geometry.buffer(0)

imp_points = soft_imp_sel[soft_imp_sel.geometry.geom_type == "Point"].copy()
imp_points = clean_geom(imp_points, "IMP points")

print(f"\n🧩 Real polygons for {COL_AREA_IMP}: {len(imp_polys)}")
print(f"📍 Points to impute for {COL_AREA_IMP}: {len(imp_points)}")

# Compute per-category polygon area stats (m²)
cat_stats = {}
if len(imp_polys) > 0 and "leisure" in imp_polys.columns:
    imp_polys["_area"] = imp_polys.geometry.area.astype("float64")
    for cat in sorted(SOFT_LEISURE_AREA):
        a = imp_polys.loc[imp_polys["leisure"] == cat, "_area"].dropna()
        if len(a) == 0:
            continue
        med = float(a.median())
        p95 = float(a.quantile(0.95))
        cat_stats[cat] = {"median": med, "p95": p95, "n": len(a)}

print("\n📌 Imputation area stats (from polygons):")
for cat in sorted(SOFT_LEISURE_AREA):
    if cat in cat_stats:
        s = cat_stats[cat]
        print(f"   {cat}: n_polys={s['n']}, median={s['median']:.1f} m², p95={s['p95']:.1f} m²")
    else:
        print(f"   {cat}: no polygon areas available -> points will NOT be imputed for this category.")

# Build imputed buffer polygons for points
buffers = []
if len(imp_points) > 0 and "leisure" in imp_points.columns:
    for cat in sorted(SOFT_LEISURE_AREA):
        pts_cat = imp_points[imp_points["leisure"] == cat].copy()
        if len(pts_cat) == 0:
            continue
        if cat not in cat_stats:
            # no empirical polygon areas -> skip imputation (conservative)
            continue

        area_val = cat_stats[cat]["median"] if USE_MEDIAN else cat_stats[cat]["median"]
        if CAP_AT_P95:
            area_val = min(area_val, cat_stats[cat]["p95"])

        # Convert area to equivalent-circle radius
        r = np.sqrt(area_val / np.pi)

        # Buffer points to polygon proxies
        pts_cat["geometry"] = pts_cat.geometry.buffer(r)
        buffers.append(pts_cat[["geometry"]])

if len(buffers) > 0:
    imp_buffers = pd.concat(buffers, ignore_index=True)
    imp_buffers = gpd.GeoDataFrame(imp_buffers, geometry="geometry", crs=tz.crs)
    imp_buffers = clean_geom(imp_buffers, "Imputed buffers")
else:
    imp_buffers = gpd.GeoDataFrame({"geometry": []}, geometry="geometry", crs=tz.crs)

print(f"🧱 Imputed buffer polygons created: {len(imp_buffers)}")

# Union real polygons + imputed buffers (overlap counted once)
all_imp_geoms = pd.concat(
    [
        imp_polys[["geometry"]].copy(),
        imp_buffers[["geometry"]].copy(),
    ],
    ignore_index=True,
)
all_imp_geoms = gpd.GeoDataFrame(all_imp_geoms, geometry="geometry", crs=tz.crs)
all_imp_geoms = clean_geom(all_imp_geoms, "All geometries for IMP union")
all_imp_geoms["geometry"] = all_imp_geoms.geometry.buffer(0)

print(f"🧩 Total polygons entering {COL_AREA_IMP} union: {len(all_imp_geoms)}")

if len(all_imp_geoms) == 0:
    tz[COL_AREA_IMP] = 0.0
    print(f"⚠ No geometries -> {COL_AREA_IMP}=0 everywhere.")
else:
    union_imp = safe_union(all_imp_geoms.geometry, label=f"{COL_AREA_IMP} union")
    if union_imp is None:
        tz[COL_AREA_IMP] = 0.0
        print(f"⚠ Union failed -> {COL_AREA_IMP}=0 everywhere.")
    else:
        print(f"📐 Computing {COL_AREA_IMP} as Area(TZ ∩ union(real polygons + imputed buffers)) ...")
        tz[COL_AREA_IMP] = tz.geometry.intersection(union_imp).area.astype("float64")

# =========================================================
# 6) SANITY CHECKS
# =========================================================
print(f"\nSanity check {COL_AREA_POLY} (m²):")
print("   zones area==0:", int((tz[COL_AREA_POLY] == 0).sum()))
print("   max area (m²):", float(tz[COL_AREA_POLY].max()))
print("   mean area (m²):", float(tz[COL_AREA_POLY].mean()))

print(f"\nSanity check {COL_AREA_IMP} (m²):")
print("   zones area==0:", int((tz[COL_AREA_IMP] == 0).sum()))
print("   max area (m²):", float(tz[COL_AREA_IMP].max()))
print("   mean area (m²):", float(tz[COL_AREA_IMP].mean()))

# =========================================================
# 7) SAVE (do NOT remove any existing attributes; only add/overwrite these two columns)
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with '{COL_AREA_POLY}' and '{COL_AREA_IMP}' to: {TZ_PATH}")

# =========================================================
# 8) PLOTS
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column=COL_AREA_POLY, ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title(f"{COL_AREA_POLY} (OSM polygons only, m²) — union(polygons) within each traffic zone")
ax.set_axis_off()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column=COL_AREA_IMP, ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title(f"{COL_AREA_IMP} (OSM polygons + imputed point buffers, m²) — NO picnic_site")
ax.set_axis_off()
plt.tight_layout()
plt.show()


## Land use mix (Arealstatistik 2025 → Traffic Zones)

## Objective
Compute a **land use mix** indicator for each **Traffic Zone (TZ)** using the Arealstatistik variable **`LU25_10`** (a **10-class** land-use classification).  
Store the final indicator in TZ as **`F3_outdoor_LUmix`** and map it (choropleth).

---

## Input
- **Traffic Zones**: `TZ/TZ.gpkg`  
  - required fields:
    - `zone_id` (unique TZ identifier)
    - `N_KT` (canton/area code; includes special cases **`LIE`** and **`Enk`**)

- **Arealstatistik 2025**: `Arealstatistik_2025/Arealstatistik_2025.gpkg`  
  - required field:
    - `LU25_10` (10-class land use)

All layers must be harmonized to **EPSG:2056**.

---

## Logical steps

### 1) Load data and clean geometries
- Read `TZ.gpkg` and `Arealstatistik_2025.gpkg`
- Ensure both layers are in **EPSG:2056**
- Drop `NULL` / `empty` geometries
- (Optional but recommended) apply `tz.geometry.buffer(0)` to fix minor self-intersections

---

### 2) Assign Arealstatistik cells to TZs (spatial join)
**Goal:** every Arealstatistik cell must be assigned to a TZ.

- Build a “Swiss-only” TZ layer by **excluding** zones with `N_KT ∈ {LIE, Enk}`
  - rationale: you do **not** want to aggregate cells directly into these special zones

- Run a **spatial join** between Arealstatistik and `TZ_swiss` using `predicate="intersects"`

- For cells still missing a `zone_id` (typically border effects / small mismatches):
  - use `sjoin_nearest` to `TZ_swiss`
  - assign each cell to the **nearest** Swiss TZ

**Important note:** if a cell falls inside `LIE` or `Enk`, it is **not** assigned to them—it is assigned to the **nearest Swiss TZ** instead.

---

### 3) Build the LU distribution per TZ
For each `zone_id`:
- count how many cells fall into each `LU25_10` class
- obtain a table like:
  - rows: `zone_id`
  - columns: `LU25_10` classes
  - values: cell counts (or hectares if you weight by cell area)

---

### 4) Water dominance correction (cap at 40%)
**Problem:** some TZs (especially lake zones) can be artificially dominated by water, producing a land use mix that is more an artefact than a meaningful “functional mix”.

**Solution implemented:** before computing entropy, **cap the total share of class `400` (lakes/rivers)** at **40%** per TZ.

- compute the initial class shares `p_i`
- if `p_400 > 0.40`:
  - set `p_400 = 0.40`
  - redistribute the removed probability mass proportionally across all **non-water** classes (renormalization)

This keeps the indicator comparable while preventing “lake-only” zones from dominating the metric.

---

### 5) Compute land use mix
Use **normalized Shannon entropy** over the 10 classes **after applying the water cap**:

- for each zone:
  - `p_i = count_i / sum(counts)` *(then apply the water cap + renormalize)*
  - `H = - Σ p_i * ln(p_i)`
  - `H_norm = H / ln(K)` with `K = number of classes (ideally 10)`

Result:
- `F3_outdoor_LUmix ∈ [0, 1]`
  - ~0: one dominant land-use class (low diversity)
  - ~1: land use evenly distributed across classes (high diversity)

---

### 6) Special handling for `Enk` (10 km neighborhood)
For TZs with `N_KT = Enk`:
- compute the **mean** `F3_outdoor_LUmix` of Swiss TZs **within a 10 km radius**  
  *(implemented by buffering each Enk TZ by 10 km and selecting intersecting Swiss TZs)*
- if an Enk zone has no Swiss TZs within 10 km:
  - fallback: assign the value of the **nearest** Swiss TZ

---

### 7) Special handling for `LIE`
For TZs with `N_KT = LIE`:
- build a reference distribution by sampling values from TZs with:
  - `N_KT ∈ {SG, AI, AR}`
- assign to each Liechtenstein TZ a value drawn (sampling with replacement) from that reference distribution

> In practice, LIE values become plausible because they are borrowed from geographically comparable Swiss zones.

---

### 8) Save and map
- save only the new column:
  - `F3_outdoor_LUmix` into `TZ/TZ.gpkg`
- plot it in QGIS or via code (choropleth)

---

## Output
- Updated `TZ/TZ.gpkg` with:
  - **`F3_outdoor_LUmix`** (float, 0–1)
- A land use mix map (choropleth)

---

## Interpretation note
This indicator captures **functional mix** (how “mixed” a zone is in terms of land uses).  
With the **water-share cap (40%)** and the **Enk 10 km neighborhood assignment**, it is less sensitive to lake artefacts and more consistent across edge/special zones. It aligns with common approaches in the literature on:
- **urban diversity / land use mix**
- **territorial attractiveness** and **proximity to opportunities** (services, leisure, amenities)
- a diversity-based proxy for the **variety of potential destinations** available within the zone


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

base = Path(".")

TZ_PATH = base / "TZ" / "TZ.gpkg"
AS_PATH = base / "Arealstatistik_2025" / "arealstatistik_2056.gpkg"

# Arealstatistik column names (adjust here if needed)
LU_COL = "LU_10"          # your 10-class code column
KT_COL = "N_KT"           # special zones: "LIE" and "Enk"

# Output column (F3 naming, as requested)
OUT_COL = "F3_outdoor_LUmix"

# Caps / params
WATER_CODE = 400          # NOLU04_10: lakes & rivers (water)
WATER_CAP = 0.40          # cap water share before entropy
ENK_RADIUS_M = 20_000     # 20 km neighborhood for Enk

# =========================================================
# Helpers
# =========================================================
def ensure_2056(gdf, label):
    if gdf.crs is None:
        raise ValueError(f"❌ {label} has no CRS.")
    if gdf.crs.to_epsg() != 2056:
        gdf = gdf.to_crs(2056)
        print(f"   ↪ Reprojected {label} to EPSG:2056")
    return gdf

def clean_geom(gdf, label):
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    after = len(gdf)
    if after != before:
        print(f"   🧽 {label}: removed {before-after} null/empty geometries.")
    return gdf

def pick_first_existing(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"❌ {label}: none of these columns exist: {candidates}")

def detect_water_cols(columns, water_code=400):
    """
    Find LU columns that correspond to water (code 400).
    Works for:
      - int 400
      - string '400'
      - string '400 Lakes and rivers' / '400 ...'
    """
    water = []
    for c in columns:
        s = str(c).strip()
        if s == str(water_code) or s.startswith(f"{water_code} "):
            water.append(c)
    return water

def normalized_shannon_entropy(p, K):
    """p: (n_zones, K) probabilities; returns H_norm in [0,1]."""
    with np.errstate(divide="ignore", invalid="ignore"):
        H = -(p * np.where(p > 0, np.log(p), 0)).sum(axis=1)
    return np.where(K > 1, H / np.log(K), 0.0)

# =========================================================
# 1) Load
# =========================================================
print(f"📂 Reading TZ: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)
tz = ensure_2056(tz, "TZ")
tz = clean_geom(tz, "TZ")
tz["geometry"] = tz.geometry.buffer(0)

ZONE_COL = pick_first_existing(
    tz,
    candidates=["zone_id", "ZONE_ID", "id", "ID", "OBJECTID"],
    label="TZ",
)
print(f"✅ Using TZ id column: {ZONE_COL}")

tz[ZONE_COL] = pd.to_numeric(tz[ZONE_COL], errors="coerce")
if tz[ZONE_COL].isna().any():
    raise ValueError(f"❌ TZ {ZONE_COL} has NaNs after numeric conversion.")
tz[ZONE_COL] = tz[ZONE_COL].astype(int)

if KT_COL not in tz.columns:
    raise ValueError(f"❌ TZ must contain '{KT_COL}'.")

print(f"\n📂 Reading Arealstatistik: {AS_PATH}")
as_gdf = gpd.read_file(AS_PATH)
as_gdf = ensure_2056(as_gdf, "Arealstatistik")
as_gdf = clean_geom(as_gdf, "Arealstatistik")

if LU_COL not in as_gdf.columns:
    raise ValueError(f"❌ Arealstatistik must contain '{LU_COL}'.")

# =========================================================
# 2) Swiss-only TZ (exclude LIE & Enk)
# =========================================================
tz_swiss = tz[~tz[KT_COL].isin(["LIE", "Enk"])][[ZONE_COL, "geometry"]].copy()
tz_swiss = clean_geom(tz_swiss, "TZ_swiss")
print(f"\n🇨🇭 TZ_swiss: {len(tz_swiss)} zones (excluded LIE + Enk)")

# =========================================================
# 3) Assign Arealstatistik cells to Swiss TZ (intersects + nearest fill)
# =========================================================
as_min = as_gdf[[LU_COL, "geometry"]].copy()

sj = gpd.sjoin(
    as_min,
    tz_swiss[[ZONE_COL, "geometry"]],
    how="left",
    predicate="intersects",
    lsuffix="as",
    rsuffix="tz",
)

zone_col_joined = ZONE_COL
if zone_col_joined not in sj.columns:
    if f"{ZONE_COL}_tz" in sj.columns:
        zone_col_joined = f"{ZONE_COL}_tz"
    else:
        raise ValueError(f"❌ After sjoin, missing zone id column. Columns: {list(sj.columns)}")

missing = int(sj[zone_col_joined].isna().sum())
print(f"🔎 Intersects join done. Missing zone assignments: {missing}")

if missing > 0:
    miss = sj[sj[zone_col_joined].isna()].copy()

    near = gpd.sjoin_nearest(
        miss[[LU_COL, "geometry"]],
        tz_swiss[[ZONE_COL, "geometry"]],
        how="left",
        distance_col="_dist",
    )

    zc_near = ZONE_COL
    if zc_near not in near.columns and f"{ZONE_COL}_right" in near.columns:
        zc_near = f"{ZONE_COL}_right"
    if zc_near not in near.columns:
        raise KeyError(f"❌ nearest join: can't find zone id. Columns: {list(near.columns)}")

    sj.loc[near.index, zone_col_joined] = near[zc_near].values

    missing2 = int(sj[zone_col_joined].isna().sum())
    print(f"✅ Filled missing with nearest. Remaining missing: {missing2}")
    if missing2 > 0:
        raise ValueError("❌ Still missing zone assignment after nearest fill (unexpected).")

sj[zone_col_joined] = sj[zone_col_joined].astype(int)

# =========================================================
# 4) Build LU counts per zone (matrix)
# =========================================================
lu_counts = (
    sj.groupby([zone_col_joined, LU_COL])
      .size()
      .unstack(fill_value=0)
)

# Ensure all Swiss zones exist (0-fill if a zone got no cells for some reason)
lu_counts = lu_counts.reindex(tz_swiss[ZONE_COL].values, fill_value=0)

# =========================================================
# 5) Compute normalized Shannon entropy with water cap
# =========================================================
counts = lu_counts.values.astype(float)
row_sums = counts.sum(axis=1, keepdims=True)

p = np.divide(counts, row_sums, out=np.zeros_like(counts), where=row_sums != 0)

water_cols = detect_water_cols(lu_counts.columns, water_code=WATER_CODE)
if len(water_cols) > 0:
    water_idx = [lu_counts.columns.get_loc(c) for c in water_cols]

    p_water = p[:, water_idx].sum(axis=1)
    p_water_new = np.minimum(p_water, WATER_CAP)
    reduced = p_water > WATER_CAP

    if reduced.any():
        # distribute capped water share proportionally among water sub-columns (if multiple)
        w = p[:, water_idx]
        w_sum = w.sum(axis=1, keepdims=True)
        w_norm = np.divide(w, w_sum, out=np.zeros_like(w), where=w_sum != 0)
        p[:, water_idx] = w_norm * p_water_new.reshape(-1, 1)

        # renormalize the remaining classes
        other_idx = [i for i in range(p.shape[1]) if i not in water_idx]
        p_other = p[:, other_idx]
        p_other_sum = p_other.sum(axis=1)

        target_other = 1.0 - p_water_new
        scale = np.divide(target_other, p_other_sum, out=np.zeros_like(target_other), where=p_other_sum != 0)
        p[:, other_idx] = p_other * scale.reshape(-1, 1)

        print(f"🌊 Water cap applied: {int(reduced.sum())} TZ had water share > {WATER_CAP:.0%}")
else:
    print("ℹ️ No water (LU=400) column found -> skipping water cap.")

K = lu_counts.shape[1]
H_norm = normalized_shannon_entropy(p, K)
lumix_swiss = pd.Series(H_norm, index=lu_counts.index, name=OUT_COL).astype("float32")

# =========================================================
# 6) Attach to full TZ (computed only for Swiss TZ)
# =========================================================
tz[OUT_COL] = np.nan
tz.loc[tz[ZONE_COL].isin(lumix_swiss.index), OUT_COL] = tz.loc[
    tz[ZONE_COL].isin(lumix_swiss.index), ZONE_COL
].map(lumix_swiss)

print(f"\n✅ {OUT_COL} computed for Swiss TZ (excluding LIE/Enk).")

# =========================================================
# 7) Enk: mean of Swiss TZs within radius buffer; fallback nearest
# =========================================================
enk_mask = tz[KT_COL].eq("Enk")
if enk_mask.any():
    enk = tz[enk_mask][[ZONE_COL, "geometry"]].copy()
    enk = clean_geom(enk, "Enk")

    enk_buf = enk.copy()
    enk_buf["geometry"] = enk_buf.geometry.buffer(ENK_RADIUS_M)

    neigh = gpd.sjoin(
        enk_buf[[ZONE_COL, "geometry"]],
        tz_swiss[[ZONE_COL, "geometry"]],
        how="left",
        predicate="intersects",
        lsuffix="enk",
        rsuffix="tz",
    )

    left_col  = f"{ZONE_COL}_enk" if f"{ZONE_COL}_enk" in neigh.columns else (f"{ZONE_COL}_left" if f"{ZONE_COL}_left" in neigh.columns else ZONE_COL)
    right_col = f"{ZONE_COL}_tz"  if f"{ZONE_COL}_tz"  in neigh.columns else (f"{ZONE_COL}_right" if f"{ZONE_COL}_right" in neigh.columns else ZONE_COL)

    neigh["_lumix"] = neigh[right_col].map(lumix_swiss)
    enk_mean = neigh.groupby(left_col)["_lumix"].mean()

    tz.loc[enk_mask, OUT_COL] = tz.loc[enk_mask, ZONE_COL].map(enk_mean)

    # fallback nearest Swiss TZ
    still = tz[enk_mask & tz[OUT_COL].isna()][[ZONE_COL, "geometry"]].copy()
    if len(still) > 0:
        near_enk = gpd.sjoin_nearest(
            still,
            tz_swiss[[ZONE_COL, "geometry"]],
            how="left",
            distance_col="_dist",
        )
        zc_right = f"{ZONE_COL}_right" if f"{ZONE_COL}_right" in near_enk.columns else ZONE_COL
        nearest_vals = near_enk.set_index(ZONE_COL)[zc_right].map(lumix_swiss)

        tz.loc[tz[ZONE_COL].isin(nearest_vals.index), OUT_COL] = tz.loc[
            tz[ZONE_COL].isin(nearest_vals.index), OUT_COL
        ].fillna(tz.loc[tz[ZONE_COL].isin(nearest_vals.index), ZONE_COL].map(nearest_vals))

    print(f"✅ Enk handled (buffer {ENK_RADIUS_M/1000:.0f} km): {int(enk_mask.sum())} zones filled.")

# =========================================================
# 8) LIE: sample from SG, AI, AR distribution
# =========================================================
lie_mask = tz[KT_COL].eq("LIE")
if lie_mask.any():
    ref = tz[tz[KT_COL].isin(["SG", "AI", "AR"]) & tz[OUT_COL].notna()][OUT_COL].values
    if len(ref) == 0:
        raise ValueError("❌ No reference values found in SG/AI/AR to sample for LIE.")
    rng = np.random.default_rng(42)
    tz.loc[lie_mask, OUT_COL] = rng.choice(ref, size=int(lie_mask.sum()), replace=True)
    print(f"✅ LIE handled: {int(lie_mask.sum())} zones sampled from SG/AI/AR.")

# =========================================================
# 9) Save + Plot
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with '{OUT_COL}' to: {TZ_PATH}")

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column=OUT_COL, ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title(f"{OUT_COL} ({LU_COL}, normalized Shannon entropy) — water capped at {WATER_CAP:.0%}")
ax.set_axis_off()
plt.tight_layout()
plt.show()


## F6 — Others

## Objective
Build an “**Others**” factor from OpenStreetMap (OSM) facilities that are **not** sport / outdoor / cultural / gastronomy, by:
1) downloading + storing them as a single OSM dataset,  
2) mapping each OSM feature into **5 macro-categories**,  
3) aggregating counts to **Traffic Zones (TZ)**,  
4) computing a **diversity score** (0–5) per TZ,  
5) saving the final indicators into `TZ/TZ.gpkg` and plotting maps.

---

## Input
- **Traffic Zones**: `TZ/TZ.gpkg`  
  - required fields:
    - `zone_id` (or equivalent ID column)
    - `geometry` (polygons)
- **OSM Others**: `OSM/Others/Others.gpkg`  
  - geometry can be Point/Line/Polygon (mixed)
  - attributes used for classification (depending on feature):
    - `amenity`, `leisure`, `tourism`, `shop`

All layers must share the same CRS (we reprojected OSM to TZ CRS if needed).

---

## Step-by-step workflow

### 1) Define the 5 macro-categories (A–E)
We keep the agreed tags and group them into **five** categories:

**A) Nightlife**
- `amenity ∈ {brothel, casino, love_hotel, nightclub, stripclub, swingerclub}`

**B) Social**
- `amenity ∈ {community_centre, social_centre}`

**C) Spiritual**
- `amenity ∈ {grave_yard, monastery, place_of_worship}`

**D) Fun**
- `leisure ∈ {adult_gaming_centre, amusement_arcade, escape_game, water_park, hackrrspace}`
- OR `tourism = theme_park`
- OR `amenity = gaming`

**E) Wellness**
- `amenity = kneipp_water_cure`
- OR `shop ∈ {massage}`

> Each OSM feature receives **one single label** `cat5` among:  
> `nightlife | social | spiritual | fun | wellness`  
> based on a stable priority order.

---

### 2) Convert ALL OSM geometries to points (centroids)
Before aggregating, we force a **point representation** for every feature:
- Points remain points
- Lines/Polygons → converted to **centroid**
- MultiPoint → centroid

This avoids multi-zone counting caused by large polygons crossing multiple TZs and makes the aggregation behave like a POI-count.

---

### 3) Assign OSM points to TZs (spatial join)
We perform a spatial join:
- **TZ polygons** × **Others centroids**
- predicate: `intersects` (equivalent to “point in polygon” for centroids)

Each centroid is counted in exactly **one** TZ.

---

### 4) Compute counts per TZ (5 output variables)
For each `zone_id`, we compute the number of features in each category and store them as:

- `F6_others_nightlife`
- `F6_others_social`
- `F6_others_spiritual`
- `F6_others_fun`
- `F6_others_wellness`

Counts are integers (0 if no facility of that category is found in the zone).

---

### 5) Compute diversity (0–5)
We define a simple diversity index:

- `F6_others_diversity = (# of categories with count ≥ 1)`

So:
- 0 = no “Others” facilities at all
- 5 = at least one facility in **all** five categories

---

### 6) Manual override for a specific TZ (zone_id = 1)
A quality check in a specific AOI showed a mismatch, so we applied a **manual correction** before saving:

For `zone_id = 1`:
- nightlife = 0  
- social = 0  
- spiritual = 1  
- fun = 0  
- wellness = 0  
- diversity = 1 (recomputed automatically)

This override is applied **after** the automated counts, and **before** saving/plotting.

---

### 7) Save + plot
- Save the updated TZ layer back to: `TZ/TZ.gpkg`
- Plot choropleths for:
  - each of the 5 category counts
  - `F6_others_diversity`

---

## Output
`TZ/TZ.gpkg` updated with:

- `F6_others_nightlife` (int)
- `F6_others_social` (int)
- `F6_others_spiritual` (int)
- `F6_others_fun` (int)
- `F6_others_wellness` (int)
- `F6_others_diversity` (int, 0–5)

---

## Interpretation (how to read F6)
- High values in **one** category → specialization (e.g., nightlife-heavy zones)
- High `F6_others_diversity` → broader “miscellaneous” offer (social + spiritual + fun + wellness + nightlife)
- Intended as an “Other facilities” proxy complementing the main factors (outdoor, sport, culture, gastronomy).


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================================================
# PATHS (read the already-prepared GeoPackage)
# =========================================================
base = Path(".")
IN_GPKG = base / "OSM" / "Others" / "Others.gpkg"

print(f"📂 Reading OSM Others from: {IN_GPKG}")

# =========================================================
# 1) Robust read
# =========================================================
try:
    gdf = gpd.read_file(IN_GPKG, on_invalid="ignore")
    print("   ✅ Read with pyogrio (on_invalid='ignore').")
except Exception as e:
    print(f"   ⚠️ pyogrio failed ({type(e).__name__}: {e}). Trying engine='fiona'...")
    gdf = gpd.read_file(IN_GPKG, engine="fiona")
    print("   ✅ Read with engine='fiona'.")

print(f"   Features read (raw): {len(gdf)}")
print(f"   CRS (as-is): {gdf.crs}")
print(f"   Initial columns: {list(gdf.columns)}")

# =========================================================
# 2) Geometry cleaning (no CRS conversion, no file conversion)
# =========================================================
before = len(gdf)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
print(f"   🔍 Removed {before - len(gdf)} rows with NaN/empty geometry.")

before = len(gdf)
gdf = gdf[gdf.geometry.is_valid].copy()
print(f"   🔍 Removed {before - len(gdf)} invalid geometries.")

print(f"   ✅ Features after geometry cleaning: {len(gdf)}")

# =========================================================
# 3) Deduplicate by OSM feature id (@id) if present
# =========================================================
if "@id" in gdf.columns:
    before = len(gdf)
    gdf = gdf.drop_duplicates(subset=["@id"]).copy()
    print(f"🧹 Dedup on '@id': removed {before - len(gdf)} duplicates.")
else:
    print("⚠️ Column '@id' not found -> dedup skipped.")

# =========================================================
# 4) Convert ALL geometries to POINTS (centroids)
#   - Points stay points
#   - Lines/Polygons -> centroid
# =========================================================
geom_types_before = gdf.geometry.geom_type.value_counts(dropna=False)
print("\n📊 Geometry types BEFORE pointification:")
print(geom_types_before)

is_point = gdf.geometry.geom_type == "Point"
gdf.loc[~is_point, "geometry"] = gdf.loc[~is_point, "geometry"].centroid

# Safety: if anything still isn't Point, use representative_point()
bad = gdf.geometry.geom_type != "Point"
if bad.any():
    gdf.loc[bad, "geometry"] = gdf.loc[bad, "geometry"].representative_point()

geom_types_after = gdf.geometry.geom_type.value_counts(dropna=False)
print("\n📊 Geometry types AFTER pointification:")
print(geom_types_after)

# =========================================================
# 5) Attribute distributions (amenity/leisure/tourism/shop)
# =========================================================
for col in ["amenity", "leisure", "tourism", "shop"]:
    if col in gdf.columns:
        vc = gdf[col].replace("None", pd.NA).value_counts(dropna=False)
        print(f"\n📊 Distribution '{col}' (value_counts):\n{vc}")
    else:
        print(f"\nℹ️ Column '{col}' not present.")

# =========================================================
# 6) Category A–E + distribution
# Priority: A > B > C > E > D
# =========================================================
A_nightlife = {"brothel", "casino", "gaming", "love_hotel", "nightclub", "stripclub", "swingerclub"}
B_social    = {"community_centre", "social_centre"}
C_spiritual = {"grave_yard", "monastery", "place_of_worship"}

E_well_amen = {"kneipp_water_cure"}
E_well_shop = {"massage"}  # <-- MASSAGE ONLY

D_fun_leis  = {"adult_gaming_centre", "amusement_arcade", "escape_game", "water_park", "hackerspace", "hackrrspace"}
D_fun_tour  = {"theme_park"}

def classify(row):
    a = row.get("amenity", None)
    l = row.get("leisure", None)
    t = row.get("tourism", None)
    s = row.get("shop", None)

    if a in A_nightlife:
        return "A) Nightlife"
    if a in B_social:
        return "B) Social"
    if a in C_spiritual:
        return "C) Spiritual"
    if (a in E_well_amen) or (s in E_well_shop):
        return "E) Wellness"
    if (l in D_fun_leis) or (t in D_fun_tour):
        return "D) Fun"
    return "Unclassified"

gdf["category"] = gdf.apply(classify, axis=1)

cat_counts = gdf["category"].value_counts()
print("\n📊 Counts by category:\n", cat_counts)

# =========================================================
# 7) Plots (category counts + quick point map)
# =========================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
cat_counts.sort_index().plot(kind="bar", ax=ax)
ax.set_title("OSM Others — counts by category (A–E) [massage-only wellness]")
ax.set_xlabel("Category")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
gdf.plot(ax=ax, markersize=1)
ax.set_title("OSM Others — quick point plot (CRS as-is)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

base = Path(".")

TZ_PATH     = base / "TZ" / "TZ.gpkg"
OTHERS_GPKG = base / "OSM" / "Others" / "Others.gpkg"

# =========================================================
# 1) Load (NO geometry fixes / NO cleaning)
# =========================================================
tz = gpd.read_file(TZ_PATH)
others = gpd.read_file(OTHERS_GPKG)

# Pick TZ id column
ZONE_COL = None
for c in ["zone_id", "ZONE_ID", "id", "ID", "OBJECTID"]:
    if c in tz.columns:
        ZONE_COL = c
        break
if ZONE_COL is None:
    raise ValueError(f"TZ: cannot find zone id column. Columns: {list(tz.columns)}")

# Make zone id int-like
tz[ZONE_COL] = pd.to_numeric(tz[ZONE_COL], errors="coerce").astype("Int64")
if tz[ZONE_COL].isna().any():
    raise ValueError(f"TZ {ZONE_COL} has NaNs after numeric conversion.")
tz[ZONE_COL] = tz[ZONE_COL].astype(int)

# Align CRS (quietly)
if tz.crs is None or others.crs is None:
    raise ValueError("Missing CRS on TZ or Others.gpkg.")
if tz.crs != others.crs:
    others = others.to_crs(tz.crs)

# =========================================================
# 2) Categories EXACTLY as agreed (A..E) — BUT wellness = massage-only
# =========================================================
nightlife_amenity = {"brothel","casino","love_hotel","nightclub","stripclub","swingerclub"}
social_amenity    = {"community_centre","social_centre"}
spiritual_amenity = {"grave_yard","monastery","place_of_worship"}
wellness_amenity  = {"kneipp_water_cure"}

fun_leisure       = {"adult_gaming_centre","amusement_arcade","escape_game","water_park","hackrrspace"}
fun_tourism       = {"theme_park"}

wellness_shop     = {"massage"}  # <-- MASSAGE ONLY

def _norm(x):
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s in ("None","nan","<NA>","null","") else s

def classify(row):
    a = _norm(row.get("amenity")) if "amenity" in row else ""
    l = _norm(row.get("leisure")) if "leisure" in row else ""
    t = _norm(row.get("tourism")) if "tourism" in row else ""
    s = _norm(row.get("shop"))    if "shop"    in row else ""

    # one label per feature (stable order)
    if a in spiritual_amenity:
        return "spiritual"
    if a in wellness_amenity or s in wellness_shop:
        return "wellness"
    if a in nightlife_amenity:
        return "nightlife"
    if a in social_amenity:
        return "social"
    if (l in fun_leisure) or (t in fun_tourism) or (a == "gaming"):
        return "fun"
    return pd.NA

others["cat5"] = others.apply(classify, axis=1)
others = others[others["cat5"].notna()].copy()  # keep only the 5 categories

# =========================================================
# 2.5) Convert EVERY feature to a POINT (centroid) before counting
# =========================================================
others_pts = others.copy()
geom_type = others_pts.geometry.geom_type

mask_not_point = ~geom_type.isin(["Point", "MultiPoint"])
others_pts.loc[mask_not_point, "geometry"] = others_pts.loc[mask_not_point, "geometry"].centroid

mask_multipoint = geom_type.eq("MultiPoint")
others_pts.loc[mask_multipoint, "geometry"] = others_pts.loc[mask_multipoint, "geometry"].centroid

# =========================================================
# 3) Count per TZ (POINT-in-POLYGON via intersects)
# =========================================================
sj = gpd.sjoin(
    tz[[ZONE_COL, "geometry"]],
    others_pts[["cat5", "geometry"]],
    how="left",
    predicate="intersects"
)

counts = (
    sj.dropna(subset=["cat5"])
      .groupby([ZONE_COL, "cat5"])
      .size()
      .unstack(fill_value=0)
)

counts = counts.reindex(tz[ZONE_COL].values, fill_value=0)

col_map = {
    "nightlife": "F6_others_nightlife",
    "social":    "F6_others_social",
    "spiritual": "F6_others_spiritual",
    "fun":       "F6_others_fun",
    "wellness":  "F6_others_wellness",
}
for k, outcol in col_map.items():
    tz[outcol] = counts[k].astype("int32") if k in counts.columns else 0

tz["F6_others_diversity"] = (
    (tz[list(col_map.values())] >= 1).sum(axis=1).astype("int32")
)

# =========================================================
# 3.5) MANUAL OVERRIDE (AOI check) for zone_id = 1
# =========================================================
MANUAL_ZONE_IDS = [1]

manual_vals = {
    "F6_others_nightlife": 0,
    "F6_others_social": 0,
    "F6_others_spiritual": 1,
    "F6_others_fun": 0,
    "F6_others_wellness": 0,
}

mask_manual = tz[ZONE_COL].isin(MANUAL_ZONE_IDS)
if mask_manual.sum() == 0:
    print("⚠️ MANUAL OVERRIDE: no TZ found for zone_id=1 (check your IDs).")
else:
    for c, v in manual_vals.items():
        tz.loc[mask_manual, c] = int(v)

    tz.loc[mask_manual, "F6_others_diversity"] = (
        (tz.loc[mask_manual, list(col_map.values())] >= 1).sum(axis=1).astype("int32")
    )

    print(f"✅ MANUAL OVERRIDE applied to TZ: {MANUAL_ZONE_IDS}")
    print(tz.loc[mask_manual, [ZONE_COL] + list(col_map.values()) + ["F6_others_diversity"]].to_string(index=False))

# =========================================================
# 4) Build F6_others_count = sum of all except spiritual
#     (nightlife + social + fun + wellness)
# =========================================================
cols_sum = [
    "F6_others_nightlife",
    "F6_others_social",
    "F6_others_fun",
    "F6_others_wellness",
]
missing = [c for c in cols_sum if c not in tz.columns]
if missing:
    raise ValueError(f"Missing columns in TZ: {missing}")

for c in cols_sum:
    tz[c] = pd.to_numeric(tz[c], errors="coerce").fillna(0)

tz["F6_others_count"] = tz[cols_sum].sum(axis=1).astype("int32")

# =========================================================
# 5) Save TZ + plots
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n✅ Saved F6_others_* to: {TZ_PATH}")

# Plot all category counts + diversity + total (non-spiritual)
for outcol in list(col_map.values()) + ["F6_others_diversity", "F6_others_count"]:
    fig, ax = plt.subplots(1, 1, figsize=(10, 9))
    tz.plot(column=outcol, ax=ax, legend=True, edgecolor="none", linewidth=0)
    ax.set_title(outcol)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(".")
OTHERS_GPKG = base / "OSM" / "Others" / "Others.gpkg"

print(f"📂 Reading: {OTHERS_GPKG}")
gdf = gpd.read_file(OTHERS_GPKG)
print(f"🔎 Features (raw): {len(gdf)} | CRS: {gdf.crs}")

# -----------------------------
# Basic geometry QC
# -----------------------------
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
print(f"✅ Features after NaN/empty geometry removal: {len(gdf)}")

# Optional validity check (can be slow)
# gdf = gdf[gdf.geometry.is_valid].copy()

gdf["geom_type"] = gdf.geometry.geom_type

print("\n📊 Geometry type distribution (ALL Others, raw):")
print(gdf["geom_type"].value_counts(dropna=False))

# -----------------------------
# Classification (same as your cat5) — wellness massage-only
# -----------------------------
nightlife_amenity = {"brothel","casino","love_hotel","nightclub","stripclub","swingerclub"}
social_amenity    = {"community_centre","social_centre"}
spiritual_amenity = {"grave_yard","monastery","place_of_worship"}
wellness_amenity  = {"kneipp_water_cure"}

fun_leisure       = {"adult_gaming_centre","amusement_arcade","escape_game","water_park","hackrrspace"}
fun_tourism       = {"theme_park"}

wellness_shop     = {"massage"}  # <-- MASSAGE ONLY

def _norm(x):
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s in ("None","nan","<NA>","null","") else s

def classify(row):
    a = _norm(row.get("amenity")) if "amenity" in row else ""
    l = _norm(row.get("leisure")) if "leisure" in row else ""
    t = _norm(row.get("tourism")) if "tourism" in row else ""
    s = _norm(row.get("shop"))    if "shop"    in row else ""

    if a in spiritual_amenity:
        return "spiritual"
    if a in wellness_amenity or s in wellness_shop:
        return "wellness"
    if a in nightlife_amenity:
        return "nightlife"
    if a in social_amenity:
        return "social"
    if (l in fun_leisure) or (t in fun_tourism) or (a == "gaming"):
        return "fun"
    return pd.NA

gdf["cat5"] = gdf.apply(classify, axis=1)

gdf_c = gdf[gdf["cat5"].notna()].copy()
print(f"\n✅ Features classified into A–E (kept): {len(gdf_c)} / {len(gdf)}")

print("\n📊 Category counts:")
print(gdf_c["cat5"].value_counts())

print("\n📊 Geometry type distribution PER category:")
for cat in ["nightlife","social","spiritual","fun","wellness"]:
    sub = gdf_c[gdf_c["cat5"] == cat]
    print(f"\n--- {cat.upper()} (n={len(sub)}) ---")
    print(sub["geom_type"].value_counts(dropna=False))

pivot = (
    gdf_c
    .groupby(["cat5", "geom_type"])
    .size()
    .unstack(fill_value=0)
    .astype(int)
)

print("\n📋 Summary (cat5 × geom_type):")
print(pivot)

pivot2 = pivot.copy()
pivot2["TOTAL"] = pivot2.sum(axis=1)
pivot2.loc["TOTAL"] = pivot2.sum(axis=0)
print("\n📋 Summary with totals:")
print(pivot2)


## F7 — Overall Diversity (Traffic Zones)

### Goal
Build a **generic diversity factor (F7)** by combining the existing factor-specific diversity counts:
- `F1_gastr_div`
- `F3_sport_div`
- `F3_sport_out_div`
- `F4_cult_diversity`
- `F5_outdoor_hard_diversity`
- `F5_outdoor_soft_diversity`
- `F6_others_diversity`

All outputs are stored back into `TZ/TZ.gpkg`.

---

### Step 1 — Normalize each diversity component (0–1)
Each diversity variable is a **count-based indicator**, so we apply **min–max scaling** (no log transform):
\[
x_{norm} = \frac{x - \min(x)}{\max(x) - \min(x)}
\]
For constant columns where \(\max=\min\), we set the normalized values to **0**.

Output fields:
- `F1_gastr_div_norm`
- `F3_sport_div_norm`
- `F3_sport_out_div_norm`
- `F4_cult_diversity_norm`
- `F5_outdoor_hard_diversity_norm`
- `F5_outdoor_soft_diversity_norm`
- `F6_others_diversity_norm`

---

### Step 2 — Raw diversity sum (un-normalized)
We compute the sum of the original diversity counts:
\[
F7\_div\_sum = \sum_k Fk\_div
\]

Output field:
- `F7_div_sum`

Then we normalize it with the same min–max scaling:
- `F7_div_sum_norm`

---

### Step 3 — Aggregation of normalized components
We compute:
- **Sum of normalized diversities**:
\[
F7\_norm\_sum = \sum_k Fk\_div\_{norm}
\]
- **Mean of normalized diversities**:
\[
F7\_norm\_mean = \frac{F7\_norm\_sum}{K}
\]
where \(K\) is the number of diversity components (here: 7).

Output fields:
- `F7_norm_sum`
- `F7_norm_mean`

(Optional but often handy)
- `F7_norm_sum_norm` = min–max normalized version of `F7_norm_sum`

---

### Interpretation
- `F7_div_sum` captures **absolute accumulated diversity** (still depends on the raw scale of the components).
- `F7_norm_mean` captures **relative diversity strength** across factors, giving **equal weight** to each factor-specific diversity measure.
- All normalized outputs lie in **[0, 1]**.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path(".")
TZ_PATH = base / "TZ" / "TZ.gpkg"

# =========================================================
# CONFIG
# =========================================================
DIV_COLS = [
    "F1_gastr_div",
    "F3_outdoor_hard_diversity",
    "F3_outdoor_soft_diversity",
    "F4_cult_diversity",
    "F5_sport_div",
    "F5_sport_out_div",
    "F6_others_diversity",
]
NORM_COLS = [c + "_norm" for c in DIV_COLS]

AGG_COLS = [
    "F7_div_sum",
    "F7_div_sum_norm",
    "F7_norm_sum",
    "F7_norm_mean",
    "F7_norm_sum_norm",
]

PLOT_COLS = DIV_COLS + NORM_COLS + AGG_COLS

# =========================================================
# HELPERS
# =========================================================
def minmax_01(s: pd.Series) -> pd.Series:
    """Min-max scale to [0,1]; constant columns -> 0."""
    s = pd.to_numeric(s, errors="coerce")
    s_min = s.min(skipna=True)
    s_max = s.max(skipna=True)
    if pd.isna(s_min) or pd.isna(s_max) or (s_max == s_min):
        return pd.Series(0.0, index=s.index)
    return (s - s_min) / (s_max - s_min)

# =========================================================
# 1) LOAD
# =========================================================
tz = gpd.read_file(TZ_PATH)

# =========================================================
# 2) CHECK INPUT COLUMNS
# =========================================================
missing_in = [c for c in DIV_COLS if c not in tz.columns]
if missing_in:
    raise ValueError(f"Missing diversity columns in TZ: {missing_in}")

# =========================================================
# 3) NORMALIZE EACH DIVERSITY COMPONENT (0–1)
# =========================================================
for c in DIV_COLS:
    tz[c] = pd.to_numeric(tz[c], errors="coerce").fillna(0)
    tz[c + "_norm"] = minmax_01(tz[c]).astype("float32")

# =========================================================
# 4) AGGREGATES
# =========================================================
# Raw sum (un-normalized)
tz["F7_div_sum"] = tz[DIV_COLS].sum(axis=1).astype("int32")

# Normalized raw sum
tz["F7_div_sum_norm"] = minmax_01(tz["F7_div_sum"]).astype("float32")

# Sum/mean of normalized components
tz["F7_norm_sum"] = tz[NORM_COLS].sum(axis=1).astype("float32")
K = len(NORM_COLS)
tz["F7_norm_mean"] = (tz["F7_norm_sum"] / K).astype("float32")

# Normalized version of F7_norm_sum (optional)
tz["F7_norm_sum_norm"] = minmax_01(tz["F7_norm_sum"]).astype("float32")

# =========================================================
# 5) SAVE
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"✅ Saved F7 diversity indicators to: {TZ_PATH}")

# =========================================================
# 6) PLOTS
# =========================================================
missing_plot = [c for c in PLOT_COLS if c not in tz.columns]
if missing_plot:
    raise ValueError(f"Missing columns for plotting (unexpected): {missing_plot}")

for c in PLOT_COLS:
    fig, ax = plt.subplots(1, 1, figsize=(10, 9))
    tz.plot(column=c, ax=ax, legend=True, edgecolor="none", linewidth=0)
    ax.set_title(c)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


### F7 Diversity Indices (Urban vs Outdoor)

### Goal
Create two composite diversity indicators at **Traffic Zone (TZ)** level:

- **`F7_urban_*`** → diversity of *urban / service-oriented* opportunities  
- **`F7_outdoor_*`** → diversity of *outdoor / nature-oriented* opportunities  

Each composite is built from existing **count-based diversity** fields (integer “how many subtypes present” style).  
For each group we compute:
- a raw **sum** (`*_sum`)
- a raw **mean** (`*_mean`)
- a **normalized** sum (`*_sum_norm`, min–max to 0–1)
- a **normalized** mean (`*_mean_norm`, min–max to 0–1)

All results are stored back into `TZ/TZ.gpkg`.

---

## Inputs (already in TZ)

### Urban diversity components
- `F1_gastr_div`
- `F3_sport_div`
- `F4_cult_diversity`
- `F6_others_diversity`

### Outdoor diversity components
- `F3_sport_out_div`
- `F5_outdoor_hard_diversity`
- `F5_outdoor_soft_diversity`

---

## Method

### 1) Define the two groups
- **Urban group** = (gastronomy + sport facilities + culture + “others”)
- **Outdoor group** = (outdoor routes diversity + outdoor hard diversity + outdoor soft diversity)

### 2) Build raw composites (no normalization yet)
For each TZ:
- `F7_urban_sum  = sum(urban components)`
- `F7_urban_mean = mean(urban components)`
- `F7_outdoor_sum  = sum(outdoor components)`
- `F7_outdoor_mean = mean(outdoor components)`

Missing values are treated as **0** (interpreted as “no presence recorded”).

### 3) Normalize (min–max scaling)
For any variable `x`, define:

- `x_norm = (x - min(x)) / (max(x) - min(x))`

If `max(x) == min(x)` (degenerate case), set `x_norm = 0`.

We apply min–max scaling to:
- `F7_urban_sum` → `F7_urban_sum_norm`
- `F7_urban_mean` → `F7_urban_mean_norm`
- `F7_outdoor_sum` → `F7_outdoor_sum_norm`
- `F7_outdoor_mean` → `F7_outdoor_mean_norm`

---

## Outputs written to TZ

### Urban composites
- `F7_urban_sum`
- `F7_urban_mean`
- `F7_urban_sum_norm`
- `F7_urban_mean_norm`

### Outdoor composites
- `F7_outdoor_sum`
- `F7_outdoor_mean`
- `F7_outdoor_sum_norm`
- `F7_outdoor_mean_norm`

---

## Interpretation
- **Urban diversity** increases where multiple urban functions co-exist (food/drink, cultural venues, sport facilities, other activities).
- **Outdoor diversity** increases where multiple outdoor opportunity types co-exist (routes + hard + soft outdoor features).

Keeping the indices separate is useful because alpine areas often score high on outdoor diversity while urban centers score high on urban diversity; combining them prematurely can hide these patterns.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path(".")
TZ_PATH = base / "TZ" / "TZ.gpkg"

# =========================================================
# CONFIG
# =========================================================
URBAN_COLS = [
    "F1_gastr_div",
    "F5_sport_div",
    "F4_cult_diversity",
    "F6_others_diversity",
]

OUTDOOR_COLS = [
    "F5_sport_out_div",
    "F3_outdoor_hard_diversity",
    "F3_outdoor_soft_diversity",
]

PLOT_COLS = [
    "F7_urban_sum", "F7_urban_mean", "F7_urban_sum_norm", "F7_urban_mean_norm",
    "F7_outdoor_sum", "F7_outdoor_mean", "F7_outdoor_sum_norm", "F7_outdoor_mean_norm",
]

# =========================================================
# HELPERS
# =========================================================
def minmax_01(s: pd.Series) -> pd.Series:
    """Min-max scale to [0,1]; constant/empty columns -> 0."""
    s = pd.to_numeric(s, errors="coerce")
    s_min = s.min(skipna=True)
    s_max = s.max(skipna=True)
    if pd.isna(s_min) or pd.isna(s_max) or (s_max == s_min):
        return pd.Series(0.0, index=s.index, dtype="float32")
    return ((s - s_min) / (s_max - s_min)).astype("float32")

# =========================================================
# 1) LOAD
# =========================================================
tz = gpd.read_file(TZ_PATH)

# =========================================================
# 2) CHECK INPUT COLUMNS
# =========================================================
missing_urban = [c for c in URBAN_COLS if c not in tz.columns]
missing_out = [c for c in OUTDOOR_COLS if c not in tz.columns]
if missing_urban or missing_out:
    raise ValueError(
        "Missing columns in TZ:\n"
        f"  urban missing: {missing_urban}\n"
        f"  outdoor missing: {missing_out}"
    )

# =========================================================
# 3) COERCE COMPONENTS TO NUMERIC + FILL NaN WITH 0
# =========================================================
for c in URBAN_COLS + OUTDOOR_COLS:
    tz[c] = pd.to_numeric(tz[c], errors="coerce").fillna(0)

# =========================================================
# 4) RAW COMPOSITES
# =========================================================
tz["F7_urban_sum"] = tz[URBAN_COLS].sum(axis=1).astype("float32")
tz["F7_urban_mean"] = tz[URBAN_COLS].mean(axis=1).astype("float32")

tz["F7_outdoor_sum"] = tz[OUTDOOR_COLS].sum(axis=1).astype("float32")
tz["F7_outdoor_mean"] = tz[OUTDOOR_COLS].mean(axis=1).astype("float32")

# =========================================================
# 5) NORMALIZE (MIN–MAX TO 0–1)
# =========================================================
tz["F7_urban_sum_norm"] = minmax_01(tz["F7_urban_sum"])
tz["F7_urban_mean_norm"] = minmax_01(tz["F7_urban_mean"])

tz["F7_outdoor_sum_norm"] = minmax_01(tz["F7_outdoor_sum"])
tz["F7_outdoor_mean_norm"] = minmax_01(tz["F7_outdoor_mean"])

# =========================================================
# 6) SAVE
# =========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"✅ Saved F7 urban/outdoor diversity composites to: {TZ_PATH}")

# =========================================================
# 7) PLOTS
# =========================================================
missing_plot = [c for c in PLOT_COLS if c not in tz.columns]
if missing_plot:
    raise ValueError(f"Missing output columns for plotting (unexpected): {missing_plot}")

for c in PLOT_COLS:
    fig, ax = plt.subplots(1, 1, figsize=(10, 9))
    tz.plot(column=c, ax=ax, legend=True, edgecolor="none", linewidth=0)
    ax.set_title(c)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


## F10 Super-Infrastructure

## Superinfrastructure (OSM) — sufficiency check

The following OSM tags are **sufficient** to represent **basic “superinfrastructure” / support amenities** (comfort + essential services):

- **Mobility / supply:** `amenity=fuel`, `amenity=charging_station`
- **Essential services:** `amenity=atm`, `amenity=toilets`, `amenity=drinking_water`
- **Micro-comfort infrastructure:** `amenity=bench`, `amenity=fountain`, `amenity=water_point`, `amenity=watering_place`

### Notes (bias control)
- `bench` can be **over-mapped** in some municipalities and **under-mapped** elsewhere → it may dominate counts.  
  *Option:* keep it, but also test a variant **without bench** or apply a **log(1+x)** transform when standardizing.
- `water_point` / `watering_place` tend to be more **rural/outdoor-oriented**, which is fine if “superinfra” includes outdoor support.

### Optional extensions (only if needed)
If you want a slightly richer—but still controlled—definition, consider adding:
- `amenity=shelter`, `amenity=shower` (mapping varies)


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================================
# PATHS
# ==========================================================
base = Path(".")

TZ_PATH  = base / "TZ" / "TZ.gpkg"
SI_GPKG  = base / "OSM" / "Superinfrastructure" / "SI.gpkg"

TARGET_EPSG = 2056

# Amenity list (EXCLUDING bench)
AMENITY_KEEP = {
    "fuel",
    "charging_station",
    "atm",
    "toilets",
    "drinking_water",
    "fountain",
    "water_point",
    "watering_place",
}

# ==========================================================
# 1) Load TZ
# ==========================================================
print(f"📂 Reading TZ from: {TZ_PATH}")
tz = gpd.read_file(TZ_PATH)

ZONE_COL = None
for c in ["zone_id", "ZONE_ID", "id", "ID", "OBJECTID"]:
    if c in tz.columns:
        ZONE_COL = c
        break
if ZONE_COL is None:
    raise ValueError(f"❌ TZ: cannot find an ID column. Columns: {list(tz.columns)}")

tz[ZONE_COL] = pd.to_numeric(tz[ZONE_COL], errors="coerce").astype("Int64")
if tz[ZONE_COL].isna().any():
    raise ValueError(f"❌ TZ {ZONE_COL} has NaNs after numeric conversion.")
tz[ZONE_COL] = tz[ZONE_COL].astype(int)

if tz.crs is None:
    raise ValueError("❌ TZ has no CRS.")
if tz.crs.to_epsg() != TARGET_EPSG:
    tz = tz.to_crs(TARGET_EPSG)
    print(f"↪ Reprojected TZ to EPSG:{TARGET_EPSG}")

tz_min = tz[[ZONE_COL, "geometry"]].copy()

# ==========================================================
# 2) Load SI.gpkg and convert ALL geometries to POINTS
# ==========================================================
print(f"\n📂 Reading super-infrastructure (SI) from: {SI_GPKG}")
si = gpd.read_file(SI_GPKG)

if si.crs is None:
    raise ValueError("❌ SI.gpkg has no CRS.")
if si.crs.to_epsg() != TARGET_EPSG:
    si = si.to_crs(TARGET_EPSG)
    print(f"↪ Reprojected SI to EPSG:{TARGET_EPSG}")

# Basic geometry cleaning
si = si[si.geometry.notna() & ~si.geometry.is_empty].copy()

# ==========================================================
# 2a) PRINT COUNTS BY AMENITY CATEGORY + TOTAL
# ==========================================================
if "amenity" not in si.columns:
    raise ValueError(f"❌ SI.gpkg has no 'amenity' column. Columns: {list(si.columns)}")

si["amenity_norm"] = (
    si["amenity"].astype("string").str.strip().str.lower()
      .replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
)

n_tot = len(si)
n_missing = int(si["amenity_norm"].isna().sum())
n_sel = int(si["amenity_norm"].isin(AMENITY_KEEP).sum())

print("\n" + "="*60)
print("SUPER-INFRASTRUCTURE: amenity counts (raw layer)")
print("="*60)
print(f"Total features (all): {n_tot}")
print(f"Missing/NA amenity:  {n_missing}")
print(f"Selected amenities:  {n_sel}")
print(f"Other amenities:     {n_tot - n_missing - n_sel}")

vc_sel = (
    si.loc[si["amenity_norm"].isin(AMENITY_KEEP), "amenity_norm"]
      .value_counts()
      .reindex(sorted(AMENITY_KEEP), fill_value=0)
)

print("\nCounts by selected amenity (bench excluded):")
print(vc_sel.to_string())

vc_other = (
    si.loc[~si["amenity_norm"].isin(AMENITY_KEEP) & si["amenity_norm"].notna(), "amenity_norm"]
      .value_counts()
      .head(15)
)
if len(vc_other) > 0:
    print("\nTop 15 'other' amenities (not in list):")
    print(vc_other.to_string())

# ==========================================================
# 2b) Convert everything to points (centroids)
# ==========================================================
si_pts = si.copy()
gtype = si_pts.geometry.geom_type

mask_not_point = ~gtype.isin(["Point", "MultiPoint"])
si_pts.loc[mask_not_point, "geometry"] = si_pts.loc[mask_not_point, "geometry"].centroid

mask_multipoint = gtype.eq("MultiPoint")
si_pts.loc[mask_multipoint, "geometry"] = si_pts.loc[mask_multipoint, "geometry"].centroid

bad = si_pts.geometry.geom_type != "Point"
if bad.any():
    si_pts.loc[bad, "geometry"] = si_pts.loc[bad, "geometry"].representative_point()

# ==========================================================
# 2c) FILTER: keep ONLY selected amenities (bench already excluded)
# ==========================================================
si_pts_sel = si_pts.loc[si_pts["amenity_norm"].isin(AMENITY_KEEP), ["geometry"]].copy()
print(f"\n📌 SI selected amenities used for counting (bench excluded): {len(si_pts_sel)}")

# ==========================================================
# 3) Count per TZ (point-in-polygon via intersects)
# ==========================================================
sj = gpd.sjoin(
    tz_min,
    si_pts_sel,
    how="left",
    predicate="intersects"
)

counts = (
    sj.dropna(subset=["index_right"])
      .groupby(ZONE_COL)
      .size()
)

tz["F10_superinfra"] = tz[ZONE_COL].map(counts).fillna(0).astype("int32")

print("\n✅ F10_superinfra computed (bench excluded).")
print("   zones with 0:", int((tz["F10_superinfra"] == 0).sum()))
print("   max:", int(tz["F10_superinfra"].max()))

# ==========================================================
# 4) Save + Plot
# ==========================================================
tz.to_file(TZ_PATH, driver="GPKG")
print(f"\n💾 Saved TZ with 'F10_superinfra' to: {TZ_PATH}")

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
tz.plot(column="F10_superinfra", ax=ax, legend=True, edgecolor="none", linewidth=0)
ax.set_title("F10_superinfra (OSM support amenities, bench excluded)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


## F8 — Density indicators (raw, per km²)

### Goal
Compute **raw density indicators** for a set of TZ-level variables by dividing each variable by zone area:

\[
\text{dens}(j)=\frac{X(j)}{\text{Area\_km2}(j)}
\]

This step produces **raw densities only** (no log/winsor/min–max normalization yet).  
Robust transformations and normalization to \([0,1]\) will be applied later in the Methods chapter.

### Inputs
- `TZ/TZ.gpkg`
  - `Area_km2` (zone area in km²)
  - the selected TZ attributes (counts, areas, lengths, population, etc.)

### Outputs
For each input variable `X`, we create a density column:

- `F8_<shortname>_dens = X / Area_km2`

We also build composite densities by summing groups first and then dividing by `Area_km2`:

- `F8_POI_urban_dens` = (F1_gastr_count + F5_sport_count + F4_cult_count + F6_others_count + F6_others_spiritual) / Area_km2  
- `F8_POI_natural_dens` = (F3_outdoor_soft_count + F3_outdoor_hard_count) / Area_km2  
- `F8_lengths_dens` = (F5_sport_out_length + F3_outdoor_walk) / Area_km2  
- `F8_POI_total_dens` = (urban POI sum + natural POI sum) / Area_km2  

All new columns are written back into `TZ/TZ.gpkg` and mapped as choropleths.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path(".")
TZ_PATH = base / "TZ" / "TZ.gpkg"

AREA_COL = "Area_km2"

# ------------------------------------------------------------
# Variables to densify (raw density = X / Area_km2)
# Output naming: F8_<shortname>_dens
# ------------------------------------------------------------
VAR_MAP = {
    # gastronomy
    "F1_gastr_count": "gastr_count",
    "F1_gastr_area_total": "gastr_area_total",

    # visits
    "F2_pop_total": "pop_total",
    "F2_hh_total": "hh_total",

    # sport facilities + surface + routes
    "F5_sport_count": "sport_count",
    "F5_sport_area_total": "sport_area_total",
    "F5_sport_out_length": "sport_out_length",

    # culture
    "F4_cult_count": "cult_count",
    "F4_cult_area_total": "cult_area_total",

    # outdoor core (note: units may differ; we keep raw density and document later)
    "F3_outdoor_walk": "outdoor_walk",
    "F3_outdoor_lake_raw": "outdoor_lake_raw",
    "F3_outdoor_lake": "outdoor_lake",
    "F3_outdoor_protected_area": "outdoor_protected_area",
    "F3_lake_area": "lake_area",
    "F3_outdoor_river": "outdoor_river",

    # outdoor POI aggregates + soft area proxies
    "F3_outdoor_hard_count": "outdoor_hard_count",
    "F3_outdoor_soft_count": "outdoor_soft_count",
    "F3_outdoor_soft_area_poly": "outdoor_soft_area_poly",
    "F3_outdoor_soft_area_imp": "outdoor_soft_area_imp",

    # others (subcategories + total)
    "F6_others_nightlife": "others_nightlife",
    "F6_others_social": "others_social",
    "F6_others_spiritual": "others_spiritual",
    "F6_others_fun": "others_fun",
    "F6_others_wellness": "others_wellness",
    "F6_others_count": "others_count",

    # super-infrastructure
    "F10_superinfra": "superinfra",
}

# ------------------------------------------------------------
# Composite density definitions (sum first, then divide by Area_km2)
# ------------------------------------------------------------
COMPOSITES = {
    "F8_POI_urban_dens": [
        "F1_gastr_count",
        "F5_sport_count",
        "F4_cult_count",
        "F6_others_count",
        "F6_others_spiritual",
    ],
    "F8_POI_natural_dens": [
        "F3_outdoor_soft_count",
        "F3_outdoor_hard_count",
    ],
    "F8_lengths_dens": [
        "F5_sport_out_length",
        "F3_outdoor_walk",
    ],
    "F8_POI_total_dens": [
        "F1_gastr_count",
        "F5_sport_count",
        "F4_cult_count",
        "F6_others_count",
        "F6_others_spiritual",
        "F3_outdoor_soft_count",
        "F3_outdoor_hard_count",
    ],
}

# ------------------------------------------------------------
# 1) Load TZ
# ------------------------------------------------------------
tz = gpd.read_file(TZ_PATH)

if AREA_COL not in tz.columns:
    raise ValueError(f"Missing '{AREA_COL}' in TZ.gpkg.")

# area numeric and safe
tz[AREA_COL] = pd.to_numeric(tz[AREA_COL], errors="coerce")
if tz[AREA_COL].isna().any():
    raise ValueError(f"'{AREA_COL}' contains NaNs after numeric conversion.")

# avoid division by zero
area_safe = tz[AREA_COL].astype("float64").copy()
area_safe[area_safe <= 0] = np.nan

# ------------------------------------------------------------
# 2) Check variables exist (and coerce numeric)
# ------------------------------------------------------------
missing_vars = [v for v in VAR_MAP.keys() if v not in tz.columns]
if missing_vars:
    raise ValueError(f"Missing source variables in TZ: {missing_vars}")

# Coerce to numeric (counts/areas/lengths can be float/int)
for v in VAR_MAP.keys():
    tz[v] = pd.to_numeric(tz[v], errors="coerce").fillna(0)

# ------------------------------------------------------------
# 3) Create per-variable density columns
# ------------------------------------------------------------
created_cols = []

for src, shortname in VAR_MAP.items():
    outcol = f"F8_{shortname}_dens"
    tz[outcol] = (tz[src] / area_safe).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float64")
    created_cols.append(outcol)

# ------------------------------------------------------------
# 4) Create composite density columns
# ------------------------------------------------------------
for outcol, components in COMPOSITES.items():
    miss = [c for c in components if c not in tz.columns]
    if miss:
        raise ValueError(f"Composite '{outcol}' missing components in TZ: {miss}")

    # sum first, then densify
    comp_sum = tz[components].sum(axis=1).astype("float64")
    tz[outcol] = (comp_sum / area_safe).replace([np.inf, -np.inf], np.nan).fillna(0).astype("float64")
    created_cols.append(outcol)

print(f"✅ Created {len(created_cols)} F8 density columns.")

# ------------------------------------------------------------
# 5) Save
# ------------------------------------------------------------
tz.to_file(TZ_PATH, driver="GPKG")
print(f"💾 Saved updated TZ to: {TZ_PATH}")

# ------------------------------------------------------------
# 6) Plot all new density columns
# ------------------------------------------------------------
for c in created_cols:
    fig, ax = plt.subplots(1, 1, figsize=(10, 9))
    tz.plot(column=c, ax=ax, legend=True, edgecolor="none", linewidth=0)
    ax.set_title(c)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


## Traffic Zone Variable Inventory

In [ ]:
import geopandas as gpd
import re

# Path
TZ_PATH = "TZ/TZ.gpkg"

# Read TZ
tz = gpd.read_file(TZ_PATH)

# Columns to exclude from printing/sorting
exclude_cols = {
    "F3_outdoor_lake",
    "F8_outdoor_lake_dens",
}

# Keep only columns starting with "F", excluding the two requested
f_cols = [
    col for col in tz.columns
    if str(col).startswith("F") and col not in exclude_cols
]

def sort_key(col):
    m = re.match(r"^F(\d+)", str(col))
    if m:
        fnum = int(m.group(1))
        # send F10 to the end
        if fnum == 10:
            return (999, str(col))
        return (fnum, str(col))
    return (1000, str(col))

f_cols = sorted(f_cols, key=sort_key)

print(f"Variables starting with 'F' (excluding selected ones): {len(f_cols)}\n")

for i, col in enumerate(f_cols, start=1):
    print(f"{i:03d}  {col}  ({tz[col].dtype})")